# Decoding analyses

Three sections:

**Section A: Thoughtblock decoding** — LOSO ridge predicting dyscontrol, permutation null, parcel coefficients, probe validation.

**Section B: SART decoding** — LOSO ridge predicting dyscontrol and probes from SART block-mean. Cross-pattern comparison with TB.

**Section C: SART-Thoughtblock pattern similarity** — does the worry pattern persist into SART?

All similarity in Section C uses **per-subject SART z-scoring**: every pattern (template and SART windows) is expressed as `(pattern - subject's mean SART pattern) / subject's SART SD`. This puts everything on a 'how much does this deviate from typical SART, in standard-deviation units' scale.

For each of three template variants (C1 same-trial, C2 subject-mean, C3 high-dyscontrol), we compute:
- **block_sim**: similarity between the block-mean pattern and the template (one number per block)
- **mean_sim**: mean across the sliding-window trajectory
- **max_sim**: peak similarity in the trajectory
- **peak_tr**: when (in TRs) the peak occurs
- **slope**: linear slope of similarity vs window position

Plus a group-level trajectory plot per template, split by within-subject median of probe1.

**Outputs** saved to `derivatives/decoding/` for figures.ipynb.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp, linregress
from sklearn.linear_model import Ridge, Lasso

sys.path.insert(0, str(Path.cwd()))

from config import (
    BASE, DERIVATIVES_DIR, PARCELS_DIR, BLOCK_FEATURES_CSV,
    EXCLUDE_SUBJECTS, TR_SECONDS, HRF_LAG_TRS,
    ATLAS_NAME, YEO17_NAMES, ATLAS_OUT_DIR,
    RIDGE_ALPHA, LASSO_ALPHA, N_PERMUTATIONS, RANDOM_SEED,
    SIMILARITY_METRIC, SART_WINDOW_TRS, SART_WINDOW_STEP, FIGURES_DIR
)
from decoding import (
    loso_decode, summarize_loso, loso_permutation_null,
    permutation_pvalue, parcel_coefficients, probe_decoder_correlations,
)
from similarity import sliding_window_similarity, get_metric
from parcellation import get_block_indices

DECODING_OUT_DIR = DERIVATIVES_DIR / 'decoding'
DECODING_OUT_DIR.mkdir(parents=True, exist_ok=True)
ATLAS_DIR = ATLAS_OUT_DIR

print(f'Atlas: {ATLAS_NAME}')
print(f'Similarity metric: {SIMILARITY_METRIC}')
print(f'Sliding window: {SART_WINDOW_TRS} TRs ({SART_WINDOW_TRS*TR_SECONDS}s), '
      f'step {SART_WINDOW_STEP} TR')
print(f'Outputs will save to: {DECODING_OUT_DIR}')

## Load block features

In [ ]:
import pandas as pd
from config import (BLOCK_FEATURES_CSV, EXCLUDE_SUBJECTS, EXCLUDE_RUNS)

# Load block features
block_df_raw = pd.read_csv(BLOCK_FEATURES_CSV)
print(f'Loaded {len(block_df_raw)} blocks ({block_df_raw["subject"].nunique()} subjects)')

# Apply subject-level exclusions
mask_subj = ~block_df_raw['subject'].isin(EXCLUDE_SUBJECTS)
n_dropped_subj = (~mask_subj).sum()

# Apply run-level exclusions
mask_run = pd.Series(True, index=block_df_raw.index)
for sub, run in EXCLUDE_RUNS:
    drop_mask = (block_df_raw['subject'] == sub) & (block_df_raw['run'] == run)
    mask_run = mask_run & ~drop_mask

# Combine
block_df = block_df_raw[mask_subj & mask_run].copy().reset_index(drop=True)

# Drop rows with NaN dyscontrol_imputed (may not have prescan ratings)
block_df = block_df[block_df['dyscontrol_imputed'].notna()].copy()

print(f'After exclusions:')
print(f'  Excluded subjects {sorted(EXCLUDE_SUBJECTS)}: {n_dropped_subj} rows dropped')
print(f'  Excluded runs {EXCLUDE_RUNS}: {(~mask_run).sum()} rows dropped')
print(f'  Final: {len(block_df)} blocks ({block_df["subject"].nunique()} subjects)')
print()
print(f'Blocks per condition: {block_df["condition"].value_counts().to_dict()}')

# Sanity check: confirm 303 run-3 is gone, 315 is gone
assert len(block_df[(block_df['subject']==303) & (block_df['run']==3)]) == 0
assert len(block_df[block_df['subject']==315]) == 0
print('\nSanity checks passed ✓')

# Define standard subsets
tb_blocks = block_df[block_df['condition'] == 'Thoughtblock'].reset_index(drop=True)
sart_blocks = block_df[block_df['condition'] == 'SART'].reset_index(drop=True)
print(f'\nTB blocks: {len(tb_blocks)}, SART blocks: {len(sart_blocks)}')
print(f'Subjects with TB+SART data: {len(set(tb_blocks["subject"]) & set(sart_blocks["subject"]))}')

# Per-subject SART block counts
sart_per_sub = sart_blocks.groupby('subject').size()
print(f'\nSART blocks per subject: M={sart_per_sub.mean():.1f}, '
      f'min={sart_per_sub.min()}, max={sart_per_sub.max()}')

In [ ]:
# =============================================================================
# Spot-check: block_features.csv against original sources
# =============================================================================
# For a few random rows, verify:
#   1. probes 1-6 match alldata.csv (the in-scanner trial-level data)
#   2. dyscontrol_imputed matches the prescan thought-ratings questionnaire
# 
# This catches drift between the merged data and the originals.

import pandas as pd
import numpy as np

from alldata_loader import load_alldata
from questionnaire_loader import load_prescan_thought_ratings, normalize_thought_key
from merge import load_manual_corrections, apply_manual_corrections
from config import (
    BLOCK_FEATURES_CSV, ALLDATA_PATH, PRESCAN_PATH, MANUAL_CORRECTIONS_PATH,
)

# Load sources
block_df = pd.read_csv(BLOCK_FEATURES_CSV)
alldata = load_alldata(ALLDATA_PATH, verbose=False)
prescan = load_prescan_thought_ratings(PRESCAN_PATH)
corrections = load_manual_corrections(MANUAL_CORRECTIONS_PATH)
alldata_corrected = apply_manual_corrections(alldata, corrections)

print(f'block_features:     {block_df.shape}')
print(f'alldata_corrected:  {alldata_corrected.shape}')
print(f'prescan:            {prescan.shape}')

# Random sample
N_CHECK = 5
sample = block_df.sample(N_CHECK, random_state=42).reset_index(drop=True)

probe_cols = [f'probe{i}' for i in range(1, 7)]
all_match = True

for _, row in sample.iterrows():
    sub = int(row['subject'])
    run = int(row['run'])
    tn = int(row['thoughtNumber'])
    condition = row['condition']
    thought_key = row.get('thought_key', '?')

    print(f'\n=== sub-{sub} run-{run} {condition} thoughtNumber={tn} '
          f'thought_key={thought_key!r} ===')

    # ----- Probes vs alldata -----
    ad_match = alldata_corrected[
        (alldata_corrected['subject'] == sub)
        & (alldata_corrected['run'] == run)
        & (alldata_corrected['ThoughtNumber'] == tn)
    ]
    print(f'\n  ALLDATA matches: {len(ad_match)}')
    if len(ad_match) == 0:
        print('    ✗ NO MATCH IN ALLDATA')
        all_match = False
    else:
        ad_row = ad_match.iloc[0]
        print(f'    {"probe":7s}  {"alldata":>10s}  {"block":>10s}  match')
        for p in probe_cols:
            ad_val = ad_row.get(p, np.nan)
            blk_val = row.get(p, np.nan)
            both_nan = pd.isna(ad_val) and pd.isna(blk_val)
            equal = both_nan or (
                not pd.isna(ad_val) and not pd.isna(blk_val) 
                and abs(ad_val - blk_val) < 1e-6
            )
            mark = '✓' if equal else '✗'
            if not equal:
                all_match = False
            print(f'    {p:7s}  {ad_val!s:>10s}  {blk_val!s:>10s}  {mark}')

    # ----- dyscontrol vs prescan -----
    ps_match = prescan[
        (prescan['subject'] == sub)
        & (prescan['thought_key'] == thought_key)
    ]
    print(f'\n  PRESCAN matches: {len(ps_match)}')
    if len(ps_match) == 0:
        print('    ✗ NO MATCH IN PRESCAN')
        all_match = False
    else:
        ps_val = ps_match['dyscontrol_imputed'].iloc[0]
        blk_val = row['dyscontrol_imputed']
        equal = (
            (pd.isna(ps_val) and pd.isna(blk_val))
            or (not pd.isna(ps_val) and not pd.isna(blk_val) 
                and abs(ps_val - blk_val) < 1e-6)
        )
        mark = '✓' if equal else '✗'
        if not equal:
            all_match = False
        print(f'    dyscontrol_imputed: prescan={ps_val:.4f}  '
              f'block={blk_val:.4f}  {mark}')

print('\n' + '=' * 70)
print(f'OVERALL: {"ALL MATCH ✓" if all_match else "SOME MISMATCHES ✗"}')

In [ ]:
print(block_df['dyscontrol_raw'].isna().sum(), 'NaN raw values')
print(block_df[['dyscontrol_raw', 'dyscontrol_imputed']].describe())

# Complete-case count (raw == imputed only when nothing was missing)
both = block_df.dropna(subset=['dyscontrol_raw'])
identical = ((both['dyscontrol_raw'] - both['dyscontrol_imputed']).abs() < 0.001).sum()
print(f'{identical} of {len(both)} present-raw rows are complete cases')

In [ ]:
# Who's in the task sample but missing trait data?
task_subs = set(block_df['subject'].unique())

trait_cols = ['pswq_total', 'ptq_total', 'ptq_core', 'ptq_unproductive', 'ptq_capacity']
sub_traits = block_df.groupby('subject')[trait_cols].first()

# Per trait column, who's NaN?
print(f'Task subjects: {len(task_subs)}')
for c in trait_cols:
    missing = sub_traits[sub_traits[c].isna()].index.tolist()
    print(f'{c}: {len(missing)} missing -> {missing}')

# Subjects missing ANY trait
missing_any = sub_traits[sub_traits.isna().any(axis=1)].index.tolist()
print(f'\nMissing at least one trait: {len(missing_any)} -> {missing_any}')

# Subjects missing ALL traits
missing_all = sub_traits[sub_traits.isna().all(axis=1)].index.tolist()
print(f'Missing ALL traits: {len(missing_all)} -> {missing_all}')

# Show the actual rows for the missing subjects
print('\nTrait values for subjects missing any:')
print(sub_traits.loc[missing_any].to_string())

In [ ]:
# Do these 5 subjects exist in the raw questionnaire data?
missing_subs = [320, 342, 353, 365, 401]

# Load the questionnaire source (adjust to your loader/path)
# This is wherever pswq_total/ptq_total originate — likely a questionnaire_loader
from questionnaire_loader import load_trait_scores   # or whatever it's called
from config import QUALTRICS_PATH
q = load_trait_scores(QUALTRICS_PATH)   # adjust

print('Columns:', q.columns.tolist())
print('Subjects in questionnaire file:', sorted(q['subject'].unique()) if 'subject' in q else 'no subject col')
print('\nAre the missing 5 present in the questionnaire source?')
for s in missing_subs:
    present = s in q['subject'].values if 'subject' in q else False
    print(f'  {s}: {"PRESENT" if present else "ABSENT"}')

---

# Section A: Thoughtblock decoding

## A1. LOSO ridge decoding

In [ ]:
parcel_cols = [c for c in block_df.columns if c.startswith('parcel_')]

ridge_results = loso_decode(
    tb_blocks, parcel_cols=parcel_cols,
    target_col='dyscontrol_imputed',
    model_class=Ridge,
    model_kwargs={'alpha': RIDGE_ALPHA},
)
ridge_summary = summarize_loso(ridge_results, label='TB Ridge')

lasso_results = loso_decode(
    tb_blocks, parcel_cols=parcel_cols,
    target_col='dyscontrol_imputed',
    model_class=Lasso,
    model_kwargs={'alpha': LASSO_ALPHA, 'max_iter': 10000},
)
lasso_summary = summarize_loso(lasso_results, label='TB Lasso')

ridge_results.to_csv(DECODING_OUT_DIR / 'loso_per_subject.csv', index=False)
print(f'\nSaved: {DECODING_OUT_DIR / "loso_per_subject.csv"}')

Compare results using imputed vs. raw dyscontrol

In [ ]:
parcel_cols = [c for c in block_df.columns if c.startswith('parcel_')]
tb_blocks = block_df[block_df['condition'] == 'Thoughtblock'].reset_index(drop=True)

# 1. Primary (imputed) — your existing r=+0.167
r_imp = loso_decode(tb_blocks, parcel_cols=parcel_cols, target_col='dyscontrol_imputed',
                    model_class=Ridge, model_kwargs={'alpha': RIDGE_ALPHA})
summarize_loso(r_imp, label='Primary (imputed)')

# 2. Observed-only — drop thoughts with missing raw, use raw
tb_obs = tb_blocks.dropna(subset=['dyscontrol_raw'])
r_obs = loso_decode(tb_obs, parcel_cols=parcel_cols, target_col='dyscontrol_raw',
                    model_class=Ridge, model_kwargs={'alpha': RIDGE_ALPHA})
summarize_loso(r_obs, label='Observed-only (raw)')

# 3. Complete-case — only fully-observed thoughts (no imputation at all)
tb_cc = tb_blocks[
    ((tb_blocks['dyscontrol_raw'] - tb_blocks['dyscontrol_imputed']).abs() < 0.001)
    & tb_blocks['dyscontrol_raw'].notna()
]
r_cc = loso_decode(tb_cc, parcel_cols=parcel_cols, target_col='dyscontrol_raw',
                   model_class=Ridge, model_kwargs={'alpha': RIDGE_ALPHA})
summarize_loso(r_cc, label='Complete-case')

print('\nTarget SD:')
print(f'  imputed:       {tb_blocks["dyscontrol_imputed"].std():.3f}')
print(f'  observed-only: {tb_obs["dyscontrol_raw"].std():.3f}')
print(f'  complete-case: {tb_cc["dyscontrol_raw"].std():.3f}')
r_cc = loso_decode(tb_cc, parcel_cols=parcel_cols, target_col='dyscontrol_raw',
                   model_class=Ridge, model_kwargs={'alpha': RIDGE_ALPHA})
summarize_loso(r_cc, label='Complete-case')

# Compare target variance (range-restriction check)
print('\nTarget SD:')
print(f'  imputed:       {tb_blocks["dyscontrol_imputed"].std():.3f}')
print(f'  observed-only: {tb_obs["dyscontrol_raw"].std():.3f}')
print(f'  complete-case: {tb_cc["dyscontrol_raw"].std():.3f}')

In [ ]:
# =============================================================================
# DIAGNOSTIC (corrected): match on THOUGHT IDENTITY (thought_key), not
# thoughtNumber (which is just the presentation slot within a block).
#
# (1) blocks per (subject, thought_key)  -> should be ~3 (the run repeats)
# (2) is dyscontrol_imputed constant within (subject, thought_key)?  -> should be YES
# (3) LOSO r two ways: all blocks  vs  thought-averaged (true ~9 thoughts)
# =============================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr
from decoding import zscore_within_subject, center_within_subject

KEY = 'thought_key'
dfd = tb_blocks.dropna(subset=['dyscontrol_imputed'] + parcel_cols).copy()
assert KEY in dfd.columns, f'{KEY} not in tb_blocks — check the column name'

# (1) granularity by true thought identity
reps = dfd.groupby(['subject', KEY]).size()
print(f'blocks per (subject, {KEY}): M={reps.mean():.2f}, '
      f'min={reps.min()}, max={reps.max()}')
print(f'distinct thoughts per subject: '
      f'M={dfd.groupby("subject")[KEY].nunique().mean():.1f}')

# (2) is dyscontrol constant within (subject, thought)?  (now it SHOULD be)
spread = (dfd.groupby(['subject', KEY])['dyscontrol_imputed']
            .agg(lambda v: v.max() - v.min()))
n_var = (spread > 1e-9).sum()
print(f'\n(subject,{KEY}) cells with non-constant dyscontrol: {n_var}/{len(spread)}')
print('  -> expect ~0 if dyscontrol is a per-thought prescan rating.')

# (3) LOSO predictions on all blocks, then r two ways
subjects = sorted(dfd['subject'].unique())
pred = pd.Series(index=dfd.index, dtype=float)
for held_out in subjects:
    train = dfd[dfd['subject'] != held_out]
    test  = dfd[dfd['subject'] == held_out]
    if len(test) < 2:
        continue
    X_tr = zscore_within_subject(train[parcel_cols].values.astype(float),
                                 train['subject'].values)
    y_tr = center_within_subject(train['dyscontrol_imputed'].values.astype(float),
                                 train['subject'].values)
    X_te = test[parcel_cols].values.astype(float)
    mu, sd = X_te.mean(0), X_te.std(0); sd[sd == 0] = 1
    m = Ridge(alpha=RIDGE_ALPHA).fit(X_tr, y_tr)
    pred.loc[test.index] = m.predict((X_te - mu) / sd)
dfd = dfd.assign(pred=pred).dropna(subset=['pred'])

# A. r across all blocks (current approach)
rA = []
for sub, g in dfd.groupby('subject'):
    if len(g) >= 5 and g['pred'].std() > 0 and g['dyscontrol_imputed'].std() > 0:
        rA.append(pearsonr(g['pred'], g['dyscontrol_imputed'])[0])
rA = np.array(rA)

# B. average predictions to one row per (subject, thought_key), then r
tl = (dfd.groupby(['subject', KEY])
         .agg(pred=('pred', 'mean'), dys=('dyscontrol_imputed', 'first'))
         .reset_index())
rB = []
for sub, g in tl.groupby('subject'):
    if len(g) >= 5 and g['pred'].std() > 0 and g['dys'].std() > 0:
        rB.append(pearsonr(g['pred'], g['dys'])[0])
rB = np.array(rB)

print('\n--- per-subject validation r (mean ± SD) ---')
print(f'A. all blocks       : r = {rA.mean():.3f} ± {rA.std():.3f} (n={len(rA)})')
print(f'B. thought-averaged : r = {rB.mean():.3f} ± {rB.std():.3f} (n={len(rB)})')
print(f'inflation (A - B)   = {rA.mean() - rB.mean():+.3f}')

## A2. Permutation null

In [ ]:
# perm_null = loso_permutation_null(
#     tb_blocks, parcel_cols=parcel_cols,
#     target_col='dyscontrol_imputed',
#     model_class=Ridge,
#     model_kwargs={'alpha': RIDGE_ALPHA},
#     n_permutations=N_PERMUTATIONS,
#     random_seed=RANDOM_SEED,
# )

# real_mean_r = ridge_summary['mean_r']
# null_means = perm_null['mean_r'].values
# p_perm = permutation_pvalue(real_mean_r, null_means)

# print(f'\nPermutation null:')
# print(f'  Permutations: {len(null_means)}')
# print(f'  Null mean r: M={null_means.mean():+.4f}, SD={null_means.std():.4f}')
# print(f'  Range: [{null_means.min():+.4f}, {null_means.max():+.4f}]')
# print(f'  Real mean r: {real_mean_r:+.4f}')
# print(f'  Two-tailed permutation p: {p_perm:.4f}')

# perm_null.to_csv(DECODING_OUT_DIR / 'loso_permutation_null.csv', index=False)

## A3. Parcel coefficients

In [ ]:
tb_coef = parcel_coefficients(
    tb_blocks, parcel_cols=parcel_cols,
    parcel_labels=YEO17_NAMES,
    target_col='dyscontrol_imputed',
    alpha=RIDGE_ALPHA,
)
print('Thoughtblock parcel coefficients (sorted by |univariate_r|):')
print(tb_coef.sort_values('univariate_r', key=abs, ascending=False).round(4)
       .to_string(index=False))
tb_coef.to_csv(DECODING_OUT_DIR / 'parcel_coefficients.csv', index=False)

In [ ]:
# =============================================================================
# Figure: TB → dyscontrol parcel coefficients (bar chart)
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load coefficients
tb_coefs = pd.read_csv(DECODING_OUT_DIR / 'parcel_coefficients.csv')
print(f'Columns: {list(tb_coefs.columns)}')
print(tb_coefs.sort_values('univariate_r', key=abs, ascending=False).round(4))


# Sort by univariate_r (descending — most positive at top, most negative at bottom)
sorted_df = tb_coefs.sort_values('univariate_r', ascending=False).reset_index(drop=True)

# Color by network (same palette as SART figure)
network_colors = {
    'Visual A':            '#7B3F9D',
    'Visual B':            '#9D60BC',
    'Somatomotor A':       '#3B7CB8',
    'Somatomotor B':       '#5B9CD8',
    'Dorsal Attention A':  '#319D5C',
    'Dorsal Attention B':  '#56C080',
    'Salience/VentAttn A': '#C25599',
    'Salience/VentAttn B': '#DC76B6',
    'Limbic A':            '#B8956F',
    'Limbic B':            '#D6B58D',
    'Control A':           '#E08130',
    'Control B':           '#F4A35E',
    'Control C':           '#F9C18C',
    'Default A':           '#C73C3C',
    'Default B':           '#E36C6C',
    'Default C':           '#F49A9A',
    'Default D':           '#F9BCBC',
}

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(sorted_df))
colors = [network_colors.get(net, '#888888') for net in sorted_df['network']]
ax.barh(y_pos, sorted_df['univariate_r'], color=colors, edgecolor='black', linewidth=0.5)

# Add significance markers
for i, row in sorted_df.iterrows():
    if row['univariate_p'] < 0.001:
        marker = '***'
    elif row['univariate_p'] < 0.01:
        marker = '**'
    elif row['univariate_p'] < 0.05:
        marker = '*'
    else:
        marker = ''
    if marker:
        x_pos = row['univariate_r'] + (0.005 if row['univariate_r'] > 0 else -0.005)
        ax.text(x_pos, i, marker, va='center',
                ha='left' if row['univariate_r'] > 0 else 'right',
                fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_df['network'], fontsize=9)
ax.set_xlabel('Univariate Pearson r (TB parcel × dyscontrol_imputed)', fontsize=10)
ax.set_title('TB → dyscontrol: parcel-wise correlations', fontsize=11)
ax.axvline(0, color='black', linewidth=0.6)
ax.invert_yaxis()  # most positive at top

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_tb_dyscontrol_parcels_bar.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## A4. Probe validation

In [ ]:
probe_cols = [f'probe{i}' for i in range(1, 7)]
available_probes = [p for p in probe_cols if p in tb_blocks.columns]

probe_results = probe_decoder_correlations(
    tb_blocks, parcel_cols=parcel_cols,
    probe_cols=available_probes,
    target_col='dyscontrol_imputed',
    alpha=RIDGE_ALPHA,
)

print('Decoder predictions × probe ratings (within-subject Pearson r):')
print(f"{'probe':6s}  {'n':>3s}  {'mean r':>8s}  {'SEM':>6s}  "
       f"{'%pos':>5s}  {'t':>6s}  {'p':>7s}")
print('-' * 60)
for _, r in probe_results.iterrows():
    sig = ('***' if r['p_value'] < 0.001 else '**' if r['p_value'] < 0.01
            else '*' if r['p_value'] < 0.05 else '')
    print(f'{r["probe"]:6s}  {r["n_subj"]:>3d}  '
           f'{r["mean_r"]:>+8.3f}  {r["sem_r"]:>6.3f}  '
           f'{r["pct_positive"]:>4.0f}%  {r["t_stat"]:>+6.2f}  '
           f'{r["p_value"]:>7.4f}  {sig}')

probe_results.to_csv(DECODING_OUT_DIR / 'probe_validation.csv', index=False)

In [ ]:
# =============================================================================
# Figure: TB decoder predictions × probes (strip plot, per-subject)
# Re-runs the same logic as probe_decoder_correlations but keeps per-subject r
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from scipy.stats import pearsonr, ttest_1samp
from decoding import zscore_within_subject, center_within_subject

PROBES_TO_PLOT = ['probe1', 'probe2', 'probe3', 'probe4', 'probe5', 'probe6']
PROBE_LABELS = {
    'probe1': 'Probe 1:\nDyscontrol',
    'probe2': 'Probe 2:\nFocus',
    'probe3': 'Probe 3:\nEffort',
    'probe4': 'Probe 4:\nNegative Valence',
    'probe5': 'Probe 5:\nAnxiety',
    'probe6': 'Probe 6:\nArousal',
}

# Re-run LOSO predictions on TB blocks
df = tb_blocks.dropna(subset=['dyscontrol_imputed'] + parcel_cols).copy()
subjects = sorted(df['subject'].unique())

pred = pd.Series(index=df.index, dtype=float)
for held_out in subjects:
    train = df[df['subject'] != held_out]
    test = df[df['subject'] == held_out]
    if len(test) < 2:
        continue
    X_train = zscore_within_subject(
        train[parcel_cols].values.astype(float),
        train['subject'].values,
    )
    y_train = center_within_subject(
        train['dyscontrol_imputed'].values.astype(float),
        train['subject'].values,
    )
    X_test = test[parcel_cols].values.astype(float)
    mu_test = X_test.mean(axis=0)
    sd_test = X_test.std(axis=0)
    sd_test[sd_test == 0] = 1
    X_test = (X_test - mu_test) / sd_test
    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(X_train, y_train)
    pred.loc[test.index] = model.predict(X_test)

df = df.assign(decoder_pred=pred)

# Per-subject Pearson r for each probe
per_sub_rs = {probe: [] for probe in PROBES_TO_PLOT}
for probe in PROBES_TO_PLOT:
    if probe not in df.columns:
        continue
    for sub, g in df.groupby('subject'):
        valid = g.dropna(subset=['decoder_pred', probe])
        if len(valid) < 5 or valid['decoder_pred'].std() == 0:
            continue
        v = pd.to_numeric(valid[probe], errors='coerce')
        if v.std() == 0:
            continue
        r, _ = pearsonr(valid['decoder_pred'], v)
        if not np.isnan(r):
            per_sub_rs[probe].append(r)

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 5))

DOT_COLOR = '#7791a8'
MEAN_COLOR = '#1f3a5f'

x_positions = np.arange(len(PROBES_TO_PLOT))
y_max_for_text = 0.65

for i, probe in enumerate(PROBES_TO_PLOT):
    r_values = np.array(per_sub_rs[probe])
    
    np.random.seed(42 + i)
    jitter = np.random.uniform(-0.12, 0.12, size=len(r_values))
    x = x_positions[i] + jitter
    
    ax.scatter(x, r_values, s=24, color=DOT_COLOR, alpha=0.6,
                edgecolor='white', linewidth=0.5, zorder=2)
    
    mean_r = r_values.mean()
    se_r = r_values.std() / np.sqrt(len(r_values))
    ax.hlines(mean_r, x_positions[i] - 0.25, x_positions[i] + 0.25,
              color=MEAN_COLOR, linewidth=2.5, zorder=3)
    ax.vlines(x_positions[i], mean_r - se_r, mean_r + se_r,
              color=MEAN_COLOR, linewidth=2, zorder=3)
    
    t, p = ttest_1samp(r_values, 0)
    if p < 0.001:
        sig = '***'
    elif p < 0.01:
        sig = '**'
    elif p < 0.05:
        sig = '*'
    else:
        sig = 'n.s.'
    ax.text(x_positions[i], y_max_for_text, sig,
            ha='center', va='bottom', fontsize=11,
            color='black' if p < 0.05 else 'gray')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

ax.set_xticks(x_positions)
ax.set_xticklabels([PROBE_LABELS[p] for p in PROBES_TO_PLOT], fontsize=9)
ax.set_ylabel('Within-subject Pearson r\n(TB decoder predictions × probe)',
              fontsize=10)
ax.set_title('Thoughtblock-trained decoder: validation against in-scanner probes',
             fontsize=11)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='both', length=4)

plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_tb_decoder_probes_stripplot.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

---

# Section B: SART decoding

In [ ]:
sart_decode_df = sart_blocks.dropna(subset=parcel_cols).copy()
targets = ['dyscontrol_imputed'] + [f'probe{i}' for i in range(1, 7)]

sart_summary_rows = []
for target in targets:
    if target not in sart_decode_df.columns:
        continue
    df = sart_decode_df.copy()
    df[target] = pd.to_numeric(df[target], errors='coerce')
    df = df.dropna(subset=[target])
    if len(df) < 100:
        continue
    
    res = loso_decode(
        df, parcel_cols=parcel_cols,
        target_col=target,
        model_class=Ridge,
        model_kwargs={'alpha': RIDGE_ALPHA},
    )
    summary = summarize_loso(res, label=f'SART -> {target:18s}')
    sart_summary_rows.append(summary)
    res.to_csv(DECODING_OUT_DIR / f'sart_loso_{target}.csv', index=False)

sart_summary_df = pd.DataFrame(sart_summary_rows)
sart_summary_df.to_csv(DECODING_OUT_DIR / 'sart_loso_summary.csv', index=False)

## B2. Cross-pattern comparison

In [ ]:
sart_p1 = sart_decode_df.copy()
sart_p1['probe1'] = pd.to_numeric(sart_p1['probe1'], errors='coerce')
sart_p1 = sart_p1.dropna(subset=['probe1'])
sart_coef_p1 = parcel_coefficients(
    sart_p1, parcel_cols=parcel_cols,
    parcel_labels=YEO17_NAMES,
    target_col='probe1', alpha=RIDGE_ALPHA,
)

compare = pd.DataFrame({
    'parcel_idx': range(len(parcel_cols)),
    'network': YEO17_NAMES,
    'TB_dyscontrol_r': tb_coef['univariate_r'].values,
    'TB_dyscontrol_p': tb_coef['univariate_p'].values,
    'SART_probe1_r': sart_coef_p1['univariate_r'].values,
    'SART_probe1_p': sart_coef_p1['univariate_p'].values,
})
rho, p = pearsonr(compare['TB_dyscontrol_r'], compare['SART_probe1_r'])
print(f'Cross-correlation between TB(dyscontrol) and SART(probe1) parcel patterns:')
print(f'  Pearson r = {rho:+.3f}, p = {p:.4f}')
compare.to_csv(DECODING_OUT_DIR / 'tb_vs_sart_parcel_comparison.csv', index=False)

In [ ]:
sart_p2 = sart_decode_df.copy()
sart_p2['probe2'] = pd.to_numeric(sart_p2['probe2'], errors='coerce')
sart_p2 = sart_p2.dropna(subset=['probe2'])

sart_coef_p2 = parcel_coefficients(
    sart_p2, parcel_cols=parcel_cols,
    parcel_labels=YEO17_NAMES,
    target_col='probe2', alpha=RIDGE_ALPHA,
)

compare_p2 = pd.DataFrame({
    'parcel_idx': range(len(parcel_cols)),
    'network': YEO17_NAMES,
    'TB_dyscontrol_r': tb_coef['univariate_r'].values,
    'TB_dyscontrol_p': tb_coef['univariate_p'].values,
    'SART_probe2_r': sart_coef_p2['univariate_r'].values,
    'SART_probe2_p': sart_coef_p2['univariate_p'].values,
})
rho, p = pearsonr(compare_p2['TB_dyscontrol_r'], compare_p2['SART_probe2_r'])
print(f'Cross-correlation TB(dyscontrol) vs SART(probe2) parcel patterns:')
print(f'  Pearson r = {rho:+.3f}, p = {p:.4f}')
print()
print(compare_p2.round(4).to_string(index=False))
compare_p2.to_csv(DECODING_OUT_DIR / 'tb_vs_sart_probe2_comparison.csv', index=False)
sart_coef_p2.to_csv(DECODING_OUT_DIR / 'parcel_coefficients_sart_probe2.csv', index=False)

In [ ]:
# TB dyscontrol vs SART dyscontrol (same target across contexts)
sart_dys = sart_decode_df.copy()
sart_dys['dyscontrol_imputed'] = pd.to_numeric(sart_dys['dyscontrol_imputed'], errors='coerce')
sart_dys = sart_dys.dropna(subset=['dyscontrol_imputed'])

sart_coef_dys = parcel_coefficients(
    sart_dys, parcel_cols=parcel_cols,
    parcel_labels=YEO17_NAMES,
    target_col='dyscontrol_imputed', alpha=RIDGE_ALPHA,
)
rho, p = pearsonr(tb_coef['univariate_r'].values, sart_coef_dys['univariate_r'].values)
print(f'TB(dyscontrol) vs SART(dyscontrol): r={rho:+.3f}, p={p:.4f}')

In [ ]:
print(sart_coef_dys.sort_values('univariate_r', key=abs, ascending=False).round(4)
       .to_string(index=False))

---

# Section C: SART-Thoughtblock pattern similarity

All similarity uses **per-subject SART z-scoring**: every pattern is z-scored against that subject's mean and SD across SART blocks. The similarity then asks: when SART activity deviates from typical-SART, does that deviation point in the same direction as the TB-state's deviation from typical-SART?

## C0. Setup: SART z-scoring stats and unified analysis function

In [ ]:
def get_sart_block_ts(subject, run, onset_sec, duration_sec):
    """Load HRF-shifted parcel timeseries for one SART block."""
    run_str = f'{int(run):02d}'
    p_ts = ATLAS_DIR / f'sub-{subject}_run-{run_str}_parcel_ts.npy'
    p_mask = ATLAS_DIR / f'sub-{subject}_run-{run_str}_spike_mask.npy'
    if not p_ts.exists():
        return None, None
    parcel_ts = np.load(p_ts)
    spike_mask = np.load(p_mask)
    block_trs = get_block_indices(
        onset_sec, duration_sec, n_total_trs=len(parcel_ts),
        tr=TR_SECONDS, hrf_lag_trs=HRF_LAG_TRS,
    )
    if len(block_trs) == 0:
        return None, None
    return parcel_ts[block_trs], spike_mask[block_trs]


# Per-subject SART z-score statistics: mean and SD across the subject's SART
# block-mean patterns. Every pattern gets z-scored against this distribution
# before similarity is computed.
sart_stats = {}
for sub, g in sart_blocks.groupby('subject'):
    patterns = g[parcel_cols].values.astype(float)
    sd = patterns.std(axis=0)
    sd[sd == 0] = 1
    sart_stats[sub] = {'mean': patterns.mean(axis=0), 'sd': sd}
print(f'Computed SART z-score stats for {len(sart_stats)} subjects')


def zscore_against_sart(pattern, sub):
    """Z-score a pattern (1D or 2D) against this subject's SART distribution."""
    s = sart_stats[sub]
    return (pattern - s['mean']) / s['sd']


def analyze_template_similarity(template_builder, label):
    """
    Run both block-level and time-resolved similarity analyses.
    All patterns z-scored against the subject's SART block-distribution.
    """
    metric_fn = get_metric(SIMILARITY_METRIC)
    
    templates_z = {}
    for sub in sart_stats.keys():
        t = template_builder(sub)
        if t is None:
            continue
        templates_z[sub] = zscore_against_sart(t, sub)
    
    block_rows = []
    traj_rows = []
    summary_rows = []
    
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        if sub not in templates_z:
            continue
        template_z = templates_z[sub]
        
        # Block-mean similarity
        block_pattern = sart_row[parcel_cols].values.astype(float)
        block_z = zscore_against_sart(block_pattern, sub)
        block_sim = metric_fn(block_z, template_z)
        
        behavioral_cols = ['NoGo_ACC', 'AccIndex', 'Mean_go_RT', 'meanRT', 'TotalOutlierRT']
        common = {
            'subject': sub, 'run': sart_row['run'],
            'thoughtNumber': sart_row['thoughtNumber'],
            'dyscontrol_imputed': sart_row['dyscontrol_imputed'],
            **{p: sart_row.get(p, np.nan) for p in [f'probe{i}' for i in range(1,7)]},
            **{c: sart_row.get(c, np.nan) for c in behavioral_cols},
        }
        block_rows.append({**common, 'block_sim': block_sim})
        
        # Time-resolved similarity (sliding window)
        sart_ts, _ = get_sart_block_ts(
            sub, sart_row['run'], sart_row['onset'], sart_row['duration'],
        )
        if sart_ts is None or len(sart_ts) < SART_WINDOW_TRS:
            continue
        sart_z = zscore_against_sart(sart_ts, sub)
        
        centers, sims = sliding_window_similarity(
            sart_z, template_z,
            window_trs=SART_WINDOW_TRS, step=SART_WINDOW_STEP,
            metric=SIMILARITY_METRIC,
        )
        for c, s in zip(centers, sims):
            traj_rows.append({**common, 'window_center_tr': float(c), 'sim': float(s)})
        
        peak_tr = float(centers[int(np.argmax(sims))])
        slope = float(linregress(centers, sims).slope) if len(sims) >= 3 else np.nan
        summary_rows.append({
            **common, 'block_sim': block_sim,
            'mean_sim': float(sims.mean()), 'max_sim': float(sims.max()),
            'peak_tr': peak_tr, 'slope': slope, 'n_windows': len(sims),
        })
    
    block_df = pd.DataFrame(block_rows)
    traj_df = pd.DataFrame(traj_rows)
    summary_df = pd.DataFrame(summary_rows)
    
    print(f'{label}: {len(summary_df)} SART blocks (with timeseries available)')
    print(f'  block_sim:  M={summary_df["block_sim"].mean():+.3f}, '
          f'SD={summary_df["block_sim"].std():.3f}')
    print(f'  mean_sim:   M={summary_df["mean_sim"].mean():+.3f}, '
          f'SD={summary_df["mean_sim"].std():.3f}')
    print(f'  max_sim:    M={summary_df["max_sim"].mean():+.3f}, '
          f'SD={summary_df["max_sim"].std():.3f}')
    
    return block_df, traj_df, summary_df


def report_correlations(summary_df, label, sim_col='block_sim'):
    """Within-subject Pearson r between sim_col and each probe target."""
    print(f'\n{label} -- correlation between {sim_col!r} and target (within-subject):')
    print(f"{'target':18s}  {'mean r':>8s}  {'%pos':>5s}  {'n_subj':>6s}  "
          f"{'t':>6s}  {'p':>7s}")
    print('-' * 60)
    rows = []
    targets = (
        ['dyscontrol_imputed']
        + [f'probe{i}' for i in range(1, 7)]
        + ['NoGo_ACC', 'AccIndex', 'Mean_go_RT', 'meanRT', 'TotalOutlierRT']
    )
    for target in targets:
        if target not in summary_df.columns:
            continue
        rs = []
        for s, g in summary_df.groupby('subject'):
            valid = g.dropna(subset=[sim_col, target])
            if len(valid) < 5 or valid[sim_col].std() == 0:
                continue
            v = pd.to_numeric(valid[target], errors='coerce')
            if v.std() == 0:
                continue
            r, _ = pearsonr(valid[sim_col], v)
            if not np.isnan(r):
                rs.append(r)
        if not rs:
            continue
        rs = np.array(rs)
        t, p = ttest_1samp(rs, 0)
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
        print(f'{target:18s}  {rs.mean():>8.3f}  {(rs>0).mean()*100:>4.0f}%  '
              f'{len(rs):>6d}  {t:>6.2f}  {p:>7.4f}  {sig}')
        rows.append({
            'label': label, 'sim_col': sim_col, 'target': target,
            'mean_r': rs.mean(), 'sem_r': rs.std()/np.sqrt(len(rs)),
            'n_subj': len(rs), 'pct_pos': (rs > 0).mean() * 100,
            't_stat': t, 'p_value': p,
        })
    return rows

## C1. Same-trial template

In [ ]:
# Build a fast lookup: (subject, run, thoughtNumber) -> raw TB pattern
tb_lookup = {}
for _, r in tb_blocks.iterrows():
    tb_lookup[(r['subject'], r['run'], r['thoughtNumber'])] = (
        r[parcel_cols].values.astype(float)
    )

metric_fn = get_metric(SIMILARITY_METRIC)

block_rows_c1 = []
traj_rows_c1 = []
summary_rows_c1 = []
for _, sart_row in sart_blocks.iterrows():
    sub = sart_row['subject']
    run = sart_row['run']
    tn = sart_row['thoughtNumber']
    if sub not in sart_stats:
        continue
    tb_pat = tb_lookup.get((sub, run, tn))
    if tb_pat is None:
        continue
    
    template_z = zscore_against_sart(tb_pat, sub)
    
    block_pattern = sart_row[parcel_cols].values.astype(float)
    block_z = zscore_against_sart(block_pattern, sub)
    block_sim = metric_fn(block_z, template_z)
    
    behavioral_cols = ['NoGo_ACC', 'AccIndex', 'Mean_go_RT', 'meanRT', 'TotalOutlierRT']
    common = {
        'subject': sub, 'run': run, 'thoughtNumber': tn,
        'dyscontrol_imputed': sart_row['dyscontrol_imputed'],
        **{p: sart_row.get(p, np.nan) for p in [f'probe{i}' for i in range(1,7)]},
        **{c: sart_row.get(c, np.nan) for c in behavioral_cols},
    }
    block_rows_c1.append({**common, 'block_sim': block_sim})
    
    sart_ts, _ = get_sart_block_ts(sub, run, sart_row['onset'], sart_row['duration'])
    if sart_ts is None or len(sart_ts) < SART_WINDOW_TRS:
        continue
    sart_z = zscore_against_sart(sart_ts, sub)
    centers, sims = sliding_window_similarity(
        sart_z, template_z,
        window_trs=SART_WINDOW_TRS, step=SART_WINDOW_STEP,
        metric=SIMILARITY_METRIC,
    )
    for c, s in zip(centers, sims):
        traj_rows_c1.append({**common, 'window_center_tr': float(c), 'sim': float(s)})
    
    peak_tr = float(centers[int(np.argmax(sims))])
    slope = float(linregress(centers, sims).slope) if len(sims) >= 3 else np.nan
    summary_rows_c1.append({
        **common, 'block_sim': block_sim,
        'mean_sim': float(sims.mean()), 'max_sim': float(sims.max()),
        'peak_tr': peak_tr, 'slope': slope, 'n_windows': len(sims),
    })

summary_c1 = pd.DataFrame(summary_rows_c1)
traj_c1 = pd.DataFrame(traj_rows_c1)
print(f'C1 (same-trial): {len(summary_c1)} SART blocks')
print(f'  block_sim:  M={summary_c1["block_sim"].mean():+.3f}, '
      f'SD={summary_c1["block_sim"].std():.3f}')
print(f'  mean_sim:   M={summary_c1["mean_sim"].mean():+.3f}, '
      f'SD={summary_c1["mean_sim"].std():.3f}')

rows_c1 = []
for sim_col in ['block_sim', 'mean_sim', 'max_sim', 'peak_tr', 'slope']:
    rows_c1 += report_correlations(summary_c1, 'C1: same-trial', sim_col=sim_col)

summary_c1.to_csv(DECODING_OUT_DIR / 'sart_similarity_C1_summary.csv', index=False)
traj_c1.to_csv(DECODING_OUT_DIR / 'sart_similarity_C1_trajectory.csv', index=False)

## C2. Subject-mean Thoughtblock template

In [ ]:
tb_subject_mean = {}
for sub, g in tb_blocks.groupby('subject'):
    tb_subject_mean[sub] = g[parcel_cols].values.astype(float).mean(axis=0)

def template_builder_c2(sub):
    return tb_subject_mean.get(sub)

block_c2, traj_c2, summary_c2 = analyze_template_similarity(
    template_builder_c2, 'C2: subject-mean'
)

rows_c2 = []
for sim_col in ['block_sim', 'mean_sim', 'max_sim', 'peak_tr', 'slope']:
    rows_c2 += report_correlations(summary_c2, 'C2: subject-mean', sim_col=sim_col)

summary_c2.to_csv(DECODING_OUT_DIR / 'sart_similarity_C2_summary.csv', index=False)
traj_c2.to_csv(DECODING_OUT_DIR / 'sart_similarity_C2_trajectory.csv', index=False)

## C3. High-dyscontrol template

In [ ]:
tb_high_mean = {}
for sub, g in tb_blocks.groupby('subject'):
    if len(g) < 6:
        continue
    median = g['dyscontrol_imputed'].median()
    high = g[g['dyscontrol_imputed'] > median]
    if len(high) < 3:
        continue
    tb_high_mean[sub] = high[parcel_cols].values.astype(float).mean(axis=0)
print(f'Built high-dyscontrol templates for {len(tb_high_mean)} subjects')

def template_builder_c3(sub):
    return tb_high_mean.get(sub)

block_c3, traj_c3, summary_c3 = analyze_template_similarity(
    template_builder_c3, 'C3: high-dyscontrol'
)

rows_c3 = []
for sim_col in ['block_sim', 'mean_sim', 'max_sim', 'peak_tr', 'slope']:
    rows_c3 += report_correlations(summary_c3, 'C3: high-dyscontrol', sim_col=sim_col)

summary_c3.to_csv(DECODING_OUT_DIR / 'sart_similarity_C3_summary.csv', index=False)
traj_c3.to_csv(DECODING_OUT_DIR / 'sart_similarity_C3_trajectory.csv', index=False)

## C4. Group-level trajectories: similarity over time, split by probe1

For each template, plot mean SART-TB similarity at each window position, separately for trials in the top vs bottom half of probe1 (within each subject).

In [ ]:
from figure_style import setup_style, COLORS
setup_style()

def plot_trajectory_split(traj_df, title, split_col='AccIndex', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 3))
    df = traj_df.dropna(subset=[split_col, 'sim']).copy()
    df[split_col] = pd.to_numeric(df[split_col], errors='coerce')
    df = df.dropna(subset=[split_col])
    
    df['high'] = False
    for sub, g in df.groupby('subject'):
        med = g[split_col].median()
        df.loc[g.index, 'high'] = g[split_col] > med
    
    for high_label, group_color, group_name in [
        (False, COLORS['low'],  f'low {split_col}'),
        (True,  COLORS['high'], f'high {split_col}'),
    ]:
        subset = df[df['high'] == high_label]
        traj = subset.groupby('window_center_tr')['sim'].agg(['mean', 'sem']).reset_index()
        ax.plot(traj['window_center_tr'] * TR_SECONDS, traj['mean'],
                color=group_color, label=group_name, linewidth=1.5)
        ax.fill_between(traj['window_center_tr'] * TR_SECONDS,
                        traj['mean'] - traj['sem'], traj['mean'] + traj['sem'],
                        color=group_color, alpha=0.2, linewidth=0)
    ax.axhline(0, color='#bbbbbb', linewidth=0.5)
    ax.set_xlabel('Time within SART block (s)')
    ax.set_ylabel(f'Similarity ({SIMILARITY_METRIC})')
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
    return ax

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
plot_trajectory_split(traj_c1, 'C1: same-trial', split_col='AccIndex', ax=axes[0])
plot_trajectory_split(traj_c2, 'C2: subject-mean', split_col='AccIndex', ax=axes[1])
plot_trajectory_split(traj_c3, 'C3: high-dyscontrol', split_col='AccIndex', ax=axes[2])
fig.suptitle('SART-TB pattern similarity over time, split by AccIndex (within-subject median)',
              fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

## C5. Summary across templates and metrics

In [ ]:
all_rows = rows_c1 + rows_c2 + rows_c3
summary = pd.DataFrame(all_rows)

sig = summary[summary['p_value'] < 0.05].sort_values('p_value')
if len(sig) > 0:
    print(f'Significant (p<0.05) results across all template/metric/target combos:')
    print(sig[['label', 'sim_col', 'target', 'mean_r', 'p_value']].round(4)
          .to_string(index=False))
else:
    print('No template/metric/target combos reach p<0.05.')

summary.to_csv(DECODING_OUT_DIR / 'sart_similarity_summary.csv', index=False)
print(f'\nSaved full summary: {DECODING_OUT_DIR / "sart_similarity_summary.csv"}')

In [ ]:
# =============================================================================
# Section C (parallel): TB decoder -> SART blocks -> probe & metric coupling
#
# This is the decoder-based counterpart to the SART-TB *similarity* analysis.
# Instead of correlating each SART block's pattern with a thought-block
# template, we apply the leave-one-subject-out Thoughtblock-trained dyscontrol
# decoder to each SART block, obtain a predicted-dyscontrol value, and then
# correlate that prediction (within subject) with all six probes and the SART
# behavioral metrics -- exactly mirroring report_correlations() so the output
# table is directly comparable to the C1/C2/C3 similarity tables.
#
# Requires (all already defined earlier in this notebook):
#   tb_blocks, sart_blocks, parcel_cols, RIDGE_ALPHA, behavioral_cols,
#   report_correlations, zscore_within_subject, center_within_subject
# =============================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from decoding import zscore_within_subject, center_within_subject

# --- 1. Train LOSO TB-dyscontrol decoder and apply each held-out model to
#        that subject's SART blocks. -----------------------------------------
train_df = tb_blocks.dropna(subset=['dyscontrol_imputed'] + parcel_cols).copy()
sart_df  = sart_blocks.dropna(subset=parcel_cols).copy()
subjects = sorted(set(train_df['subject']) & set(sart_df['subject']))

sart_pred = pd.Series(index=sart_df.index, dtype=float)
for held_out in subjects:
    train = train_df[train_df['subject'] != held_out]
    test  = sart_df[sart_df['subject'] == held_out]
    if len(test) < 2 or len(train) < 2:
        continue

    # Same training idiom as the TB decoder validation cell:
    X_train = zscore_within_subject(
        train[parcel_cols].values.astype(float),
        train['subject'].values,
    )
    y_train = center_within_subject(
        train['dyscontrol_imputed'].values.astype(float),
        train['subject'].values,
    )

    # Standardize the held-out subject's SART blocks against their own SART
    # distribution (parallels zscore_against_sart used in the similarity path).
    X_test = test[parcel_cols].values.astype(float)
    mu = X_test.mean(axis=0)
    sd = X_test.std(axis=0); sd[sd == 0] = 1
    X_test = (X_test - mu) / sd

    model = Ridge(alpha=RIDGE_ALPHA)
    model.fit(X_train, y_train)
    sart_pred.loc[test.index] = model.predict(X_test)

# --- 2. Build a summary frame in the SAME shape report_correlations expects,
#        with the decoder prediction in a 'block_sim'-style column. ----------
summary_dec = sart_df.assign(decoder_pred=sart_pred).dropna(subset=['decoder_pred'])

# report_correlations groups by 'subject' and correlates `sim_col` with each
# target; pass sim_col='decoder_pred'.
print(f'Decoder applied to {len(summary_dec)} SART blocks '
      f'across {summary_dec["subject"].nunique()} subjects')

report_correlations(
    summary_dec,
    label='Decoder -> SART (TB-dyscontrol decoder applied to SART blocks)',
    sim_col='decoder_pred',
)

# --- 3. (Optional) save per-subject r's for each probe so they can be plotted
#        alongside the similarity strip plots / used in a supplementary table.
out_rows = []
targets = (['dyscontrol_imputed'] + [f'probe{i}' for i in range(1, 7)]
           + ['NoGo_ACC', 'AccIndex', 'Mean_go_RT', 'meanRT', 'TotalOutlierRT'])
from scipy.stats import pearsonr, ttest_1samp
for target in targets:
    if target not in summary_dec.columns:
        continue
    rs = []
    for sub, g in summary_dec.groupby('subject'):
        valid = g.dropna(subset=['decoder_pred', target])
        v = pd.to_numeric(valid[target], errors='coerce')
        if len(valid) < 5 or valid['decoder_pred'].std() == 0 or v.std() == 0:
            continue
        r, _ = pearsonr(valid['decoder_pred'], v)
        if not np.isnan(r):
            rs.append(r)
    if len(rs) >= 3:
        t, p = ttest_1samp(rs, 0.0)
        out_rows.append({'target': target, 'mean_r': np.mean(rs),
                         'n_subj': len(rs), 't': t, 'p': p})
decoder_sart_probe_summary = pd.DataFrame(out_rows)
decoder_sart_probe_summary.to_csv(
    DECODING_OUT_DIR / 'decoder_sart_probe_coupling.csv', index=False)
print('\nSaved: decoder_sart_probe_coupling.csv')
decoder_sart_probe_summary.round(4)

In [ ]:
# =============================================================================
# Bonferroni correction for the decoder -> SART -> probe coupling.
#
# Family = 7 outcomes: the dyscontrol rating (dyscontrol_imputed) + the 6 probes,
# matching the "Bonferroni across 7 outcomes" family used for the SART-trained
# decoder -> probe analysis. Behavioral metrics (NoGo_ACC, AccIndex, RTs) are
# reported separately and are NOT part of this family.
#
# Requires `decoder_sart_probe_summary` from the previous cell.
# =============================================================================
import numpy as np
import pandas as pd

# The 7-outcome probe family (order doesn't matter for Bonferroni)
PROBE_FAMILY = ['dyscontrol_imputed', 'probe1', 'probe2', 'probe3',
                'probe4', 'probe5', 'probe6']
N_FAMILY = len(PROBE_FAMILY)          # 7
ALPHA = 0.05 / N_FAMILY               # 0.00714...

PROBE_LABELS = {
    'dyscontrol_imputed': 'Dyscontrol (rating)',
    'probe1': 'Probe 1: Dyscontrol',
    'probe2': 'Probe 2: Focus',
    'probe3': 'Probe 3: Effort',
    'probe4': 'Probe 4: Negative Valence',
    'probe5': 'Probe 5: Anxiety',
    'probe6': 'Probe 6: Arousal',
}

fam = decoder_sart_probe_summary[
    decoder_sart_probe_summary['target'].isin(PROBE_FAMILY)
].copy()

# Bonferroni: p_corr = min(p * N, 1)
fam['p_corr'] = (fam['p'] * N_FAMILY).clip(upper=1.0)
fam['survives'] = fam['p'] < ALPHA
fam['label'] = fam['target'].map(PROBE_LABELS)

# Order the table the way the family is listed
fam['__order'] = fam['target'].map({t: k for k, t in enumerate(PROBE_FAMILY)})
fam = fam.sort_values('__order').drop(columns='__order')

print(f'Bonferroni family: {N_FAMILY} outcomes (dyscontrol rating + 6 probes), '
      f'alpha = {ALPHA:.5f}\n')
print(f"{'outcome':26s} {'r':>8s} {'t':>7s} {'p':>8s} {'p_corr':>8s}  survives")
print('-' * 70)
for _, row in fam.iterrows():
    print(f"{row['label']:26s} {row['mean_r']:>+8.3f} {row['t']:>7.2f} "
          f"{row['p']:>8.4f} {row['p_corr']:>8.4f}  "
          f"{'YES' if row['survives'] else 'no'}")

# Tidy frame for saving / pasting into a table
decoder_sart_probe_corrected = fam[
    ['target', 'label', 'mean_r', 'n_subj', 't', 'p', 'p_corr', 'survives']
].reset_index(drop=True)
decoder_sart_probe_corrected.to_csv(
    DECODING_OUT_DIR / 'decoder_sart_probe_coupling_bonferroni.csv', index=False)
print('\nSaved: decoder_sart_probe_coupling_bonferroni.csv')
decoder_sart_probe_corrected.round(4)

# Section D: Trial-level SART analysis

In [ ]:

# Test whether single SART trials are more likely to be errors when the
# brain pattern at that moment resembles the same-trial Thoughtblock template.
#
# For each SART trial:
#   - Find its onset, HRF-shift to TR window
#   - Compute single-TR similarity to the same-trial TB template (z-scored against
#     subject's SART distribution, same as Section C)
#   - Classify trial as correct or error based on response
#
# Two analyses:
#   D1: NoGo trials — within-subject t-test, similarity on commission errors vs
#       correct withholds
#   D2: Go trials — within-subject correlation between similarity and RT,
#       and t-test on omissions vs correct responses

from config import DATA_ROOT, SESSION, TASK

def load_events_tsv(subject, run):
    p = (DATA_ROOT / 'rawdata' / f'sub-{subject}' / SESSION / 'func'
         / f'sub-{subject}_{SESSION}_task-{TASK}_run-{int(run):02d}_events.tsv')
    if not p.exists():
        return None
    return pd.read_csv(p, sep='\t')


def classify_sart_trial(row):
    """Return 'correct' or 'error' for a SART trial. corr_ans encodes outcome."""
    if pd.isna(row['corr_ans']):
        return None
    if row['corr_ans'] == 1:
        return 'correct'
    elif row['corr_ans'] == 0:
        if row['trial_type'] == 'go':
            return 'omission'
        elif row['trial_type'] == 'nogo':
            return 'commission'
    return None


# Build trial-level dataframe with similarity at each trial's HRF-shifted TR
trial_rows = []
for _, sart_row in sart_blocks.iterrows():
    sub = sart_row['subject']
    run = sart_row['run']
    tn = sart_row['thoughtNumber']
    
    # Same-trial TB template (z-scored against SART distribution)
    if sub not in sart_stats:
        continue
    tb_pat = tb_lookup.get((sub, run, tn))
    if tb_pat is None:
        continue
    template_z = zscore_against_sart(tb_pat, sub)
    
    # Load full-run parcel timeseries
    run_str = f'{int(run):02d}'
    p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
    if not p_ts.exists():
        continue
    parcel_ts = np.load(p_ts)
    
    # Load events to get this run's SART trials
    events = load_events_tsv(sub, run)
    if events is None:
        continue
    
    sart_trials = events[
        (events['trial_type'].isin(['go', 'nogo']))
        & (events['thoughtNumber'] == tn)
    ]
    
    metric_fn = get_metric(SIMILARITY_METRIC)
    
    for _, trial in sart_trials.iterrows():
        # HRF-shifted TR for this trial
        trial_tr = int(round(trial['onset'] / TR_SECONDS)) + HRF_LAG_TRS
        if trial_tr < 0 or trial_tr >= len(parcel_ts):
            continue
        
        # Single-TR pattern, z-scored
        pattern_z = zscore_against_sart(parcel_ts[trial_tr], sub)
        sim = metric_fn(pattern_z, template_z)
        
        outcome = classify_sart_trial(trial)
        
        trial_rows.append({
            'subject': sub, 'run': run, 'thoughtNumber': tn,
            'trial_onset': trial['onset'],
            'trial_type': trial['trial_type'],
            'outcome': outcome,
            'rt': trial['response_time'],
            'sim': sim,
            'dyscontrol_imputed': sart_row['dyscontrol_imputed'],
        })

trial_df = pd.DataFrame(trial_rows)
print(f'Total SART trials with similarity: {len(trial_df)}')
print(f'Trial type x outcome:')
print(pd.crosstab(trial_df['trial_type'], trial_df['outcome']))

In [ ]:
# === D1: NoGo trials ===
nogo = trial_df[trial_df['trial_type'] == 'nogo'].copy()
nogo = nogo[nogo['outcome'].isin(['correct', 'commission'])]
print(f'\n=== D1: NoGo trials ===')
print(f'  Total nogo trials: {len(nogo)}')
print(f'  Outcome counts: {nogo["outcome"].value_counts().to_dict()}')

per_sub = nogo.groupby(['subject', 'outcome'])['sim'].mean().unstack()
# Keep only subjects with BOTH outcomes
per_sub = per_sub.dropna(subset=['correct', 'commission'], how='any') \
                 if {'correct', 'commission'}.issubset(per_sub.columns) \
                 else pd.DataFrame()

if len(per_sub) > 0:
    diff = per_sub['commission'] - per_sub['correct']
    t, p = ttest_1samp(diff, 0)
    print(f'  Subjects with both outcomes: {len(per_sub)}')
    print(f'  Within-subject diff (commission - correct): M={diff.mean():+.4f}, SD={diff.std():.4f}')
    print(f'  t({len(diff)-1}) = {t:.2f}, p = {p:.4f}, '
          f'%pos = {(diff > 0).sum()}/{len(diff)} ({100*(diff>0).mean():.0f}%)')
else:
    print('  Not enough data for D1 test')


# === D2a: Go trials, similarity vs RT ===
print('\n=== D2a: Go trials, similarity vs RT ===')
go_correct = trial_df[
    (trial_df['trial_type'] == 'go')
    & (trial_df['outcome'] == 'correct')
].dropna(subset=['rt', 'sim']).copy()

print(f'  Correct go trials with valid RT: {len(go_correct)}')
print(f'  Subjects: {go_correct["subject"].nunique()}')

rs = []
for sub, g in go_correct.groupby('subject'):
    if len(g) < 10:
        continue
    if g['rt'].std() == 0 or g['sim'].std() == 0:
        continue
    r, _ = pearsonr(g['sim'], g['rt'])
    if not np.isnan(r):
        rs.append(r)

if len(rs) >= 5:
    rs = np.array(rs)
    t, p = ttest_1samp(rs, 0)
    print(f'  n_subj with sufficient trials = {len(rs)}')
    print(f'  Within-subject Pearson r (sim vs RT): M = {rs.mean():+.4f}, '
          f't = {t:.2f}, p = {p:.4f}, %pos = {(rs>0).mean()*100:.0f}%')
else:
    print(f'  Only {len(rs)} subjects had sufficient trials, skipping')


# === D2b: Go trials, omissions vs correct ===
print('\n=== D2b: Go trials (omissions vs correct) ===')
go = trial_df[
    (trial_df['trial_type'] == 'go')
    & (trial_df['outcome'].isin(['correct', 'omission']))
].copy()
print(f'  Outcome counts: {go["outcome"].value_counts().to_dict()}')

per_sub_go = go.groupby(['subject', 'outcome'])['sim'].mean().unstack()
if {'correct', 'omission'}.issubset(per_sub_go.columns):
    per_sub_go = per_sub_go.dropna(subset=['correct', 'omission'], how='any')
    if len(per_sub_go) > 0:
        diff_go = per_sub_go['omission'] - per_sub_go['correct']
        t, p = ttest_1samp(diff_go, 0)
        print(f'  Subjects with both outcomes: {len(per_sub_go)}')
        print(f'  Within-subject diff (omission - correct): '
              f'M={diff_go.mean():+.4f}, SD={diff_go.std():.4f}')
        print(f'  t({len(diff_go)-1}) = {t:.2f}, p = {p:.4f}, '
              f'%pos = {(diff_go > 0).sum()}/{len(diff_go)} ({100*(diff_go>0).mean():.0f}%)')
    else:
        print('  No subjects had both outcomes')
else:
    print(f'  Missing columns. Available: {list(per_sub_go.columns)}')

In [ ]:
from scipy.stats import ttest_1samp

# Sweep HRF lag and window width for D1 (nogo commission errors vs correct)
def trial_level_d1(hrf_lag_trs, window_trs):
    """Recompute D1 at given HRF lag and window width."""
    metric_fn = get_metric(SIMILARITY_METRIC)
    rows = []
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in sart_stats:
            continue
        tb_pat = tb_lookup.get((sub, run, tn))
        if tb_pat is None:
            continue
        template_z = zscore_against_sart(tb_pat, sub)

        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)

        events = load_events_tsv(sub, run)
        if events is None:
            continue
        nogo_trials = events[
            (events['trial_type'] == 'nogo')
            & (events['thoughtNumber'] == tn)
        ]
        for _, trial in nogo_trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'correct' if trial['corr_ans'] == 1 else 'commission'
            
            start_tr = int(round(trial['onset'] / TR_SECONDS)) + hrf_lag_trs
            end_tr = start_tr + window_trs
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue
            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)
            sim = metric_fn(pattern_z, template_z)
            rows.append({
                'subject': sub, 'outcome': outcome, 'sim': sim,
            })
    
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return None
    per_sub = df.groupby(['subject', 'outcome'])['sim'].mean().unstack()
    if not {'correct', 'commission'}.issubset(per_sub.columns):
        return None
    per_sub = per_sub.dropna(subset=['correct', 'commission'], how='any')
    if len(per_sub) < 10:
        return None
    diff = per_sub['commission'] - per_sub['correct']
    t, p = ttest_1samp(diff, 0)
    return {
        'hrf_lag_trs': hrf_lag_trs, 'window_trs': window_trs,
        'n_subj': len(diff), 'mean_diff': diff.mean(),
        't': t, 'p': p, 'pct_pos': 100 * (diff > 0).mean(),
    }

# Sweep
sweep_results = []
for lag in [1, 2, 3, 4, 5, 6]:
    for window in [1, 2, 3, 4, 5]:
        result = trial_level_d1(lag, window)
        if result is not None:
            sweep_results.append(result)
            print(f'  lag={lag} TR ({lag*TR_SECONDS:.0f}s), window={window} TR '
                  f'({window*TR_SECONDS:.0f}s): '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, '
                  f'diff={result["mean_diff"]:+.4f}')

sweep_df = pd.DataFrame(sweep_results)
print('\nMost extreme p-value:')
print(sweep_df.iloc[sweep_df['p'].idxmin()].to_string())

In [ ]:
for lag in [0, 1]:
    for window in [1, 2]:
        result = trial_level_d1(lag, window)
        if result is not None:
            print(f'  lag={lag} TR ({lag*TR_SECONDS:.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, diff={result["mean_diff"]:+.4f}')

In [ ]:
# Try testing even earlier — using the TR BEFORE trial onset
for lag in [-2, -1, 0]:
    for window in [1, 2, 3]:
        result = trial_level_d1(lag, window)
        if result is not None:
            print(f'  lag={lag} TR ({lag*TR_SECONDS:.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, diff={result["mean_diff"]:+.4f}')

In [ ]:
def trial_level_d1_template(template_builder, hrf_lag_trs, window_trs):
    """
    Trial-level D1 test (commission errors vs correct nogo) with arbitrary
    template builder and timing.
    
    template_builder(subject) -> raw template pattern, or None to skip
    """
    metric_fn = get_metric(SIMILARITY_METRIC)
    
    # Build z-scored templates
    templates_z = {}
    for sub in sart_stats.keys():
        t = template_builder(sub)
        if t is None:
            continue
        templates_z[sub] = zscore_against_sart(t, sub)
    
    rows = []
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in templates_z:
            continue
        template_z = templates_z[sub]
        
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)
        
        events = load_events_tsv(sub, run)
        if events is None:
            continue
        nogo_trials = events[
            (events['trial_type'] == 'nogo')
            & (events['thoughtNumber'] == tn)
        ]
        for _, trial in nogo_trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'correct' if trial['corr_ans'] == 1 else 'commission'
            
            start_tr = int(round(trial['onset'] / TR_SECONDS)) + hrf_lag_trs
            end_tr = start_tr + window_trs
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue
            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)
            sim = metric_fn(pattern_z, template_z)
            rows.append({'subject': sub, 'outcome': outcome, 'sim': sim})
    
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return None
    per_sub = df.groupby(['subject', 'outcome'])['sim'].mean().unstack()
    if not {'correct', 'commission'}.issubset(per_sub.columns):
        return None
    per_sub = per_sub.dropna(subset=['correct', 'commission'], how='any')
    if len(per_sub) < 10:
        return None
    diff = per_sub['commission'] - per_sub['correct']
    t, p = ttest_1samp(diff, 0)
    return {
        'hrf_lag_trs': hrf_lag_trs, 'window_trs': window_trs,
        'n_subj': len(diff), 'mean_diff': diff.mean(),
        't': t, 'p': p,
    }


# Sweep C2 (subject-mean template)
print('=== C2: subject-mean template, trial-level sweep ===')
sweep_c2 = []
for lag in range(-3, 4):
    for window in [1, 2, 3]:
        result = trial_level_d1_template(template_builder_c2, lag, window)
        if result is not None:
            sweep_c2.append({**result, 'template': 'C2'})
            print(f'  lag={lag:+d} TR ({lag*TR_SECONDS:+.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, diff={result["mean_diff"]:+.4f}')


# Sweep C3 (high-dyscontrol template)
print('\n=== C3: high-dyscontrol template, trial-level sweep ===')
sweep_c3 = []
for lag in range(-3, 4):
    for window in [1, 2, 3]:
        result = trial_level_d1_template(template_builder_c3, lag, window)
        if result is not None:
            sweep_c3.append({**result, 'template': 'C3'})
            print(f'  lag={lag:+d} TR ({lag*TR_SECONDS:+.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, diff={result["mean_diff"]:+.4f}')

# Save
pd.DataFrame(sweep_c2 + sweep_c3).to_csv(
    DECODING_OUT_DIR / 'trial_level_sweep_C2_C3.csv', index=False
)

In [ ]:
# Re-run C1 across full lag range for plotting (consolidates pre and post)
print('=== C1 full sweep for figure ===')
sweep_c1_full = []
for lag in range(-3, 4):
    for window in [1, 2, 3]:
        result = trial_level_d1(lag, window)
        if result is not None:
            sweep_c1_full.append({**result, 'template': 'C1'})
            print(f'  lag={lag:+d} TR ({lag*TR_SECONDS:+.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, diff={result["mean_diff"]:+.4f}')

sweep_c1_df = pd.DataFrame(sweep_c1_full)
all_sweeps = pd.concat([sweep_c1_df, pd.DataFrame(sweep_c2), pd.DataFrame(sweep_c3)],
                        ignore_index=True)
all_sweeps.to_csv(DECODING_OUT_DIR / 'trial_level_sweep_all.csv', index=False)
print(f'Saved unified sweep to {DECODING_OUT_DIR / "trial_level_sweep_all.csv"}')

In [ ]:
# =============================================================================
# Visualize: pre-error temporal profile of worry-pattern reactivation
# =============================================================================
# Shows the t-statistic from each template's commission-vs-correct test, 
# plotted as a function of HRF lag relative to trial onset. Pre-trial = 
# negative lags, post-trial = positive lags. Window width fixed at 2 TR.

from figure_style import setup_style, COLORS, save_fig

setup_style()

# Pull the C1, C2, C3 sweep data into one DataFrame.
# (sweep_results is C1 from earlier; sweep_c2 and sweep_c3 are above.)
sweep_c1 = pd.DataFrame(sweep_results).assign(template='C1')
sweep_c2_df = pd.DataFrame(sweep_c2)
sweep_c3_df = pd.DataFrame(sweep_c3)
#all_sweeps = pd.concat([sweep_c1, sweep_c2_df, sweep_c3_df], ignore_index=True)

# Pick window=2 for comparable curves
WINDOW = 2
plot_df = all_sweeps[all_sweeps['window_trs'] == WINDOW].copy()

template_styles = {
    'C1': {'label': 'Same-trial',     'color': '#C73C3C', 'lw': 2.0},
    'C2': {'label': 'Mean Thought',   'color': '#2E5EAA', 'lw': 1.5},
    'C3': {'label': 'High-dyscontrol','color': '#888888', 'lw': 1.5},
}

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.0))

# Panel a: t-statistic vs lag
ax = axes[0]
for tmpl in ['C2', 'C3', 'C1']:  # C1 last so it's drawn on top
    sub = plot_df[plot_df['template'] == tmpl].sort_values('hrf_lag_trs')
    if len(sub) == 0:
        continue
    style = template_styles[tmpl]
    ax.plot(sub['hrf_lag_trs'] * TR_SECONDS, sub['t'],
            color=style['color'], lw=style['lw'], label=style['label'],
            marker='o', markersize=4)
ax.axhline(0, color='#bbbbbb', lw=0.5)
ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.5)
# Mark p < 0.05 thresholds (two-tailed t for ~55 df)
ax.axhline(2.005, color='#cccccc', lw=0.5, linestyle=':', zorder=0)
ax.axhline(-2.005, color='#cccccc', lw=0.5, linestyle=':', zorder=0)
ax.text(ax.get_xlim()[1], 2.05, ' p<.05', fontsize=7, color='#888888',
        va='bottom')
ax.text(ax.get_xlim()[1], -2.05, ' p<.05', fontsize=7, color='#888888',
        va='top')
ax.set_xlabel('HRF lag relative to trial onset (s)')
ax.set_ylabel('t-statistic\n(commission − correct)')
ax.set_title('a. Worry-pattern reactivation around commission errors')
ax.legend(frameon=False, loc='upper right', fontsize=8)

# Panel b: signed mean difference (effect size, more interpretable scale)
ax = axes[1]
for tmpl in ['C2', 'C3', 'C1']:
    sub = plot_df[plot_df['template'] == tmpl].sort_values('hrf_lag_trs')
    if len(sub) == 0:
        continue
    style = template_styles[tmpl]
    ax.plot(sub['hrf_lag_trs'] * TR_SECONDS, sub['mean_diff'],
            color=style['color'], lw=style['lw'], label=style['label'],
            marker='o', markersize=4)
ax.axhline(0, color='#bbbbbb', lw=0.5)
ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.5)
ax.set_xlabel('HRF lag relative to trial onset (s)')
ax.set_ylabel('Mean similarity difference\n(commission − correct)')
ax.set_title('b. Effect size (window = '
             f'{WINDOW} TR / {WINDOW*TR_SECONDS:.0f}s)')

plt.tight_layout()
save_fig(fig, 'fig_trial_level_worry_reactivation')
plt.show()

In [ ]:
# =============================================================================
# Peristimulus similarity around each trial outcome
# =============================================================================
# For each SART trial outcome (hit, miss, commission, correct withhold),
# extract the parcel timeseries window around trial onset (-3 TR to +6 TR),
# compute similarity to the same-trial TB template at each TR within the window,
# average within outcome.

PERI_PRE_TRS = 3   # number of TRs before trial onset
PERI_POST_TRS = 6  # number of TRs after trial onset
PERI_TRS = list(range(-PERI_PRE_TRS, PERI_POST_TRS + 1))

# For C1 (same-trial template), we need the trial-by-trial TB pattern
metric_fn = get_metric(SIMILARITY_METRIC)

peri_rows = []
for _, sart_row in sart_blocks.iterrows():
    sub = sart_row['subject']
    run = sart_row['run']
    tn = sart_row['thoughtNumber']
    
    if sub not in sart_stats:
        continue
    tb_pat = tb_lookup.get((sub, run, tn))
    if tb_pat is None:
        continue
    template_z = zscore_against_sart(tb_pat, sub)
    
    run_str = f'{int(run):02d}'
    p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
    if not p_ts.exists():
        continue
    parcel_ts = np.load(p_ts)
    
    events = load_events_tsv(sub, run)
    if events is None:
        continue
    sart_trials = events[
        (events['trial_type'].isin(['go', 'nogo']))
        & (events['thoughtNumber'] == tn)
    ]
    
    for _, trial in sart_trials.iterrows():
        outcome = classify_sart_trial(trial)
        if outcome is None:
            continue
        # Map go-correct -> 'hit', go-omission stays as 'omission'
        if trial['trial_type'] == 'go' and outcome == 'correct':
            label = 'hit'
        elif trial['trial_type'] == 'go' and outcome == 'omission':
            label = 'miss'
        elif trial['trial_type'] == 'nogo' and outcome == 'correct':
            label = 'correct_withhold'
        elif trial['trial_type'] == 'nogo' and outcome == 'commission':
            label = 'commission'
        else:
            continue
        
        center_tr = int(round(trial['onset'] / TR_SECONDS))
        for offset in PERI_TRS:
            tr = center_tr + offset
            if tr < 0 or tr >= len(parcel_ts):
                continue
            pattern_z = zscore_against_sart(parcel_ts[tr], sub)
            sim = metric_fn(pattern_z, template_z)
            peri_rows.append({
                'subject': sub, 'outcome': label,
                'tr_offset': offset, 'sim': sim,
            })

peri_df = pd.DataFrame(peri_rows)
print(f'Peristimulus dataframe: {len(peri_df)} rows')
print(f'Trial counts by outcome:')
print(peri_df.groupby(['outcome', 'tr_offset']).size().unstack().iloc[:, 0])

In [ ]:
# Per-subject means, then SE across subjects
per_sub_means = (peri_df.groupby(['subject', 'outcome', 'tr_offset'])['sim']
                 .mean().reset_index())

agg = (per_sub_means.groupby(['outcome', 'tr_offset'])['sim']
       .agg(['mean', 'sem', 'count']).reset_index())

# Plot
from figure_style import setup_style, save_fig
setup_style()

outcome_styles = {
    'hit':              {'color': '#2E5EAA', 'lw': 1.5,
                         'label': 'Go, correct (hit)'},
    'miss':             {'color': '#88AADD', 'lw': 1.5,
                         'label': 'Go, miss (omission)'},
    'correct_withhold': {'color': '#888888', 'lw': 1.5,
                         'label': 'NoGo, correct withhold'},
    'commission':       {'color': '#C73C3C', 'lw': 2.0,
                         'label': 'NoGo, error (commission)'},
}

fig, ax = plt.subplots(figsize=(5.5, 3.5))

for outcome, style in outcome_styles.items():
    sub = agg[agg['outcome'] == outcome].sort_values('tr_offset')
    if len(sub) == 0:
        continue
    x = sub['tr_offset'] * TR_SECONDS
    ax.plot(x, sub['mean'], color=style['color'], lw=style['lw'],
            label=style['label'], marker='o', markersize=3)
    ax.fill_between(x, sub['mean'] - sub['sem'], sub['mean'] + sub['sem'],
                    color=style['color'], alpha=0.2, linewidth=0)

ax.axhline(0, color='#bbbbbb', lw=0.5)
ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.7)
ax.text(0, ax.get_ylim()[1], 'trial onset', fontsize=7,
        ha='center', va='bottom', color='#888888')
ax.set_xlabel('Time relative to trial onset (s)')
ax.set_ylabel(f'Similarity to same-trial TB template ({SIMILARITY_METRIC})')
ax.set_title('Similarity to thoughtblock')
ax.legend(frameon=False, fontsize=8, loc='best')
ymin = agg['mean'].min() - 0.02
ymax = agg['mean'].max() + 0.02
ax.set_ylim(ymin, ymax)
plt.tight_layout()
save_fig(fig, 'fig_peristimulus_by_outcome')
plt.show()

In [ ]:
# =============================================================================
# Build peristimulus dataframe for ALL three templates
# =============================================================================
PERI_PRE_TRS = 3
PERI_POST_TRS = 6
PERI_TRS = list(range(-PERI_PRE_TRS, PERI_POST_TRS + 1))

metric_fn = get_metric(SIMILARITY_METRIC)

# Pre-build C2 and C3 z-scored templates
templates_z_c2 = {sub: zscore_against_sart(template_builder_c2(sub), sub)
                   for sub in sart_stats if template_builder_c2(sub) is not None}
templates_z_c3 = {sub: zscore_against_sart(template_builder_c3(sub), sub)
                   for sub in sart_stats if template_builder_c3(sub) is not None}

peri_rows = []
for _, sart_row in sart_blocks.iterrows():
    sub = sart_row['subject']
    run = sart_row['run']
    tn = sart_row['thoughtNumber']
    
    if sub not in sart_stats:
        continue
    
    # Build C1 template (same-trial) and look up C2/C3 templates
    tb_pat = tb_lookup.get((sub, run, tn))
    if tb_pat is None:
        continue
    template_c1 = zscore_against_sart(tb_pat, sub)
    template_c2 = templates_z_c2.get(sub)
    template_c3 = templates_z_c3.get(sub)
    
    run_str = f'{int(run):02d}'
    p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
    if not p_ts.exists():
        continue
    parcel_ts = np.load(p_ts)
    
    events = load_events_tsv(sub, run)
    if events is None:
        continue
    sart_trials = events[
        (events['trial_type'].isin(['go', 'nogo']))
        & (events['thoughtNumber'] == tn)
    ]
    
    for _, trial in sart_trials.iterrows():
        outcome = classify_sart_trial(trial)
        if outcome is None:
            continue
        if trial['trial_type'] == 'go' and outcome == 'correct':
            label = 'hit'
        elif trial['trial_type'] == 'go' and outcome == 'omission':
            label = 'miss'
        elif trial['trial_type'] == 'nogo' and outcome == 'correct':
            label = 'correct_withhold'
        elif trial['trial_type'] == 'nogo' and outcome == 'commission':
            label = 'commission'
        else:
            continue
        
        center_tr = int(round(trial['onset'] / TR_SECONDS))
        for offset in PERI_TRS:
            tr = center_tr + offset
            if tr < 0 or tr >= len(parcel_ts):
                continue
            pattern_z = zscore_against_sart(parcel_ts[tr], sub)
            row_base = {'subject': sub, 'outcome': label, 'tr_offset': offset}
            row_base['sim_C1'] = metric_fn(pattern_z, template_c1)
            if template_c2 is not None:
                row_base['sim_C2'] = metric_fn(pattern_z, template_c2)
            if template_c3 is not None:
                row_base['sim_C3'] = metric_fn(pattern_z, template_c3)
            peri_rows.append(row_base)

peri_all_df = pd.DataFrame(peri_rows)
print(f'Peristimulus dataframe: {len(peri_all_df)} rows')
print(f'Trial counts by outcome:')
print(peri_all_df.groupby('outcome')['subject'].count())

peri_all_df.to_csv(DECODING_OUT_DIR / 'peristimulus_all_templates.csv', index=False)

In [ ]:
# =============================================================================
# Three-panel peristimulus figure: subplots by template, lines by outcome
# =============================================================================
from figure_style import setup_style, save_fig
setup_style()

outcome_styles = {
    'hit':              {'color': '#2E5EAA', 'lw': 1.5,
                         'label': 'Go, correct (hit)'},
    'miss':             {'color': '#88AADD', 'lw': 1.5,
                         'label': 'Go, miss (omission)'},
    'correct_withhold': {'color': '#888888', 'lw': 1.5,
                         'label': 'NoGo, correct withhold'},
    'commission':       {'color': '#C73C3C', 'lw': 2.0,
                         'label': 'NoGo, error (commission)'},
}

template_titles = {
    'C1': 'a. Same-trial template',
    'C2': 'b. Mean thought template',
    'C3': 'c. High-dyscontrol template',
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)

for ax, tmpl in zip(axes, ['C1', 'C2', 'C3']):
    sim_col = f'sim_{tmpl}'
    if sim_col not in peri_all_df.columns:
        continue
    
    for outcome, style in outcome_styles.items():
        sub_df = peri_all_df[peri_all_df['outcome'] == outcome]
        if len(sub_df) == 0:
            continue
        per_sub = (sub_df.groupby(['subject', 'tr_offset'])[sim_col]
                    .mean().reset_index())
        agg = (per_sub.groupby('tr_offset')[sim_col]
               .agg(['mean', 'sem']).reset_index().sort_values('tr_offset'))
        x = agg['tr_offset'] * TR_SECONDS
        ax.plot(x, agg['mean'], color=style['color'], lw=style['lw'],
                label=style['label'], marker='o', markersize=3)
        ax.fill_between(x, agg['mean'] - agg['sem'], agg['mean'] + agg['sem'],
                        color=style['color'], alpha=0.2, linewidth=0)
    
    ax.axhline(0, color='#bbbbbb', lw=0.5)
    ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.7)
    ax.set_xlabel('Time relative to trial onset (s)')
    ax.set_title(template_titles[tmpl], fontsize=10, loc='left')
    
    # Tighten y-axis to data range with small margin
    ymin = (agg['mean'] - agg['sem']).min()
    ymax = (agg['mean'] + agg['sem']).max()
    
    # Re-collect across all outcomes for this panel
    panel_data = peri_all_df[peri_all_df['outcome'].isin(outcome_styles.keys())]
    panel_per_sub = (panel_data.groupby(['subject', 'outcome', 'tr_offset'])[sim_col]
                      .mean().reset_index())
    panel_agg = (panel_per_sub.groupby(['outcome', 'tr_offset'])[sim_col]
                 .agg(['mean', 'sem']).reset_index())
    pad = 0.02 * (panel_agg['mean'].max() - panel_agg['mean'].min())
    ax.set_ylim(
        (panel_agg['mean'] - panel_agg['sem']).min() - pad,
        (panel_agg['mean'] + panel_agg['sem']).max() + pad,
    )
    
    if tmpl == 'C1':
        ax.set_ylabel(f'Similarity ({SIMILARITY_METRIC})')
        ax.legend(frameon=False, fontsize=7, loc='best')

plt.tight_layout()
save_fig(fig, 'fig_peristimulus_3panel_by_template')
plt.show()

In [ ]:
# =============================================================================
# Three-panel peristimulus figure: subplots by template, lines by outcome
# Significance markers placed on error lines (commission, miss) at TRs where
# they differ from their respective correct comparison.
# =============================================================================
from figure_style import setup_style, save_fig
from scipy.stats import ttest_1samp

setup_style()

outcome_styles = {
    'hit':              {'color': '#2E5EAA', 'lw': 1.5,
                         'label': 'Go, correct (hit)'},
    'miss':             {'color': '#88AADD', 'lw': 1.5,
                         'label': 'Go, miss (omission)'},
    'correct_withhold': {'color': '#888888', 'lw': 1.5,
                         'label': 'NoGo, correct withhold'},
    'commission':       {'color': '#C73C3C', 'lw': 2.0,
                         'label': 'NoGo, error (commission)'},
}

template_titles = {
    'C1': 'a. Same-trial template',
    'C2': 'b. Mean thought template',
    'C3': 'c. High-dyscontrol template',
}

# Significance contrasts: error vs correct, within each trial type
contrasts = [
    ('commission', 'correct_withhold'),
    ('miss',       'hit'),
]

n_trs = len(PERI_TRS)
alpha_corrected = 0.05 / n_trs

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)

for ax, tmpl in zip(axes, ['C1', 'C2', 'C3']):
    sim_col = f'sim_{tmpl}'
    if sim_col not in peri_all_df.columns:
        continue

    # --- Plot one line per outcome ---
    line_aggs = {}  # store per-outcome agg DataFrames for sig-marker placement
    for outcome, style in outcome_styles.items():
        sub_df = peri_all_df[peri_all_df['outcome'] == outcome]
        if len(sub_df) == 0:
            continue
        per_sub = (sub_df.groupby(['subject', 'tr_offset'])[sim_col]
                    .mean().reset_index())
        agg = (per_sub.groupby('tr_offset')[sim_col]
               .agg(['mean', 'sem']).reset_index().sort_values('tr_offset'))
        line_aggs[outcome] = agg
        x = agg['tr_offset'] * TR_SECONDS
        ax.plot(x, agg['mean'], color=style['color'], lw=style['lw'],
                label=style['label'], marker='o', markersize=3)
        ax.fill_between(x, agg['mean'] - agg['sem'], agg['mean'] + agg['sem'],
                        color=style['color'], alpha=0.2, linewidth=0)

    # --- Significance markers: error vs correct, paired across subjects ---
    per_sub_long = (peri_all_df.groupby(['subject', 'outcome', 'tr_offset'])[sim_col]
                     .mean().reset_index())
    wide = per_sub_long.pivot_table(
        index=['subject', 'tr_offset'], columns='outcome', values=sim_col
    ).reset_index()

    for err_label, ok_label in contrasts:
        if not {err_label, ok_label}.issubset(wide.columns):
            continue
        contrast_df = wide.dropna(subset=[err_label, ok_label]).copy()
        contrast_df['diff'] = contrast_df[err_label] - contrast_df[ok_label]

        style = outcome_styles[err_label]
        err_agg = line_aggs.get(err_label)
        if err_agg is None:
            continue

        for tr_offset, g in contrast_df.groupby('tr_offset'):
            diffs = g['diff'].dropna()
            if len(diffs) < 5:
                continue
            _, p = ttest_1samp(diffs, 0)
            if p >= 0.05:
                continue
            y_at_tr = err_agg[err_agg['tr_offset'] == tr_offset]['mean'].values
            if len(y_at_tr) == 0:
                continue
            x_pos = tr_offset * TR_SECONDS
            y_pos = y_at_tr[0]
            if p < alpha_corrected:
                ax.plot(x_pos, y_pos, 'o', markersize=8,
                        markeredgecolor=style['color'],
                        markerfacecolor=style['color'],
                        markeredgewidth=1.5, zorder=10)
            else:
                ax.plot(x_pos, y_pos, 'o', markersize=8,
                        markeredgecolor=style['color'],
                        markerfacecolor='white',
                        markeredgewidth=1.5, zorder=10)

    # --- Axis cosmetics ---
    ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.7)
    ax.set_xlabel('Time relative to trial onset (s)')
    ax.set_title(template_titles[tmpl], fontsize=10, loc='left')
    if tmpl != 'C1':
        ax.axhline(0, color='#bbbbbb', lw=0.5)
        ax.set_ylabel(f'Similarity ({SIMILARITY_METRIC})')
    else:
        ax.set_ylabel(f'Similarity ({SIMILARITY_METRIC})')

# C1 needs a tighter y-axis since absolute similarities are high
#axes[0].set_ylim(0.46, 0.72)

# Legend on first panel only
axes[0].legend(frameon=False, fontsize=7, loc='lower right')

# Footer for marker meanings
fig.text(0.5, -0.04,
         '○ p < 0.05 vs correct (uncorrected)    '
         f'● p < 0.05 vs correct (Bonferroni-corrected across {n_trs} TRs)',
         ha='center', fontsize=8, color='#555555')

plt.tight_layout()
save_fig(fig, 'fig_peristimulus_3panel_with_sig')
plt.show()

In [ ]:
# =============================================================================
# Six-panel peristimulus figure: omissions on top, commissions on bottom,
# columns by template. Cleaner than 4-line panels — each panel is a 2-line
# error-vs-correct contrast.
# =============================================================================
from figure_style import setup_style, save_fig
from scipy.stats import ttest_1samp

setup_style()

outcome_styles = {
    'hit':              {'color': '#2E5EAA', 'lw': 1.5,
                         'label': 'Go, correct (hit)'},
    'miss':             {'color': '#88AADD', 'lw': 2.0,
                         'label': 'Go, miss (omission)'},
    'correct_withhold': {'color': '#888888', 'lw': 1.5,
                         'label': 'NoGo, correct withhold'},
    'commission':       {'color': '#C73C3C', 'lw': 2.0,
                         'label': 'NoGo, error (commission)'},
}

template_titles = {
    'C1': 'a. Same-trial template',
    'C2': 'b. Mean thought template',
    'C3': 'c. High-dyscontrol template',
}

# Each row: (error_outcome, correct_outcome, row_label)
rows = [
    ('miss',       'hit',              'Omissions'),
    ('commission', 'correct_withhold', 'Commissions'),
]

n_trs = len(PERI_TRS)
alpha_corrected = 0.05 / n_trs

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)

for row_idx, (err_label, ok_label, row_title) in enumerate(rows):
    row_outcomes = [ok_label, err_label]  # correct first (so error is on top)
    
    for col_idx, tmpl in enumerate(['C1', 'C2', 'C3']):
        ax = axes[row_idx, col_idx]
        sim_col = f'sim_{tmpl}'
        if sim_col not in peri_all_df.columns:
            continue
        
        # --- Plot two lines (correct + error) ---
        line_aggs = {}
        for outcome in row_outcomes:
            style = outcome_styles[outcome]
            sub_df = peri_all_df[peri_all_df['outcome'] == outcome]
            if len(sub_df) == 0:
                continue
            per_sub = (sub_df.groupby(['subject', 'tr_offset'])[sim_col]
                        .mean().reset_index())
            agg = (per_sub.groupby('tr_offset')[sim_col]
                   .agg(['mean', 'sem']).reset_index().sort_values('tr_offset'))
            line_aggs[outcome] = agg
            x = agg['tr_offset'] * TR_SECONDS
            ax.plot(x, agg['mean'], color=style['color'], lw=style['lw'],
                    label=style['label'], marker='o', markersize=3)
            ax.fill_between(x, agg['mean'] - agg['sem'], agg['mean'] + agg['sem'],
                            color=style['color'], alpha=0.2, linewidth=0)
        
        # --- Significance markers: error vs correct, paired across subjects ---
        per_sub_long = (peri_all_df.groupby(['subject', 'outcome', 'tr_offset'])[sim_col]
                         .mean().reset_index())
        wide = per_sub_long.pivot_table(
            index=['subject', 'tr_offset'], columns='outcome', values=sim_col
        ).reset_index()
        
        if {err_label, ok_label}.issubset(wide.columns):
            contrast_df = wide.dropna(subset=[err_label, ok_label]).copy()
            contrast_df['diff'] = contrast_df[err_label] - contrast_df[ok_label]
            style = outcome_styles[err_label]
            err_agg = line_aggs.get(err_label)
            if err_agg is not None:
                for tr_offset, g in contrast_df.groupby('tr_offset'):
                    diffs = g['diff'].dropna()
                    if len(diffs) < 5:
                        continue
                    _, p = ttest_1samp(diffs, 0)
                    if p >= 0.05:
                        continue
                    y_at_tr = err_agg[err_agg['tr_offset'] == tr_offset]['mean'].values
                    if len(y_at_tr) == 0:
                        continue
                    x_pos = tr_offset * TR_SECONDS
                    y_pos = y_at_tr[0]
                    if p < alpha_corrected:
                        ax.plot(x_pos, y_pos, 'o', markersize=8,
                                markeredgecolor=style['color'],
                                markerfacecolor=style['color'],
                                markeredgewidth=1.5, zorder=10)
                    else:
                        ax.plot(x_pos, y_pos, 'o', markersize=8,
                                markeredgecolor=style['color'],
                                markerfacecolor='white',
                                markeredgewidth=1.5, zorder=10)
        
        # --- Axis cosmetics ---
        ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.7)
        if tmpl != 'C1':  # add zero line for C2/C3
            ax.axhline(0, color='#bbbbbb', lw=0.5)
        
        # Title only for top row
        if row_idx == 0:
            ax.set_title(template_titles[tmpl], fontsize=10, loc='center')
        
        # X-label only for bottom row
        if row_idx == 1:
            ax.set_xlabel('Time relative to trial onset (s)')
        
        # Y-label only for left column (with row label)
        if col_idx == 0:
            ax.set_ylabel(f'{row_title}\nSimilarity ({SIMILARITY_METRIC})',
                          fontsize=10)
        
        # Legend in each row's first panel
        if col_idx == 0:
            ax.legend(frameon=False, fontsize=7, loc='lower right')

# Footer for marker meanings
fig.text(0.5, -0.02,
         '○ p < 0.05 vs correct (uncorrected)    '
         f'● p < 0.05 vs correct (Bonferroni-corrected across {n_trs} TRs)',
         ha='center', fontsize=8, color='#555555')

plt.tight_layout()
save_fig(fig, 'fig_peristimulus_6panel')
plt.show()

In [ ]:
# =============================================================================
# Trial-level sweep: Go omissions vs Go correct (hits)
# =============================================================================
def trial_level_omission(template_builder, hrf_lag_trs, window_trs):
    """
    Trial-level test: similarity on Go omissions vs Go hits.
    Returns within-subject diff (omission - hit), t-test across subjects.
    """
    metric_fn = get_metric(SIMILARITY_METRIC)
    templates_z = {}
    for sub in sart_stats.keys():
        t = template_builder(sub)
        if t is None:
            continue
        templates_z[sub] = zscore_against_sart(t, sub)
    
    rows = []
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in templates_z:
            continue
        template_z = templates_z[sub]
        
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)
        
        events = load_events_tsv(sub, run)
        if events is None:
            continue
        go_trials = events[
            (events['trial_type'] == 'go')
            & (events['thoughtNumber'] == tn)
        ]
        for _, trial in go_trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'hit' if trial['corr_ans'] == 1 else 'miss'
            
            start_tr = int(round(trial['onset'] / TR_SECONDS)) + hrf_lag_trs
            end_tr = start_tr + window_trs
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue
            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)
            sim = metric_fn(pattern_z, template_z)
            rows.append({'subject': sub, 'outcome': outcome, 'sim': sim})
    
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return None
    per_sub = df.groupby(['subject', 'outcome'])['sim'].mean().unstack()
    if not {'hit', 'miss'}.issubset(per_sub.columns):
        return None
    per_sub = per_sub.dropna(subset=['hit', 'miss'], how='any')
    if len(per_sub) < 10:
        return None
    diff = per_sub['miss'] - per_sub['hit']
    t, p = ttest_1samp(diff, 0)
    return {
        'hrf_lag_trs': hrf_lag_trs, 'window_trs': window_trs,
        'n_subj': len(diff), 'mean_diff': diff.mean(),
        't': t, 'p': p,
    }


# Wrapper for C1 (same-trial template per trial)
def trial_level_omission_c1(hrf_lag_trs, window_trs):
    """C1 version: each trial has its own template (the corresponding TB pattern)."""
    metric_fn = get_metric(SIMILARITY_METRIC)
    rows = []
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in sart_stats:
            continue
        tb_pat = tb_lookup.get((sub, run, tn))
        if tb_pat is None:
            continue
        template_z = zscore_against_sart(tb_pat, sub)
        
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)
        
        events = load_events_tsv(sub, run)
        if events is None:
            continue
        go_trials = events[
            (events['trial_type'] == 'go')
            & (events['thoughtNumber'] == tn)
        ]
        for _, trial in go_trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'hit' if trial['corr_ans'] == 1 else 'miss'
            
            start_tr = int(round(trial['onset'] / TR_SECONDS)) + hrf_lag_trs
            end_tr = start_tr + window_trs
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue
            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)
            sim = metric_fn(pattern_z, template_z)
            rows.append({'subject': sub, 'outcome': outcome, 'sim': sim})
    
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return None
    per_sub = df.groupby(['subject', 'outcome'])['sim'].mean().unstack()
    if not {'hit', 'miss'}.issubset(per_sub.columns):
        return None
    per_sub = per_sub.dropna(subset=['hit', 'miss'], how='any')
    if len(per_sub) < 10:
        return None
    diff = per_sub['miss'] - per_sub['hit']
    t, p = ttest_1samp(diff, 0)
    return {
        'hrf_lag_trs': hrf_lag_trs, 'window_trs': window_trs,
        'n_subj': len(diff), 'mean_diff': diff.mean(),
        't': t, 'p': p,
    }


# Run sweep across templates
print('=== Trial-level sweep: Go omissions (miss) vs Go hits ===\n')

print('--- C1: same-trial template ---')
sweep_om_c1 = []
for lag in range(-3, 4):
    for window in [1, 2, 3]:
        result = trial_level_omission_c1(lag, window)
        if result is not None:
            sweep_om_c1.append({**result, 'template': 'C1'})
            sig = ('***' if result['p']<0.001 else '**' if result['p']<0.01 
                   else '*' if result['p']<0.05 else '')
            print(f'  lag={lag:+d} TR ({lag*TR_SECONDS:+.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, '
                  f'diff={result["mean_diff"]:+.4f}  {sig}')

print('\n--- C2: subject-mean template ---')
sweep_om_c2 = []
for lag in range(-3, 4):
    for window in [1, 2, 3]:
        result = trial_level_omission(template_builder_c2, lag, window)
        if result is not None:
            sweep_om_c2.append({**result, 'template': 'C2'})
            sig = ('***' if result['p']<0.001 else '**' if result['p']<0.01 
                   else '*' if result['p']<0.05 else '')
            print(f'  lag={lag:+d} TR ({lag*TR_SECONDS:+.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, '
                  f'diff={result["mean_diff"]:+.4f}  {sig}')

print('\n--- C3: high-dyscontrol template ---')
sweep_om_c3 = []
for lag in range(-3, 4):
    for window in [1, 2, 3]:
        result = trial_level_omission(template_builder_c3, lag, window)
        if result is not None:
            sweep_om_c3.append({**result, 'template': 'C3'})
            sig = ('***' if result['p']<0.001 else '**' if result['p']<0.01 
                   else '*' if result['p']<0.05 else '')
            print(f'  lag={lag:+d} TR ({lag*TR_SECONDS:+.0f}s), window={window} TR: '
                  f't={result["t"]:+.2f}, p={result["p"]:.4f}, '
                  f'diff={result["mean_diff"]:+.4f}  {sig}')

# Save
all_om_sweeps = pd.concat([
    pd.DataFrame(sweep_om_c1),
    pd.DataFrame(sweep_om_c2),
    pd.DataFrame(sweep_om_c3),
], ignore_index=True)
all_om_sweeps.to_csv(DECODING_OUT_DIR / 'trial_level_omission_sweep.csv', index=False)

In [ ]:
# =============================================================================
# Difference figure: error - correct similarity, by TR offset
# Significance markers: open = p<0.05 uncorrected, filled = Bonferroni-corrected
# =============================================================================
from scipy.stats import ttest_1samp

contrasts = {
    'commission − correct withhold': ('commission', 'correct_withhold'),
    'miss − hit':                    ('miss',       'hit'),
}

# Bonferroni correction across the TR positions tested per (template × contrast)
n_trs = len(PERI_TRS)
alpha_corrected = 0.05 / n_trs
print(f'Bonferroni-corrected alpha (across {n_trs} TRs): p < {alpha_corrected:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=False)

for ax, (contrast_label, (err_label, ok_label)) in zip(axes, contrasts.items()):
    for tmpl in ['C1', 'C2', 'C3']:
        sim_col = f'sim_{tmpl}'
        if sim_col not in peri_all_df.columns:
            continue
        # Per subject × TR offset, mean similarity for each outcome
        per_sub = (peri_all_df.groupby(['subject', 'outcome', 'tr_offset'])[sim_col]
                    .mean().reset_index())
        # Pivot wide: rows=(subject, tr_offset), cols=outcomes
        wide = per_sub.pivot_table(
            index=['subject', 'tr_offset'], columns='outcome', values=sim_col
        ).reset_index()
        wide = wide.dropna(subset=[err_label, ok_label])
        if len(wide) == 0:
            continue
        wide['diff'] = wide[err_label] - wide[ok_label]
        
        # At each TR offset: mean diff, SE, t-test
        agg = (wide.groupby('tr_offset')['diff']
               .agg(['mean', 'sem']).reset_index().sort_values('tr_offset'))
        ps = []
        for tr, g in wide.groupby('tr_offset'):
            t_stat, p = ttest_1samp(g['diff'].dropna(), 0)
            ps.append({'tr_offset': tr, 't': t_stat, 'p': p})
        ps_df = pd.DataFrame(ps)
        agg = agg.merge(ps_df, on='tr_offset')
        
        style = template_styles[tmpl]
        x = (agg['tr_offset'] * TR_SECONDS).values
        ax.plot(x, agg['mean'], color=style['color'], lw=style['lw'],
                label=style['label'], marker='o', markersize=3)
        ax.fill_between(x, agg['mean'] - agg['sem'], agg['mean'] + agg['sem'],
                        color=style['color'], alpha=0.2, linewidth=0)
        
        # Significance markers
        unc_mask = (agg['p'] < 0.05) & (agg['p'] >= alpha_corrected)
        bonf_mask = agg['p'] < alpha_corrected
        # Open circles: uncorrected only
        ax.plot(x[unc_mask.values], agg.loc[unc_mask, 'mean'],
                'o', markersize=8, markeredgecolor=style['color'],
                markerfacecolor='white', markeredgewidth=1.5, zorder=10)
        # Filled circles: Bonferroni-significant
        ax.plot(x[bonf_mask.values], agg.loc[bonf_mask, 'mean'],
                'o', markersize=8, markeredgecolor=style['color'],
                markerfacecolor=style['color'], markeredgewidth=1.5, zorder=11)
    
    ax.axhline(0, color='#888888', lw=0.5)
    ax.axvline(0, color='#888888', lw=0.5, linestyle='--', alpha=0.7)
    ax.set_xlabel('Time relative to trial onset (s)')
    ax.set_ylabel('Difference in similarity\n(error − correct)')
    ax.set_title(contrast_label, fontsize=10, loc='left')
    ax.legend(frameon=False, fontsize=7, loc='best')

# Custom legend for significance markers (add as a footer)
fig.text(0.5, -0.04,
         '○ p < 0.05 (uncorrected)    ● p < 0.05 (Bonferroni-corrected '
         f'across {n_trs} TRs)',
         ha='center', fontsize=8, color='#555555')

fig.suptitle('Error-related worry-pattern reactivation by trial type',
             fontsize=11, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig_peristimulus_error_minus_correct')
plt.show()

# Similarity to decoder template

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.linear_model import Ridge
from scipy.stats import ttest_rel, ttest_1samp

LAGS = [-3, -2, -1, 0, 1, 2, 3]
WINDOW = 3
parcel_cols = [c for c in block_df.columns if c.startswith('parcel_')]

# --- Refit LOSO Ridge models + capture each held-out subject's TB-block scaling ---
df_tb = tb_blocks.dropna(subset=['dyscontrol_imputed'] + parcel_cols).copy()
subjects = sorted(df_tb['subject'].unique())

loso_models = {}
loso_tb_scaling = {}   # (mu, sd) from held-out subject's TB blocks

for held_out in subjects:
    train = df_tb[df_tb['subject'] != held_out]
    test = df_tb[df_tb['subject'] == held_out]
    if len(test) < 2:
        continue

    X_train = train[parcel_cols].values.astype(float)
    y_train = train['dyscontrol_imputed'].values.astype(float)
    train_subs = train['subject'].values

    X_train = zscore_within_subject(X_train, train_subs)
    y_train = center_within_subject(y_train, train_subs)

    model = Ridge(alpha=RIDGE_ALPHA)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(X_train, y_train)
    loso_models[held_out] = model

    X_test = test[parcel_cols].values.astype(float)
    mu = X_test.mean(axis=0)
    sd = X_test.std(axis=0); sd[sd == 0] = 1
    loso_tb_scaling[held_out] = (mu, sd)


def get_subject_decoder_trial_preds(sub, lag, window, trial_kind='nogo'):
    """Apply held-out subject's LOSO decoder to their trial windows.
    Returns dict {'correct': [...preds...], 'error': [...preds...]}."""
    if sub not in loso_models:
        return None
    mu, sd = loso_tb_scaling[sub]
    model = loso_models[sub]
    out = {'correct': [], 'error': []}

    for _, sart_row in sart_blocks[sart_blocks['subject'] == sub].iterrows():
        run = sart_row['run']
        tn = sart_row['thoughtNumber']

        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)

        events = load_events_tsv(sub, run)
        if events is None:
            continue
        trials = events[
            (events['trial_type'] == trial_kind)
            & (events['thoughtNumber'] == tn)
        ]

        for _, trial in trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'correct' if trial['corr_ans'] == 1 else 'error'

            start_tr = int(round(trial['onset'] / TR_SECONDS)) + lag
            end_tr = start_tr + window
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue

            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            # Scale by held-out subject's TB-block stats (match loso_decode test scaling)
            pattern_scaled = (window_pattern - mu) / sd
            pred = model.predict(pattern_scaled.reshape(1, -1))[0]
            out[outcome].append(pred)

    return out


# --- Run across lags ---
results = []
for lag in LAGS:
    comm_means, with_means = [], []
    for sub in subjects:
        preds = get_subject_decoder_trial_preds(sub, lag=lag, window=WINDOW,
                                                 trial_kind='nogo')
        if preds is None or len(preds['error']) == 0 or len(preds['correct']) == 0:
            continue
        comm_means.append(np.mean(preds['error']))    # commission = NoGo error
        with_means.append(np.mean(preds['correct']))  # correct withhold

    comm_arr, with_arr = np.array(comm_means), np.array(with_means)
    if len(comm_arr) < 10:
        print(f'lag={lag:+d}: insufficient data (n={len(comm_arr)})')
        continue

    # Paired: commission vs withhold within subject
    t, p = ttest_rel(comm_arr, with_arr)
    # Also: is the difference's group mean different from 0 (same thing, paired)
    diff = comm_arr - with_arr
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '†' if p<0.10 else ''
    print(f'lag={lag:+d} ({lag*2:+d}s): comm={comm_arr.mean():+.4f} '
          f'with={with_arr.mean():+.4f} diff={diff.mean():+.4f} '
          f't({len(comm_arr)-1})={t:+.2f} p={p:.4f} {sig}')
    results.append({'lag': lag, 'n': len(comm_arr), 't': t, 'p': p,
                     'comm_mean': comm_arr.mean(), 'with_mean': with_arr.mean(),
                     'diff': diff.mean()})

results_df = pd.DataFrame(results)
results_df.to_csv(DECODING_OUT_DIR / 'decoder_pre_error.csv', index=False)

print('\nSorted by p:')
print(results_df.sort_values('p').round(4).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

# Reuses loso_models from the previous cell (already fitted).
# Only difference: scale trial windows with zscore_against_sart instead of TB-block mu/sd.

def get_subject_decoder_trial_preds_sartscale(sub, lag, window, trial_kind='nogo'):
    if sub not in loso_models:
        return None
    model = loso_models[sub]
    out = {'correct': [], 'error': []}

    for _, sart_row in sart_blocks[sart_blocks['subject'] == sub].iterrows():
        run = sart_row['run']
        tn = sart_row['thoughtNumber']

        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)

        events = load_events_tsv(sub, run)
        if events is None:
            continue
        trials = events[
            (events['trial_type'] == trial_kind)
            & (events['thoughtNumber'] == tn)
        ]

        for _, trial in trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'correct' if trial['corr_ans'] == 1 else 'error'

            start_tr = int(round(trial['onset'] / TR_SECONDS)) + lag
            end_tr = start_tr + window
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue

            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_scaled = zscore_against_sart(window_pattern, sub)  # <-- the change
            pred = model.predict(pattern_scaled.reshape(1, -1))[0]
            out[outcome].append(pred)

    return out


results_sart = []
for lag in LAGS:
    comm_means, with_means = [], []
    for sub in subjects:
        preds = get_subject_decoder_trial_preds_sartscale(sub, lag=lag, window=WINDOW,
                                                          trial_kind='nogo')
        if preds is None or len(preds['error']) == 0 or len(preds['correct']) == 0:
            continue
        comm_means.append(np.mean(preds['error']))
        with_means.append(np.mean(preds['correct']))

    comm_arr, with_arr = np.array(comm_means), np.array(with_means)
    if len(comm_arr) < 10:
        print(f'lag={lag:+d}: insufficient data (n={len(comm_arr)})')
        continue

    t, p = ttest_rel(comm_arr, with_arr)
    diff = comm_arr - with_arr
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '†' if p<0.10 else ''
    print(f'lag={lag:+d} ({lag*2:+d}s): comm={comm_arr.mean():+.4f} '
          f'with={with_arr.mean():+.4f} diff={diff.mean():+.4f} '
          f't({len(comm_arr)-1})={t:+.2f} p={p:.4f} {sig}')
    results_sart.append({'lag': lag, 'n': len(comm_arr), 't': t, 'p': p,
                          'comm_mean': comm_arr.mean(), 'with_mean': with_arr.mean(),
                          'diff': diff.mean()})

results_sart_df = pd.DataFrame(results_sart)
results_sart_df.to_csv(DECODING_OUT_DIR / 'decoder_pre_error_sartscale.csv', index=False)
print('\nSorted by p:')
print(results_sart_df.sort_values('p').round(4).to_string(index=False))

# individual template vs. group template

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, ttest_1samp

metric_fn = get_metric(SIMILARITY_METRIC)

# tb_high_mean is {sub: raw high-dyscontrol mean TB pattern}
# Build LOSO common template per subject: average of OTHERS' raw high-dys patterns
def build_common_template_raw(held_out):
    others = [tb_high_mean[s] for s in tb_high_mean if s != held_out]
    if len(others) == 0:
        return None
    return np.mean(np.array(others), axis=0)  # raw parcel space


def get_own_vs_common_effects(lag=-1, window=3, trial_kind='nogo'):
    """For each subject, compute pre-error effect (error - correct) using
    (a) their OWN high-dys template and (b) the LOSO group-COMMON template.
    Both templates z-scored against the held-out subject's SART distribution.
    Returns DataFrame: subject, own_effect, common_effect."""
    rows = []

    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in sart_stats or sub not in tb_high_mean:
            continue

        # Own template (= C3) and common template, both in this subject's SART z-space
        own_z = zscore_against_sart(tb_high_mean[sub], sub)
        common_raw = build_common_template_raw(sub)
        if common_raw is None:
            continue
        common_z = zscore_against_sart(common_raw, sub)

        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)

        events = load_events_tsv(sub, run)
        if events is None:
            continue
        trials = events[
            (events['trial_type'] == trial_kind)
            & (events['thoughtNumber'] == tn)
        ]

        for _, trial in trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            outcome = 'correct' if trial['corr_ans'] == 1 else 'error'

            start_tr = int(round(trial['onset'] / TR_SECONDS)) + lag
            end_tr = start_tr + window
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue

            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)

            rows.append({
                'subject': sub, 'outcome': outcome,
                'own_sim': metric_fn(pattern_z, own_z),
                'common_sim': metric_fn(pattern_z, common_z),
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return pd.DataFrame()

    out = []
    for sub, g in df.groupby('subject'):
        means = g.groupby('outcome')[['own_sim', 'common_sim']].mean()
        if not {'correct', 'error'}.issubset(means.index):
            continue
        out.append({
            'subject': sub,
            'own_effect': means.loc['error', 'own_sim'] - means.loc['correct', 'own_sim'],
            'common_effect': means.loc['error', 'common_sim'] - means.loc['correct', 'common_sim'],
        })
    return pd.DataFrame(out)


# Run at the pre-error peak (lag=-1, window=3)
eff = get_own_vs_common_effects(lag=-1, window=3, trial_kind='nogo')
print(f'n subjects = {len(eff)}\n')

# Group-level: is each effect different from 0?
for col, label in [('own_effect', 'OWN (C3)'), ('common_effect', 'COMMON (LOSO group avg)')]:
    t, p = ttest_1samp(eff[col], 0)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '†' if p<0.10 else ''
    print(f'{label:28s}: M={eff[col].mean():+.4f}, t({len(eff)-1})={t:+.2f}, p={p:.4f} {sig}')

# Head-to-head: is own > common, within subject?
t, p = ttest_rel(eff['own_effect'], eff['common_effect'])
diff = (eff['own_effect'] - eff['common_effect'])
sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '†' if p<0.10 else ''
print(f'\nOWN vs COMMON (paired): diff={diff.mean():+.4f}, '
      f't({len(eff)-1})={t:+.2f}, p={p:.4f} {sig}')
print(f'  subjects with own > common: {(diff > 0).sum()}/{len(eff)}')

eff.to_csv(DECODING_OUT_DIR / 'own_vs_common_preerror.csv', index=False)

# Section E - relationships to trait measures

# E0 diagnostic

In [ ]:
# Diagnostic: check trait columns and coverage
df = pd.read_csv(BLOCK_FEATURES_CSV)
trait_cols = [c for c in df.columns if any(t in c.lower() for t in ['ptq', 'pswq'])]
print(f'Trait columns in block_features.csv: {trait_cols}')

per_sub = df.groupby('subject')[trait_cols].first()
for col in trait_cols:
    n_with = per_sub[col].notna().sum()
    print(f'  {col}: n={n_with}, M={per_sub[col].mean():.1f}, '
          f'SD={per_sub[col].std():.1f}, '
          f'range [{per_sub[col].min():.0f}, {per_sub[col].max():.0f}]')

if 'ptq_total' in trait_cols and 'pswq_total' in trait_cols:
    valid = per_sub.dropna(subset=['ptq_total', 'pswq_total'])
    if len(valid) > 5:
        from scipy.stats import pearsonr
        r, p = pearsonr(valid['ptq_total'], valid['pswq_total'])
        print(f'\nPTQ-PSWQ correlation across subjects: r={r:+.3f}, p={p:.4f}, n={len(valid)}')

# E1: Trait → TB decoding strength

In [ ]:
# =============================================================================
# E1: Trait → TB decoding strength
# =============================================================================
from scipy.stats import pearsonr

# Per-subject trait scores
trait_df = (block_df.groupby('subject')[
    ['ptq_total', 'ptq_core', 'ptq_unproductive', 'ptq_capacity', 'pswq_total']
].first().reset_index())

# Read per-subject decoder performance and merge with traits
loso_per_subject = pd.read_csv(DECODING_OUT_DIR / 'loso_per_subject.csv')
loso_traits = loso_per_subject.merge(
    trait_df, left_on='held_out_subject', right_on='subject', how='left'
)

print('=== E1: Trait → TB decoding strength ===')
print('  (per-subject held-out LOSO r vs trait)')
print()
print(f'{"trait":20s}  {"r":>7s}  {"p":>7s}  {"n":>3s}')
print('-' * 45)
results_e1 = []
for trait in ['ptq_total', 'ptq_core', 'ptq_unproductive', 'ptq_capacity', 'pswq_total']:
    valid = loso_traits.dropna(subset=['r', trait])
    if len(valid) < 10:
        continue
    r, p = pearsonr(valid['r'], valid[trait])
    sig = ('***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
    print(f'{trait:20s}  {r:>+7.3f}  {p:>7.4f}  {len(valid):>3d}  {sig}')
    results_e1.append({'trait': trait, 'r': r, 'p': p, 'n': len(valid)})

pd.DataFrame(results_e1).to_csv(
    DECODING_OUT_DIR / 'trait_decoder_correlations.csv', index=False
)

# E2: Trait → SART-TB similarity strength (per-subject mean by template)

In [ ]:
# =============================================================================
# E2: Trait → SART-TB similarity strength (per-subject mean by template)
# =============================================================================
print('=== E2: Trait → SART-TB similarity strength ===')
print('  (per-subject mean similarity to TB template)')

similarity_summary_files = {
    'C1: same-trial':       DECODING_OUT_DIR / 'sart_similarity_C1_summary.csv',
    'C2: subject-mean':     DECODING_OUT_DIR / 'sart_similarity_C2_summary.csv',
    'C3: high-dyscontrol':  DECODING_OUT_DIR / 'sart_similarity_C3_summary.csv',
}

trait_sim_results = []
for tmpl_label, path in similarity_summary_files.items():
    if not path.exists():
        print(f'  Missing: {path.name}')
        continue
    sim_df = pd.read_csv(path)
    
    for sim_col in ['block_sim', 'mean_sim', 'max_sim']:
        if sim_col not in sim_df.columns:
            continue
        per_sub = sim_df.groupby('subject')[sim_col].mean().reset_index()
        per_sub = per_sub.merge(trait_df, on='subject', how='left')
        
        for trait in ['ptq_total', 'ptq_core', 'ptq_unproductive', 
                      'ptq_capacity', 'pswq_total']:
            valid = per_sub.dropna(subset=[sim_col, trait])
            if len(valid) < 10:
                continue
            r, p = pearsonr(valid[sim_col], valid[trait])
            trait_sim_results.append({
                'template': tmpl_label, 'sim_metric': sim_col,
                'trait': trait, 'n': len(valid), 'r': r, 'p': p,
            })

trait_sim_df = pd.DataFrame(trait_sim_results)

# Print a summary: just the significant ones
print()
print(f'{"template":22s}  {"metric":10s}  {"trait":18s}  {"r":>7s}  {"p":>7s}')
print('-' * 75)
sig = trait_sim_df[trait_sim_df['p'] < 0.10].sort_values('p')
if len(sig) > 0:
    for _, r in sig.iterrows():
        marker = ('***' if r['p']<0.001 else '**' if r['p']<0.01 
                  else '*' if r['p']<0.05 else '†')
        print(f'{r["template"]:22s}  {r["sim_metric"]:10s}  '
              f'{r["trait"]:18s}  {r["r"]:>+7.3f}  '
              f'{r["p"]:>7.4f}  {marker}')
else:
    print('  No template/metric/trait combos reach p<0.10.')

trait_sim_df.to_csv(DECODING_OUT_DIR / 'trait_similarity_correlations.csv', index=False)
print(f'\nFull table saved: {len(trait_sim_df)} rows')

# E3: Trait moderation of within-subject probe2/3 coupling

In [ ]:
# =============================================================================
# E3: Trait moderation of within-subject probe2/3 coupling
# =============================================================================
print('=== E3: Trait moderation of within-subject probe coupling ===')
print('  (per-subject within-subject r between similarity and probe,')
print('   correlated with trait scores)')

# Use the templates and sim_cols that gave block-level probe effects
TEMPLATES_TO_TEST = {
    'C1: same-trial':      DECODING_OUT_DIR / 'sart_similarity_C1_summary.csv',
    'C2: subject-mean':    DECODING_OUT_DIR / 'sart_similarity_C2_summary.csv',
    'C3: high-dyscontrol': DECODING_OUT_DIR / 'sart_similarity_C3_summary.csv',
}
SIM_COLS = ['block_sim', 'mean_sim', 'max_sim']
PROBES = ['probe2', 'probe3']  # the two task-engagement probes that work

results_e3 = []
for tmpl_label, path in TEMPLATES_TO_TEST.items():
    if not path.exists():
        continue
    sim_df = pd.read_csv(path)
    
    for sim_col in SIM_COLS:
        if sim_col not in sim_df.columns:
            continue
        for probe in PROBES:
            if probe not in sim_df.columns:
                continue
            
            # Per-subject within-subject Pearson r between sim and probe
            sub_slopes = []
            for sub, g in sim_df.groupby('subject'):
                valid = g.dropna(subset=[sim_col, probe])
                if (len(valid) < 5 
                    or valid[sim_col].std() == 0 
                    or valid[probe].std() == 0):
                    continue
                r, _ = pearsonr(valid[sim_col], valid[probe])
                if not np.isnan(r):
                    sub_slopes.append({'subject': sub, 'within_r': r})
            slopes_df = pd.DataFrame(sub_slopes).merge(trait_df, on='subject')
            
            for trait in ['ptq_total', 'ptq_core', 'ptq_unproductive', 
                          'ptq_capacity', 'pswq_total']:
                valid = slopes_df.dropna(subset=['within_r', trait])
                if len(valid) < 10:
                    continue
                r, p = pearsonr(valid['within_r'], valid[trait])
                results_e3.append({
                    'template': tmpl_label, 'sim_metric': sim_col,
                    'probe': probe, 'trait': trait,
                    'n': len(valid), 'r': r, 'p': p,
                })

results_e3_df = pd.DataFrame(results_e3)

# Print significant findings only
print()
print(f'{"template":22s}  {"metric":10s}  {"probe":7s}  {"trait":18s}  '
      f'{"r":>7s}  {"p":>7s}')
print('-' * 90)
sig = results_e3_df[results_e3_df['p'] < 0.10].sort_values('p')
if len(sig) > 0:
    for _, r in sig.iterrows():
        marker = ('***' if r['p']<0.001 else '**' if r['p']<0.01 
                  else '*' if r['p']<0.05 else '†')
        print(f'{r["template"]:22s}  {r["sim_metric"]:10s}  '
              f'{r["probe"]:7s}  {r["trait"]:18s}  '
              f'{r["r"]:>+7.3f}  {r["p"]:>7.4f}  {marker}')
else:
    print('  No template/metric/probe/trait combos reach p<0.10.')

results_e3_df.to_csv(DECODING_OUT_DIR / 'trait_moderation_correlations.csv', index=False)
print(f'\nFull table saved: {len(results_e3_df)} rows')

# E3b: Trait moderation of within-subject ACCURACY coupling

In [ ]:
# =============================================================================
# E3b: Trait moderation of within-subject ACCURACY coupling
# =============================================================================
print('=== E3b: Trait moderation of within-subject accuracy coupling ===')
print('  (per-subject within-subject r between similarity and accuracy,')
print('   correlated with trait scores)')

ACC_TARGETS = ['NoGo_ACC', 'AccIndex', 'Mean_go_RT', 'meanRT']

results_e3b = []
for tmpl_label, path in TEMPLATES_TO_TEST.items():
    if not path.exists():
        continue
    sim_df = pd.read_csv(path)
    
    for sim_col in SIM_COLS:
        if sim_col not in sim_df.columns:
            continue
        for target in ACC_TARGETS:
            if target not in sim_df.columns:
                continue
            
            sub_slopes = []
            for sub, g in sim_df.groupby('subject'):
                valid = g.dropna(subset=[sim_col, target])
                if (len(valid) < 5 
                    or valid[sim_col].std() == 0 
                    or valid[target].std() == 0):
                    continue
                r, _ = pearsonr(valid[sim_col], valid[target])
                if not np.isnan(r):
                    sub_slopes.append({'subject': sub, 'within_r': r})
            slopes_df = pd.DataFrame(sub_slopes).merge(trait_df, on='subject')
            
            for trait in ['ptq_total', 'ptq_core', 'ptq_unproductive',
                          'ptq_capacity', 'pswq_total']:
                valid = slopes_df.dropna(subset=['within_r', trait])
                if len(valid) < 10:
                    continue
                r, p = pearsonr(valid['within_r'], valid[trait])
                results_e3b.append({
                    'template': tmpl_label, 'sim_metric': sim_col,
                    'target': target, 'trait': trait,
                    'n': len(valid), 'r': r, 'p': p,
                })

results_e3b_df = pd.DataFrame(results_e3b)

print()
print(f'{"template":22s}  {"metric":10s}  {"target":12s}  {"trait":18s}  '
      f'{"r":>7s}  {"p":>7s}')
print('-' * 95)
sig = results_e3b_df[results_e3b_df['p'] < 0.10].sort_values('p')
if len(sig) > 0:
    for _, row in sig.iterrows():
        marker = ('***' if row['p']<0.001 else '**' if row['p']<0.01
                  else '*' if row['p']<0.05 else '†')
        print(f'{row["template"]:22s}  {row["sim_metric"]:10s}  '
              f'{row["target"]:12s}  {row["trait"]:18s}  '
              f'{row["r"]:>+7.3f}  {row["p"]:>7.4f}  {marker}')
else:
    print('  No combos reach p<0.10.')

results_e3b_df.to_csv(
    DECODING_OUT_DIR / 'trait_moderation_accuracy_correlations.csv', index=False
)
print(f'\nFull table saved: {len(results_e3b_df)} rows')

In [ ]:
# =============================================================================
# Section E (focused-strict): Trait moderation of pre-specified effects
# Each effect tested across 3 metrics (block_sim, mean_sim, max_sim) treated
# as a family. Within-family correction (Bonferroni ×3), then across-family FDR.
# =============================================================================
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Pre-specified effects (template, csv_filename, target) where the
# within-subject relationship was significant (p<0.05) for at least one metric
SIG_EFFECTS = [
    ('C1 (same-trial)',      'sart_similarity_C1_summary.csv', 'probe2'),
    ('C1 (same-trial)',      'sart_similarity_C1_summary.csv', 'NoGo_ACC'),
    ('C1 (same-trial)',      'sart_similarity_C1_summary.csv', 'AccIndex'),
    ('C2 (subject-mean)',    'sart_similarity_C2_summary.csv', 'probe2'),
    ('C3 (high-dyscontrol)', 'sart_similarity_C3_summary.csv', 'probe2'),
    ('C3 (high-dyscontrol)', 'sart_similarity_C3_summary.csv', 'probe6'),
]
METRICS = ['block_sim', 'mean_sim', 'max_sim']
FOCUSED_TRAITS = ['ptq_total', 'pswq_total']

trait_df = (block_df.groupby('subject')[FOCUSED_TRAITS].first().reset_index())


def fdr_correct(p_values):
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranks = np.empty(n, int)
    ranks[order] = np.arange(1, n + 1)
    p_adj = p * n / ranks
    sorted_p_adj = p_adj[order]
    for i in range(n - 2, -1, -1):
        sorted_p_adj[i] = min(sorted_p_adj[i], sorted_p_adj[i + 1])
    p_adj_final = np.empty(n)
    p_adj_final[order] = np.minimum(sorted_p_adj, 1.0)
    return p_adj_final


# Compute all metric-level results first, then aggregate
metric_results = []
for tmpl_label, csv_fname, target in SIG_EFFECTS:
    sim_df = pd.read_csv(DECODING_OUT_DIR / csv_fname)
    for metric in METRICS:
        if metric not in sim_df.columns:
            continue
        # Per-subject within-subject Pearson r
        sub_slopes = []
        for sub, g in sim_df.groupby('subject'):
            valid = g.dropna(subset=[metric, target])
            if (len(valid) < 5 
                or valid[metric].std() == 0 
                or valid[target].std() == 0):
                continue
            r, _ = pearsonr(valid[metric], valid[target])
            if not np.isnan(r):
                sub_slopes.append({'subject': sub, 'within_r': r})
        slopes_df = pd.DataFrame(sub_slopes).merge(trait_df, on='subject')
        for trait in FOCUSED_TRAITS:
            valid = slopes_df.dropna(subset=['within_r', trait])
            if len(valid) < 10:
                continue
            r, p = pearsonr(valid['within_r'], valid[trait])
            metric_results.append({
                'template': tmpl_label, 'target': target, 'trait': trait,
                'metric': metric, 'n': len(valid), 'r': r, 'p': p,
            })

metric_df = pd.DataFrame(metric_results)

# Aggregate across metrics for each (template, target, trait)
# Apply Bonferroni-within-family: best p × 3, capped at 1
families = []
for (tmpl, target, trait), grp in metric_df.groupby(['template', 'target', 'trait']):
    best = grp.loc[grp['p'].idxmin()]  # best metric for this family
    p_within = min(best['p'] * len(grp), 1.0)  # Bonferroni within family
    families.append({
        'template': tmpl,
        'target': target,
        'trait': trait,
        'best_metric': best['metric'],
        'best_r': best['r'],
        'best_p_raw': best['p'],
        'p_within_family': p_within,
        'n': best['n'],
        'all_metrics': '; '.join(
            f'{row["metric"]}=r{row["r"]:+.3f}/p{row["p"]:.3f}' 
            for _, row in grp.iterrows()
        ),
    })

family_df = pd.DataFrame(families).sort_values('p_within_family').reset_index(drop=True)
family_df['p_fdr'] = fdr_correct(family_df['p_within_family'].values)
family_df['p_bonf'] = np.minimum(family_df['p_within_family'] * len(family_df), 1.0)

# Print
print(f'Pre-specified focused trait analysis')
print(f'  Families: {len(family_df)} (effect-trait combinations)')
print(f'  Within-family correction: Bonferroni across 3 metrics')
print(f'  Across-family correction: FDR (Benjamini-Hochberg) and Bonferroni')
print('=' * 110)
print(f'  {"contrast":42s}  {"best_metric":11s}  {"best_r":>7s}  '
      f'{"p_raw":>7s}  {"p_fam":>7s}  {"p_fdr":>7s}  {"p_bonf":>7s}')
print('-' * 110)
for _, row in family_df.iterrows():
    contrast = f'{row["template"]:20s} {row["target"]:10s} {row["trait"]:11s}'
    sig_fam = '*' if row['p_within_family']<0.05 else ''
    sig_fdr = '*' if row['p_fdr']<0.05 else ''
    sig_bonf = '*' if row['p_bonf']<0.05 else ''
    print(f'  {contrast:42s}  {row["best_metric"]:11s}  '
          f'{row["best_r"]:>+7.3f}  '
          f'{row["best_p_raw"]:>7.4f}  '
          f'{row["p_within_family"]:>7.4f}{sig_fam:2s}  '
          f'{row["p_fdr"]:>7.4f}{sig_fdr:2s}  '
          f'{row["p_bonf"]:>7.4f}{sig_bonf:2s}')

# Show full breakdown of metrics for completeness
print('\nFull metric breakdown for each family:')
for _, row in family_df.iterrows():
    print(f'  {row["template"]} × {row["target"]} × {row["trait"]}: '
          f'{row["all_metrics"]}')

family_df.to_csv(DECODING_OUT_DIR / 'trait_focused_family.csv', index=False)
metric_df.to_csv(DECODING_OUT_DIR / 'trait_focused_family_full.csv', index=False)
print(f'\nSaved to trait_focused_family.csv and trait_focused_family_full.csv')

In [ ]:
# E_extra: C3 × probe1 × trait moderation
sim_df = pd.read_csv(DECODING_OUT_DIR / 'sart_similarity_C3_summary.csv')
trait_df = (block_df.groupby('subject')[['ptq_total', 'pswq_total']]
             .first().reset_index())

print(f'C3 × probe1 trait moderation:')
print(f'{"metric":12s}  {"trait":12s}  {"r":>7s}  {"p":>7s}')
print('-' * 50)
for metric in ['block_sim', 'mean_sim', 'max_sim']:
    sub_slopes = []
    for sub, g in sim_df.groupby('subject'):
        valid = g.dropna(subset=[metric, 'probe1'])
        if (len(valid) < 5 
            or valid[metric].std() == 0 
            or valid['probe1'].std() == 0):
            continue
        r, _ = pearsonr(valid[metric], valid['probe1'])
        if not np.isnan(r):
            sub_slopes.append({'subject': sub, 'within_r': r})
    slopes_df = pd.DataFrame(sub_slopes).merge(trait_df, on='subject')
    print(f'  Per-subject within-r distribution: M={slopes_df["within_r"].mean():+.3f}, '
          f'SD={slopes_df["within_r"].std():.3f}, n={len(slopes_df)}')
    for trait in ['ptq_total', 'pswq_total']:
        valid = slopes_df.dropna(subset=['within_r', trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid['within_r'], valid[trait])
        sig = ('***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
        print(f'  {metric:12s}  {trait:12s}  {r:>7.3f}  {p:>+7.4f}  {sig}')

In [ ]:
# =============================================================================
# Trait moderation of trial-level pre-error reactivation
# Tests whether chronic worry sharpens the pre-error worry-pattern reactivation
# Replicates trial_level_d1() but keeps per-subject effects, then correlates with trait
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

def get_subject_trial_level_effects(template_kind='C1', lag=-1, window=3,
                                     trial_kind='nogo'):
    """
    For each subject, compute the trial-level effect:
        mean(similarity for error trials) - mean(similarity for correct trials)
    at a specified lag and window.
    
    template_kind: 'C1', 'C2', or 'C3'
    trial_kind: 'nogo' (commission errors vs correct withhold)
                or 'go' (omission errors vs correct hits)
    Returns: dict {subject: effect}
    """
    metric_fn = get_metric(SIMILARITY_METRIC)
    rows = []
    
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in sart_stats:
            continue
        
        # Build the template for this subject
        if template_kind == 'C1':
            tb_pat = tb_lookup.get((sub, run, tn))
            if tb_pat is None:
                continue
            template_z = zscore_against_sart(tb_pat, sub)
        elif template_kind == 'C2':
            template_z = zscore_against_sart(tb_subject_mean[sub], sub)
        elif template_kind == 'C3':
            if sub not in tb_high_mean:
                continue
            template_z = zscore_against_sart(tb_high_mean[sub], sub)
        
        # Load the parcel timeseries
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)
        
        # Get the trial events
        events = load_events_tsv(sub, run)
        if events is None:
            continue
        
        # Filter to NoGo or Go trials within this thoughtNumber
        trials = events[
            (events['trial_type'] == trial_kind)
            & (events['thoughtNumber'] == tn)
        ]
        
        for _, trial in trials.iterrows():
            if pd.isna(trial['corr_ans']):
                continue
            
            # For NoGo: corr_ans=1 = correct withhold, 0 = commission
            # For Go:   corr_ans=1 = correct hit, 0 = omission
            outcome = 'correct' if trial['corr_ans'] == 1 else 'error'
            
            start_tr = int(round(trial['onset'] / TR_SECONDS)) + lag
            end_tr = start_tr + window
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue
            
            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)
            sim = metric_fn(pattern_z, template_z)
            
            rows.append({
                'subject': sub, 'outcome': outcome, 'sim': sim,
            })
    
    df = pd.DataFrame(rows)
    if len(df) == 0:
        return {}
    
    # Per-subject effect: mean(error) - mean(correct)
    per_sub = df.groupby(['subject', 'outcome'])['sim'].mean().unstack()
    if not {'correct', 'error'}.issubset(per_sub.columns):
        return {}
    per_sub = per_sub.dropna()
    return (per_sub['error'] - per_sub['correct']).to_dict()


# Load trait data per subject
trait_lookup = (
    block_df.groupby('subject')[['pswq_total', 'ptq_total', 'ptq_core',
                                   'ptq_unproductive', 'ptq_capacity']]
    .first().to_dict('index')
)

# Compute trait moderation across templates, trial types, and key lag/window combos
print('=' * 100)
print('Trait moderation of trial-level reactivation')
print('  Per-subject (error - correct) similarity at specified lag/window,')
print('  correlated with trait scores across subjects')
print('=' * 100)

LAGS_TO_TEST = [
    # Pre-error (worry reactivation): expect positive effect, expect chronic worriers to have larger
    (-1, 3),  # Peak pre-error effect in main analysis
    (0, 2),   # Trial onset
    # Post-error (anti-worry monitoring): expect negative effect
    (3, 3),
]
TEMPLATES = ['C1', 'C2', 'C3']
TRIAL_KINDS = ['nogo', 'go']
TRAITS = ['pswq_total', 'ptq_total', 'ptq_core',
          'ptq_unproductive', 'ptq_capacity']

results = []
for trial_kind in TRIAL_KINDS:
    for template in TEMPLATES:
        for lag, window in LAGS_TO_TEST:
            print(f'\n{trial_kind.upper()}, {template}, lag={lag} TR ({lag*2:+d}s), '
                  f'window={window} TR:')
            
            effects = get_subject_trial_level_effects(
                template_kind=template, lag=lag, window=window,
                trial_kind=trial_kind,
            )
            if len(effects) == 0:
                print('  No data')
                continue
            
            # Build per-subject DataFrame
            df_eff = pd.DataFrame([
                {'subject': sub, 'effect': eff,
                 **{trait: trait_lookup.get(sub, {}).get(trait, np.nan)
                    for trait in TRAITS}}
                for sub, eff in effects.items()
            ])
            df_eff = df_eff.dropna(subset=['effect'])
            
            # Group-level: is effect different from 0?
            from scipy.stats import ttest_1samp
            t_group, p_group = ttest_1samp(df_eff['effect'], 0)
            sig_group = ('***' if p_group < 0.001 else '**' if p_group < 0.01
                          else '*' if p_group < 0.05 else '')
            print(f'  Group: M={df_eff["effect"].mean():+.4f}, '
                  f'SD={df_eff["effect"].std():.4f}, '
                  f't={t_group:+.2f}, p={p_group:.4f} {sig_group}')
            
            # Trait correlations
            for trait in TRAITS:
                df_valid = df_eff.dropna(subset=[trait])
                if len(df_valid) < 10:
                    continue
                r, p = pearsonr(df_valid['effect'], df_valid[trait])
                sig = ('***' if p < 0.001 else '**' if p < 0.01
                       else '*' if p < 0.05 else '†' if p < 0.10 else '')
                results.append({
                    'trial_kind': trial_kind,
                    'template': template,
                    'lag': lag,
                    'window': window,
                    'trait': trait,
                    'n_subj': len(df_valid),
                    'group_effect': df_eff['effect'].mean(),
                    'group_t': t_group,
                    'group_p': p_group,
                    'trait_r': r,
                    'trait_p': p,
                })
                print(f'    {trait:22s}  r = {r:+.3f}, p = {p:.4f}  {sig}')

results_df = pd.DataFrame(results)
results_df.to_csv(DECODING_OUT_DIR / 'trait_moderation_trial_level.csv', index=False)

# Summary: pre-error (lag=-1) sorted by significance
print('\n' + '=' * 100)
print('SUMMARY: Pre-error effects, sorted by trait correlation significance')
print('=' * 100)
pre_error = results_df[(results_df['lag'] == -1) & (results_df['trial_kind'] == 'nogo')]
pre_error_sorted = pre_error.sort_values('trait_p')
print(pre_error_sorted[['template', 'trait', 'trait_r', 'trait_p',
                          'group_effect', 'group_p']].round(4).to_string(index=False))

print('\n' + '=' * 100)
print('SUMMARY: Post-error effects, sorted by trait correlation significance')
print('=' * 100)
post_error = results_df[(results_df['lag'] == 3) & (results_df['trial_kind'] == 'nogo')]
post_error_sorted = post_error.sort_values('trait_p')
print(post_error_sorted[['template', 'trait', 'trait_r', 'trait_p',
                           'group_effect', 'group_p']].round(4).to_string(index=False))

In [ ]:
# =============================================================================
# Trait moderation of trial-level RT coupling with worry-pattern similarity
# Within-subject: does worry-pattern similarity at each Go trial predict RT?
# Across-subject: is this coupling stronger in chronic worriers?
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_1samp

def get_subject_trial_level_rt_coupling(template_kind='C1', lag=-1, window=3):
    """
    For each subject:
    1. For each Go trial, compute similarity of brain pattern (at lag/window 
       around trial onset) to the appropriate TB template
    2. Compute within-subject Pearson r between trial similarity and trial RT
    
    Returns: dict {subject: within_subject_r}
    """
    metric_fn = get_metric(SIMILARITY_METRIC)
    trial_data = []
    
    for _, sart_row in sart_blocks.iterrows():
        sub = sart_row['subject']
        run = sart_row['run']
        tn = sart_row['thoughtNumber']
        if sub not in sart_stats:
            continue
        
        # Build template
        if template_kind == 'C1':
            tb_pat = tb_lookup.get((sub, run, tn))
            if tb_pat is None:
                continue
            template_z = zscore_against_sart(tb_pat, sub)
        elif template_kind == 'C2':
            template_z = zscore_against_sart(tb_subject_mean[sub], sub)
        elif template_kind == 'C3':
            if sub not in tb_high_mean:
                continue
            template_z = zscore_against_sart(tb_high_mean[sub], sub)
        
        # Load parcel timeseries
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)
        
        # Get Go trials with RT
        events = load_events_tsv(sub, run)
        if events is None:
            continue
        
        go_trials = events[
            (events['trial_type'] == 'go')
            & (events['thoughtNumber'] == tn)
            & events['response_time'].notna()
        ]
        
        for _, trial in go_trials.iterrows():
            start_tr = int(round(trial['onset'] / TR_SECONDS)) + lag
            end_tr = start_tr + window
            if start_tr < 0 or end_tr > len(parcel_ts):
                continue
            
            window_pattern = parcel_ts[start_tr:end_tr].mean(axis=0)
            pattern_z = zscore_against_sart(window_pattern, sub)
            sim = metric_fn(pattern_z, template_z)
            
            trial_data.append({
                'subject': sub,
                'sim': sim,
                'rt': trial['response_time'],
            })
    
    df = pd.DataFrame(trial_data)
    if len(df) == 0:
        return {}
    
    # Per-subject within-r between sim and rt
    coupling = {}
    for sub, g in df.groupby('subject'):
        if len(g) < 10:
            continue
        if g['sim'].std() == 0 or g['rt'].std() == 0:
            continue
        r, _ = pearsonr(g['sim'], g['rt'])
        if not np.isnan(r):
            coupling[sub] = r
    return coupling


# Trait data
trait_lookup = (
    block_df.groupby('subject')[['pswq_total', 'ptq_total', 'ptq_core',
                                   'ptq_unproductive', 'ptq_capacity']]
    .first().to_dict('index')
)

# Test across templates and key lag/window combos
print('=' * 100)
print('Trait moderation of trial-level worry-pattern similarity × RT coupling')
print('  Within-subject: r between trial similarity (at lag/window) and trial RT')
print('  Across-subject: correlation with trait')
print('=' * 100)

LAGS_TO_TEST = [(-1, 3), (0, 2), (-2, 3)]
TEMPLATES = ['C1', 'C2', 'C3']
TRAITS = ['pswq_total', 'ptq_total', 'ptq_core',
          'ptq_unproductive', 'ptq_capacity']

results = []
for template in TEMPLATES:
    for lag, window in LAGS_TO_TEST:
        print(f'\n{template}, lag={lag} TR ({lag*2:+d}s), window={window} TR:')
        
        coupling = get_subject_trial_level_rt_coupling(
            template_kind=template, lag=lag, window=window,
        )
        if len(coupling) == 0:
            print('  No data')
            continue
        
        # Build DataFrame
        df_eff = pd.DataFrame([
            {'subject': sub, 'within_r': r,
             **{trait: trait_lookup.get(sub, {}).get(trait, np.nan)
                for trait in TRAITS}}
            for sub, r in coupling.items()
        ])
        df_eff = df_eff.dropna(subset=['within_r'])
        
        # Group test
        t_group, p_group = ttest_1samp(df_eff['within_r'], 0)
        sig_group = ('***' if p_group < 0.001 else '**' if p_group < 0.01
                      else '*' if p_group < 0.05 else '')
        print(f'  Group: M={df_eff["within_r"].mean():+.4f}, '
              f'SD={df_eff["within_r"].std():.4f}, '
              f'n={len(df_eff)}, '
              f't={t_group:+.2f}, p={p_group:.4f} {sig_group}')
        
        # Trait correlations
        for trait in TRAITS:
            df_valid = df_eff.dropna(subset=[trait])
            if len(df_valid) < 10:
                continue
            r, p = pearsonr(df_valid['within_r'], df_valid[trait])
            sig = ('***' if p < 0.001 else '**' if p < 0.01
                   else '*' if p < 0.05 else '†' if p < 0.10 else '')
            results.append({
                'template': template, 'lag': lag, 'window': window,
                'trait': trait, 'n_subj': len(df_valid),
                'group_mean_r': df_eff['within_r'].mean(),
                'group_t': t_group, 'group_p': p_group,
                'trait_r': r, 'trait_p': p,
            })
            print(f'    {trait:22s}  r = {r:+.3f}, p = {p:.4f}  {sig}')

results_df = pd.DataFrame(results)
results_df.to_csv(DECODING_OUT_DIR / 'trait_moderation_trial_rt_coupling.csv', index=False)

# Summary sorted by significance
print('\n' + '=' * 100)
print('SUMMARY: All trait correlations, sorted by p-value')
print('=' * 100)
print(results_df.sort_values('trait_p').round(4).to_string(index=False))

In [ ]:
trait_lookup = (
    block_df.groupby('subject')[['pswq_total', 'ptq_total', 'ptq_core',
                                   'ptq_unproductive', 'ptq_capacity']]
    .first().to_dict('index')
)

In [ ]:
# Compute average pre-trial coupling per subject per template
PRE_TRIAL_LAGS = [-3, -2, -1, 0]
WINDOW = 3

results = []
for tmpl in ['C1', 'C2', 'C3']:
    # Get coupling at each pre-trial lag
    per_subject_couplings = {sub: [] for sub in sart_stats.keys()}
    for lag in PRE_TRIAL_LAGS:
        coupling = get_subject_trial_level_rt_coupling(
            template_kind=tmpl, lag=lag, window=WINDOW,
        )
        for sub, r in coupling.items():
            per_subject_couplings[sub].append(r)
    
    # Average across pre-trial lags per subject
    avg_coupling = {sub: np.mean(rs) for sub, rs in per_subject_couplings.items() 
                     if len(rs) == len(PRE_TRIAL_LAGS)}
    
    # Build per-subject DataFrame with traits
    df_avg = pd.DataFrame([
        {'subject': sub, 'avg_pre_trial_r': r,
         **trait_lookup.get(sub, {})}
        for sub, r in avg_coupling.items()
    ]).dropna()
    
    # Group test
    rs_arr = df_avg['avg_pre_trial_r'].values
    t, p = ttest_1samp(rs_arr, 0)
    print(f'\n{tmpl} — averaged pre-trial coupling (lags -3 to 0):')
    print(f'  Group: M={rs_arr.mean():+.3f}, t={t:+.2f}, p={p:.4f}, n={len(rs_arr)}')
    
    # Trait correlations
    for trait in ['pswq_total', 'ptq_total', 'ptq_core', 
                   'ptq_unproductive', 'ptq_capacity']:
        valid = df_avg.dropna(subset=[trait])
        if len(valid) < 10:
            continue
        r, p_trait = pearsonr(valid['avg_pre_trial_r'], valid[trait])
        sig = ('***' if p_trait < 0.001 else '**' if p_trait < 0.01
               else '*' if p_trait < 0.05 else '†' if p_trait < 0.10 else '')
        print(f'    {trait:22s}  r = {r:+.3f}, p = {p_trait:.4f}  {sig}')
        results.append({'template': tmpl, 'trait': trait, 'r': r, 'p': p_trait,
                          'n': len(valid)})

pd.DataFrame(results).to_csv(DECODING_OUT_DIR / 'trait_moderation_pretrial_rt_avg.csv', 
                              index=False)

In [ ]:
# =============================================================================
# Figure: Trait moderation of trial-level RT coupling
# Within-subject sim × RT coupling, correlated with trait worry across subjects
# Three panels (C1/C2/C3), lag=-2 TR (4s pre-trial), window=3 TR
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Use PSWQ as the trait dimension (most consistent across templates)
TRAIT = 'pswq_total'
TRAIT_LABEL = 'PSWQ total\n(chronic worry)'
LAG = -2
WINDOW = 3

# Best-performing trait per template at lag=-2 (from output)
TEMPLATES = {
    'C1: same-trial':       'C1',
    'C2: subject-mean':     'C2',
    'C3: high-dyscontrol':  'C3',
}

# Trait data
trait_lookup = (
    block_df.groupby('subject')[[TRAIT]].first().to_dict('index')
)

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

DOT_COLOR = '#7791a8'
LINE_COLOR = '#1f3a5f'

for ax, (label, tmpl) in zip(axes, TEMPLATES.items()):
    coupling = get_subject_trial_level_rt_coupling(
        template_kind=tmpl, lag=LAG, window=WINDOW,
    )
    # Build plot dataframe
    plot_data = []
    for sub, r in coupling.items():
        trait_val = trait_lookup.get(sub, {}).get(TRAIT, np.nan)
        if not np.isnan(trait_val):
            plot_data.append({'subject': sub, 'within_r': r, 'trait': trait_val})
    plot_df = pd.DataFrame(plot_data)
    
    # Cross-subject correlation
    between_r, between_p = pearsonr(plot_df['trait'], plot_df['within_r'])
    
    # Scatter
    ax.scatter(plot_df['trait'], plot_df['within_r'],
               s=40, color=DOT_COLOR, alpha=0.7,
               edgecolor='white', linewidth=0.6, zorder=2)
    
    # Regression line
    xs = np.array([plot_df['trait'].min(), plot_df['trait'].max()])
    slope, intercept = np.polyfit(plot_df['trait'], plot_df['within_r'], 1)
    ax.plot(xs, slope * xs + intercept, color=LINE_COLOR, lw=2, zorder=3)
    
    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.6, zorder=1)
    
    sig = ('***' if between_p < 0.001 else '**' if between_p < 0.01
           else '*' if between_p < 0.05 else 'n.s.')
    ax.set_title(f'{label}\nr = {between_r:+.3f}, p = {between_p:.3f}{sig} '
                 f'(n = {len(plot_df)})',
                 fontsize=10)
    
    ax.set_xlabel(TRAIT_LABEL, fontsize=10)
    
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', length=4)

axes[0].set_ylabel('Within-subject Pearson r\n(worry-pattern similarity × Go-trial RT)',
                   fontsize=10)

plt.suptitle('Chronic worry sharpens within-subject coupling between '
             'pre-trial worry-state and response slowing\n'
             f'(at lag = {LAG} TR / {LAG*2:+d}s, window = {WINDOW} TR)',
             fontsize=11, y=1.04)
plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_trait_trial_rt_coupling_scatter.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# =============================================================================
# Figure: Peristimulus RT coupling, six panels
# Top row (a-c): trait moderation by PSWQ_total
# Bottom row (d-f): trait moderation by PTQ_total
# Single y-axis: group mean within-subject sim × RT coupling
# Point fill encodes: trait moderation p-value (uncorrected and Bonferroni)
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp
import matplotlib.lines as mlines

LAGS = [-3, -2, -1, 0, 1, 2, 3]
WINDOW = 3
TEMPLATES = {
    'C1: same-trial':       'C1',
    'C2: subject-mean':     'C2',
    'C3: high-dyscontrol':  'C3',
}
TRAITS_TO_PLOT = ['pswq_total', 'ptq_total']
TRAIT_LABELS = {
    'pswq_total': 'PSWQ total (chronic worry)',
    'ptq_total':  'PTQ total (perseverative thinking)',
}

# Trait data
trait_lookup = (
    block_df.groupby('subject')[TRAITS_TO_PLOT].first().to_dict('index')
)

# Compute per-lag, per-template, per-trait stats
print('Computing peristimulus RT coupling and trait moderation...')
results = []
for tmpl_label, tmpl in TEMPLATES.items():
    for lag in LAGS:
        coupling = get_subject_trial_level_rt_coupling(
            template_kind=tmpl, lag=lag, window=WINDOW,
        )
        # Group level (across all subjects with valid coupling)
        rs = np.array(list(coupling.values()))
        t_group, p_group = ttest_1samp(rs, 0)
        
        # Trait correlations
        trait_results = {}
        for trait in TRAITS_TO_PLOT:
            df_lag = pd.DataFrame([
                {'within_r': r, 'trait': trait_lookup.get(sub, {}).get(trait, np.nan)}
                for sub, r in coupling.items()
            ]).dropna()
            if len(df_lag) >= 10 and df_lag['trait'].std() > 0:
                tr_r, tr_p = pearsonr(df_lag['within_r'], df_lag['trait'])
            else:
                tr_r, tr_p = np.nan, np.nan
            trait_results[trait] = {'r': tr_r, 'p': tr_p, 'n': len(df_lag)}
        
        results.append({
            'template': tmpl_label,
            'tmpl_short': tmpl,
            'lag': lag,
            'time_s': lag * TR_SECONDS,
            'group_mean_r': rs.mean(),
            'group_sem_r': rs.std() / np.sqrt(len(rs)),
            'group_t': t_group,
            'group_p': p_group,
            'n_subj_group': len(rs),
            **{f'{t}_r': trait_results[t]['r'] for t in TRAITS_TO_PLOT},
            **{f'{t}_p': trait_results[t]['p'] for t in TRAITS_TO_PLOT},
            **{f'{t}_n': trait_results[t]['n'] for t in TRAITS_TO_PLOT},
        })

results_df = pd.DataFrame(results)
print('Done.\n')

# Bonferroni threshold for trait correlation across 7 lags
N_LAGS = len(LAGS)
BONF_ALPHA_TRAIT = 0.05 / N_LAGS

# --- Plot 2x3 ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)

LINE_COLOR = '#1f3a5f'
SIG_COLOR = '#C73C3C'
NS_COLOR = '#bbbbbb'

# Determine consistent y-axis
all_means = results_df['group_mean_r'].values
all_sems = results_df['group_sem_r'].values
y_min = (all_means - all_sems).min() - 0.02
y_max = (all_means + all_sems).max() + 0.04

for row_idx, trait in enumerate(TRAITS_TO_PLOT):
    for col_idx, (tmpl_label, tmpl) in enumerate(TEMPLATES.items()):
        ax = axes[row_idx, col_idx]
        sub_df = results_df[results_df['template'] == tmpl_label].sort_values('lag')
        
        times = sub_df['time_s'].values
        means = sub_df['group_mean_r'].values
        sems = sub_df['group_sem_r'].values
        trait_rs = sub_df[f'{trait}_r'].values
        trait_ps = sub_df[f'{trait}_p'].values
        
        # Line + SE shading
        ax.fill_between(times, means - sems, means + sems,
                         color=LINE_COLOR, alpha=0.18, zorder=1)
        ax.plot(times, means, color=LINE_COLOR, linewidth=1.5, zorder=2)
        
        # Points colored by trait significance
        for t, m, tr_r, tr_p in zip(times, means, trait_rs, trait_ps):
            if pd.isna(tr_p):
                ax.scatter(t, m, s=70, color=NS_COLOR,
                            edgecolor='black', linewidth=0.4, zorder=3)
                continue
            
            if tr_p < BONF_ALPHA_TRAIT:
                ax.scatter(t, m, s=110, color=SIG_COLOR,
                            edgecolor='black', linewidth=0.8, zorder=4)
            elif tr_p < 0.05:
                ax.scatter(t, m, s=90, color='white',
                            edgecolor=SIG_COLOR, linewidth=2, zorder=4)
            else:
                ax.scatter(t, m, s=70, color=NS_COLOR,
                            edgecolor='black', linewidth=0.4, zorder=3)
            
            # Annotate r value if trait sig
            if not pd.isna(tr_p) and tr_p < 0.05:
                ax.annotate(f'r = {tr_r:+.2f}',
                             xy=(t, m), xytext=(0, 14),
                             textcoords='offset points',
                             ha='center', fontsize=8, color=SIG_COLOR)
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle=':', alpha=0.7, zorder=1)
        ax.axvline(0, color='gray', linewidth=0.5, linestyle=':', alpha=0.7, zorder=1)
        
        # Panel labels
        panel_letter = chr(ord('a') + row_idx * 3 + col_idx)
        title = f'{panel_letter}. {tmpl_label}'
        ax.set_title(title, fontsize=10, loc='left')
        
        # X axis only on bottom row
        if row_idx == len(TRAITS_TO_PLOT) - 1:
            ax.set_xlabel('Time relative to trial onset (s)', fontsize=10)
        
        # Y axis label on first column
        if col_idx == 0:
            ax.set_ylabel(f'Coupling: worry sim × RT\n'
                           f'(trait: {TRAIT_LABELS[trait].split(" (")[0]})',
                           fontsize=9)
        
        ax.set_ylim(y_min, y_max)
        for spine in ['top', 'right']:
            ax.spines[spine].set_visible(False)
        ax.tick_params(axis='both', length=4)

# Legend
legend_handles = [
    mlines.Line2D([], [], color=LINE_COLOR, linewidth=1.8,
                   label='Group mean coupling (shading = SE)'),
    mlines.Line2D([], [], marker='o', markersize=10, color='lightgray',
                   markeredgecolor='black', markeredgewidth=0.5, linewidth=0,
                   label='n.s. trait moderation'),
    mlines.Line2D([], [], marker='o', markersize=10, color='white',
                   markeredgecolor=SIG_COLOR, markeredgewidth=2, linewidth=0,
                   label='p < 0.05 (uncorrected)'),
    mlines.Line2D([], [], marker='o', markersize=11, color=SIG_COLOR,
                   markeredgecolor='black', markeredgewidth=0.8, linewidth=0,
                   label=f'p < 0.05 (Bonferroni-corrected across {N_LAGS} TRs)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=9)

plt.suptitle('Trial-level worry-pattern reactivation predicts Go-trial RT slowing\n'
             'with stronger pre-trial coupling in chronic worriers and perseverative thinkers',
             fontsize=11, y=1.00)
plt.tight_layout()
plt.subplots_adjust(bottom=0.10, top=0.92, hspace=0.30)

fig_path = FIGURES_DIR / 'fig_trait_trial_rt_coupling_peristimulus_2trait.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# Section F: resting state analyses

In [ ]:
# =============================================================================
# Section F0: Parcellate resting-state data
# Uses same FEAT-like-no-smoothing config (motion-24 + cosine, no aCompCor)
# Output: <DERIVATIVES_DIR>/parcels/yeo_2011_rest/sub-XXX_run-01_parcel_ts.npy
# =============================================================================
import os
import time
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
from parcellation import (
    load_atlas, build_confound_matrix, compute_spike_mask, parcellate_run
)
from config import (
    ATLAS_KWARGS, FMRIPREP_ROOT, CONFOUND_GROUPS, ACTIVE_CONFOUND_GROUPS,
    SMOOTHING_FWHM, FD_SPIKE_THRESHOLD
)

REST_TASK = 'restingstate'
REST_ATLAS_SUBDIR = f'{ATLAS_NAME}_rest'
REST_PARCELS_DIR = PARCELS_DIR / REST_ATLAS_SUBDIR
REST_PARCELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Rest parcels output: {REST_PARCELS_DIR}')

# Load atlas
atlas_img, atlas_labels = load_atlas(ATLAS_NAME, ATLAS_KWARGS)

# Find subjects with rest data
all_rest_subjects = []
for sub_dir in sorted(os.listdir(FMRIPREP_ROOT)):
    if not sub_dir.startswith('sub-'):
        continue
    # Skip HTML reports and other non-directory items
    sub_path = FMRIPREP_ROOT / sub_dir
    if not sub_path.is_dir():
        continue
    sub_num = int(sub_dir.split('-')[1])
    if sub_num in EXCLUDE_SUBJECTS:
        continue
    
    bold_path = (FMRIPREP_ROOT / sub_dir / SESSION / 'func' /
                  f'{sub_dir}_{SESSION}_task-{REST_TASK}_run-01'
                  f'_space-MNI152NLin6Asym_res-2_desc-preproc_bold.nii.gz')
    confounds_path = (FMRIPREP_ROOT / sub_dir / SESSION / 'func' /
                       f'{sub_dir}_{SESSION}_task-{REST_TASK}_run-01'
                       f'_desc-confounds_timeseries.tsv')
    mask_path = (FMRIPREP_ROOT / sub_dir / SESSION / 'func' /
                  f'{sub_dir}_{SESSION}_task-{REST_TASK}_run-01'
                  f'_space-MNI152NLin6Asym_res-2_desc-brain_mask.nii.gz')
    
    if bold_path.exists() and confounds_path.exists():
        all_rest_subjects.append({
            'subject': sub_num,
            'bold': bold_path,
            'confounds': confounds_path,
            'mask': mask_path if mask_path.exists() else None,
        })

print(f'Found rest data for {len(all_rest_subjects)} subjects')

def parcellate_rest_run(sub_info, overwrite=False):
    sub = sub_info['subject']
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    
    if ts_path.exists() and spike_path.exists() and not overwrite:
        return 'cached', None
    
    # Unpack the tuple return
    confounds_df, _ = build_confound_matrix(
        confounds_path=sub_info['confounds'],
        confound_groups=CONFOUND_GROUPS,
        active_groups=ACTIVE_CONFOUND_GROUPS,
    )
    
    parcel_ts = parcellate_run(
        bold_path=sub_info['bold'],
        atlas_img=atlas_img,
        mask_path=sub_info['mask'],
        confounds_df=confounds_df,
        tr=TR_SECONDS,
        smoothing_fwhm=SMOOTHING_FWHM,
    )
    
    spike_mask = compute_spike_mask(
        confounds_path=sub_info['confounds'],
        fd_threshold=FD_SPIKE_THRESHOLD,
    )
    
    np.save(ts_path, parcel_ts)
    np.save(spike_path, spike_mask)
    return 'parcellated', parcel_ts.shape


# Run
results = {'parcellated': 0, 'cached': 0, 'failed': 0}
for i, sub_info in enumerate(all_rest_subjects):
    sub = sub_info['subject']
    start = time.time()
    try:
        status, shape = parcellate_rest_run(sub_info)
        results[status] += 1
        dur = time.time() - start
        shape_str = f' shape={shape}' if shape else ''
        print(f'[{i+1}/{len(all_rest_subjects)}] sub-{sub}: {status}{shape_str} '
              f'({dur:.1f}s) at {datetime.now().strftime("%H:%M:%S")}')
    except Exception as e:
        results['failed'] += 1
        print(f'[{i+1}/{len(all_rest_subjects)}] sub-{sub}: FAILED — {e}')

print(f'\nSummary: {results}')
print(f'Rest parcels saved to: {REST_PARCELS_DIR}')

In [ ]:
# =============================================================================
# Section F: Resting-state evidence
# F0a: Build TB templates (C2 subject-mean, C3 high-dyscontrol mean,
#                          C3_gradient continuous dyscontrol-weighted)
# =============================================================================
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, ttest_1samp

# Confirm rest parcellation done
REST_PARCELS_DIR = PARCELS_DIR / f'{ATLAS_NAME}_rest'
print(f'Rest parcels: {REST_PARCELS_DIR}')
print(f'Files: {len(list(REST_PARCELS_DIR.glob("*_parcel_ts.npy")))}')

# Build templates per subject
# Restrict to subjects with both task and rest data
task_subjects = set(sart_blocks['subject'].unique())
rest_subjects_with_data = {
    int(f.stem.split('_')[0].split('-')[1])
    for f in REST_PARCELS_DIR.glob('*_parcel_ts.npy')
}
analysis_subjects = sorted(task_subjects & rest_subjects_with_data)
print(f'\nSubjects in task analysis: {len(task_subjects)}')
print(f'Subjects with rest data: {len(rest_subjects_with_data)}')
print(f'Subjects in both: {len(analysis_subjects)}')

# Build templates
tb_subject_mean = {}        # C2
tb_high_mean = {}           # C3 (above-median dyscontrol)
tb_gradient = {}            # C3_gradient (continuous dyscontrol-weighted)

for sub in analysis_subjects:
    sub_tb = tb_blocks[tb_blocks['subject'] == sub].copy()
    if len(sub_tb) < 3:
        continue
    
    patterns = sub_tb[parcel_cols].values.astype(float)
    dyscontrol = sub_tb['dyscontrol_imputed'].values.astype(float)
    
    # C2: subject-mean
    tb_subject_mean[sub] = patterns.mean(axis=0)
    
    # C3: above-median dyscontrol mean
    median_dyscontrol = np.median(dyscontrol)
    high_mask = dyscontrol > median_dyscontrol
    if high_mask.sum() >= 2:
        tb_high_mean[sub] = patterns[high_mask].mean(axis=0)
    
    # C3_gradient: continuous-weighted
    # Within-subject mean-center dyscontrol, then weight patterns
    weights = dyscontrol - dyscontrol.mean()
    if np.abs(weights).sum() > 0:
        tb_gradient[sub] = (patterns * weights[:, None]).sum(axis=0) / np.abs(weights).sum()

print(f'\nTemplates built:')
print(f'  C2 (subject-mean): {len(tb_subject_mean)} subjects')
print(f'  C3 (high-dyscontrol mean): {len(tb_high_mean)} subjects')
print(f'  C3_gradient (continuous-weighted): {len(tb_gradient)} subjects')


# =============================================================================
# F1: Mean similarity at rest to each template
# =============================================================================
print('\n' + '=' * 80)
print('F1: Mean similarity at rest to TB templates')
print('=' * 80)

# Recompute SART stats for z-scoring (same as task analyses)
# sart_stats should already be in memory; if not, rebuild
if 'sart_stats' not in dir():
    sart_stats = {}
    for sub, g in sart_blocks.groupby('subject'):
        patterns_sart = g[parcel_cols].values.astype(float)
        sd = patterns_sart.std(axis=0)
        sd[sd == 0] = 1
        sart_stats[sub] = {'mean': patterns_sart.mean(axis=0), 'sd': sd}

metric_fn = get_metric(SIMILARITY_METRIC)

# Compute mean rest similarity per subject per template
rest_similarity = []
for sub in analysis_subjects:
    # Load rest data
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    if not ts_path.exists():
        continue
    rest_ts = np.load(ts_path)
    spike_mask = np.load(spike_path) if spike_path.exists() else np.ones(len(rest_ts), dtype=bool)
    
    # Use only non-spike TRs
    valid_trs = ~spike_mask
    if valid_trs.sum() < 30:
        continue
    
    if sub not in sart_stats:
        continue
    
    # Z-score against SART distribution (consistency with task analyses)
    rest_z = (rest_ts - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
    
    row = {'subject': sub, 'n_valid_trs': int(valid_trs.sum())}
    for tmpl_name, tmpl_dict in [
        ('C2', tb_subject_mean), ('C3', tb_high_mean), ('C3_grad', tb_gradient)
    ]:
        if sub not in tmpl_dict:
            row[f'mean_sim_{tmpl_name}'] = np.nan
            continue
        template_z = (tmpl_dict[sub] - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
        sims = np.array([metric_fn(rest_z[t], template_z) for t in range(len(rest_z)) if valid_trs[t]])
        row[f'mean_sim_{tmpl_name}'] = sims.mean()
        row[f'std_sim_{tmpl_name}'] = sims.std()
        row[f'median_sim_{tmpl_name}'] = np.median(sims)
    
    rest_similarity.append(row)

rest_sim_df = pd.DataFrame(rest_similarity)
print(f'\nRest similarity computed for {len(rest_sim_df)} subjects')
print(rest_sim_df[['subject', 'n_valid_trs', 'mean_sim_C2', 'mean_sim_C3', 
                     'mean_sim_C3_grad']].describe().round(4))

# Merge with trait data
trait_cols = ['pswq_total', 'ptq_total', 'ptq_core', 'ptq_unproductive', 'ptq_capacity']
sub_traits = block_df.groupby('subject')[trait_cols].first().reset_index()
rest_sim_df = rest_sim_df.merge(sub_traits, on='subject', how='left')

# Group test: does rest look like each template?
print('\n--- Group: does rest look like each template? ---')
for tmpl in ['C2', 'C3', 'C3_grad']:
    col = f'mean_sim_{tmpl}'
    vals = rest_sim_df[col].dropna().values
    t, p = ttest_1samp(vals, 0)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f'  {tmpl:10s}  M={vals.mean():+.4f}  SD={vals.std():.4f}  '
           f'n={len(vals)}  t={t:+.2f}  p={p:.4f}  {sig}')

# Trait correlations: chronic worriers' rest looks more like worry?
print('\n--- Trait correlations: rest similarity × trait worry ---')
trait_results = []
for tmpl in ['C2', 'C3', 'C3_grad']:
    col = f'mean_sim_{tmpl}'
    print(f'\n{tmpl}:')
    for trait in trait_cols:
        valid = rest_sim_df.dropna(subset=[col, trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid[col], valid[trait])
        sig = ('***' if p < 0.001 else '**' if p < 0.01
               else '*' if p < 0.05 else '†' if p < 0.10 else '')
        print(f'  {trait:22s}  r = {r:+.3f}  p = {p:.4f}  n={len(valid)}  {sig}')
        trait_results.append({
            'template': tmpl, 'trait': trait,
            'r': r, 'p': p, 'n': len(valid),
        })

trait_results_df = pd.DataFrame(trait_results)
rest_sim_df.to_csv(DECODING_OUT_DIR / 'rest_similarity_F1.csv', index=False)
trait_results_df.to_csv(DECODING_OUT_DIR / 'rest_similarity_F1_trait_corrs.csv', index=False)
print(f'\nSaved: rest_similarity_F1.csv')

# Differential: does sticky-thought similarity exceed mean-pattern similarity?
print('\n--- Differential: C3_grad similarity correlated with traits, controlling for C2 ---')
# Partial r: trait × C3_grad similarity, controlling for C2 similarity
from scipy.stats import pearsonr
import numpy as np

def partial_pearson(x, y, z):
    """Pearson r between x and y, partialling out z."""
    # Residualize each from z
    x_arr, y_arr, z_arr = np.array(x), np.array(y), np.array(z)
    bx = np.polyfit(z_arr, x_arr, 1)
    by = np.polyfit(z_arr, y_arr, 1)
    x_res = x_arr - (bx[0] * z_arr + bx[1])
    y_res = y_arr - (by[0] * z_arr + by[1])
    return pearsonr(x_res, y_res)

for trait in trait_cols:
    valid = rest_sim_df.dropna(subset=['mean_sim_C3_grad', 'mean_sim_C2', trait])
    if len(valid) < 10:
        continue
    r_part, p_part = partial_pearson(
        valid['mean_sim_C3_grad'], valid[trait], valid['mean_sim_C2']
    )
    sig = ('*' if p_part < 0.05 else '†' if p_part < 0.10 else '')
    print(f'  {trait:22s}  partial r = {r_part:+.3f}  p = {p_part:.4f}  n={len(valid)}  {sig}')

In [ ]:
# =============================================================================
# F2: Temporal dynamics of rest worry-pattern state
# Worry state = TRs where rest pattern Pearson-correlates positively (r > 0)
# with subject's C3 high-dyscontrol template
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_1samp

print('=' * 80)
print('F2: Temporal dynamics of rest worry-pattern state')
print('(Worry state = TRs where rest pattern positively correlates with C3 template)')
print('=' * 80)

THRESHOLD = 0.0  # Pearson r threshold for "in worry state"

rest_dynamics = []
for sub in analysis_subjects:
    if sub not in tb_high_mean or sub not in sart_stats:
        continue
    
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    if not ts_path.exists():
        continue
    
    rest_ts = np.load(ts_path)
    spike_mask = np.load(spike_path) if spike_path.exists() else np.zeros(len(rest_ts), dtype=bool)
    valid = ~spike_mask
    if valid.sum() < 60:
        continue
    
    rest_z = (rest_ts - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
    template_z = (tb_high_mean[sub] - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
    
    sims = np.array([metric_fn(rest_z[t], template_z) for t in range(len(rest_z))])
    sims_valid = sims[valid]
    
    in_state = sims_valid > THRESHOLD
    
    # Fraction of time in worry state
    frac_in_state = in_state.mean()
    
    # Number of entries
    transitions = np.diff(in_state.astype(int))
    n_entries = (transitions == 1).sum()
    
    # Dwell times
    runs = []
    in_run = False
    run_len = 0
    for v in in_state:
        if v:
            if not in_run:
                in_run = True
                run_len = 1
            else:
                run_len += 1
        else:
            if in_run:
                runs.append(run_len)
                in_run = False
                run_len = 0
    if in_run:
        runs.append(run_len)
    mean_dwell = np.mean(runs) if runs else 0
    median_dwell = np.median(runs) if runs else 0
    max_dwell = np.max(runs) if runs else 0
    log_mean_dwell = np.log1p(mean_dwell)  # log-transformed for skew
    log_max_dwell = np.log1p(max_dwell)
    
    # Similarity while in state
    mean_sim_in_state = sims_valid[in_state].mean() if in_state.any() else np.nan
    
    rest_dynamics.append({
        'subject': sub,
        'n_valid_trs': int(valid.sum()),
        'frac_in_state': frac_in_state,
        'n_entries': n_entries,
        'mean_dwell_trs': mean_dwell,
        'median_dwell_trs': median_dwell,
        'max_dwell_trs': max_dwell,
        'log_mean_dwell': log_mean_dwell,
        'log_max_dwell': log_max_dwell,
        'mean_sim_in_state': mean_sim_in_state,
    })

rest_dyn_df = pd.DataFrame(rest_dynamics)
print(f'\nDynamics computed for {len(rest_dyn_df)} subjects')
print(rest_dyn_df.describe().round(3))

# Merge traits
rest_dyn_df = rest_dyn_df.merge(sub_traits, on='subject', how='left')

# Trait correlations — compute and print ALL
METRICS = ['frac_in_state', 'n_entries', 'mean_dwell_trs', 'median_dwell_trs',
           'max_dwell_trs', 'log_mean_dwell', 'log_max_dwell', 'mean_sim_in_state']

print('\n--- Trait correlations ---')
results = []
for metric in METRICS:
    print(f'\n{metric}:')
    for trait in trait_cols:
        valid = rest_dyn_df.dropna(subset=[metric, trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid[metric], valid[trait])
        sig = ('***' if p < 0.001 else '**' if p < 0.01
               else '*' if p < 0.05 else '†' if p < 0.10 else '')
        print(f'  {trait:22s}  r = {r:+.3f}  p = {p:.4f}  n={len(valid)}  {sig}')
        results.append({
            'metric': metric, 'trait': trait,
            'r': r, 'p': p, 'n': len(valid),
        })

results_df = pd.DataFrame(results)
rest_dyn_df.to_csv(DECODING_OUT_DIR / 'rest_dynamics_F2.csv', index=False)
results_df.to_csv(DECODING_OUT_DIR / 'rest_dynamics_F2_trait_corrs.csv', index=False)

print('\n' + '=' * 80)
print('Top trait correlations (all metrics, sorted by p):')
print('=' * 80)
print(results_df.sort_values('p').head(20).round(4).to_string(index=False))

In [ ]:
# =============================================================================
# F3 (revised): Individual thought reactivation at rest
# For each rest TR, find which of subject's 9 TB patterns it most resembles
# 
# Key questions:
# 1. Does rest disproportionately favor high-dyscontrol thoughts?
# 2. Is this stronger in chronic worriers?
# 3. What are the dwell/switching dynamics?
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

print('=' * 80)
print('F3: Individual thought reactivation at rest')
print('=' * 80)

# Build per-thought templates
per_thought_templates = {}
per_thought_dyscontrol = {}

for sub in analysis_subjects:
    sub_tb = tb_blocks[tb_blocks['subject'] == sub].copy()
    if len(sub_tb) < 3:
        continue
    
    by_thought = {}
    dys_by_thought = {}
    for tk, g in sub_tb.groupby('thought_key'):
        patterns = g[parcel_cols].values.astype(float)
        by_thought[tk] = patterns.mean(axis=0)
        dys_by_thought[tk] = g['dyscontrol_imputed'].mean()
    per_thought_templates[sub] = by_thought
    per_thought_dyscontrol[sub] = dys_by_thought


thought_dynamics = []
for sub in analysis_subjects:
    if sub not in per_thought_templates or sub not in sart_stats:
        continue
    
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    if not ts_path.exists():
        continue
    rest_ts = np.load(ts_path)
    spike_mask = np.load(spike_path) if spike_path.exists() else np.zeros(len(rest_ts), dtype=bool)
    valid = ~spike_mask
    if valid.sum() < 60:
        continue
    
    rest_z = (rest_ts - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
    
    thoughts = sorted(per_thought_templates[sub].keys())
    if len(thoughts) < 3:
        continue
    
    template_array = np.array([
        (per_thought_templates[sub][tk] - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
        for tk in thoughts
    ])
    dyscontrol_array = np.array([per_thought_dyscontrol[sub][tk] for tk in thoughts])
    
    # Rank thoughts by dyscontrol (0 = lowest, n_thoughts-1 = highest)
    dyscontrol_ranks = np.argsort(np.argsort(dyscontrol_array))  # ranks
    
    # At each TR, find winning thought
    winning_thoughts = []
    winning_sims = []
    for t in range(len(rest_z)):
        if not valid[t]:
            winning_thoughts.append(-1)
            winning_sims.append(np.nan)
            continue
        sims_at_t = np.array([metric_fn(rest_z[t], template_array[i])
                                for i in range(len(thoughts))])
        winner = int(np.argmax(sims_at_t))
        winning_thoughts.append(winner)
        winning_sims.append(sims_at_t[winner])
    winning_thoughts = np.array(winning_thoughts)
    winning_sims = np.array(winning_sims)
    
    valid_mask = winning_thoughts >= 0
    valid_winners = winning_thoughts[valid_mask]
    valid_sims = winning_sims[valid_mask]
    if len(valid_winners) < 30:
        continue
    
    # === SCALAR METRICS ===
    
    # 1. Weighted mean dyscontrol (already had this)
    counts = np.bincount(valid_winners, minlength=len(thoughts))
    probs = counts / counts.sum()
    weighted_mean_dyscontrol = (probs * dyscontrol_array).sum()
    
    # 2. Weighted rank: what's the average rank of winning thoughts?
    # Higher = rest favors high-dyscontrol thoughts
    weighted_mean_rank = (probs * dyscontrol_ranks).sum()
    
    # 3. Sticky-thought share: fraction of TRs at top-3 highest-dyscontrol thoughts
    top_k = 3 if len(thoughts) >= 6 else max(1, len(thoughts) // 3)
    top_thought_indices = np.argsort(dyscontrol_array)[-top_k:]
    sticky_share = sum(counts[i] for i in top_thought_indices) / counts.sum()
    
    # 4. Spearman correlation between thought-count and dyscontrol
    # Per-subject: do high-dyscontrol thoughts get more TRs?
    if dyscontrol_array.std() > 0 and counts.std() > 0:
        sticky_match_r, _ = spearmanr(counts, dyscontrol_array)
    else:
        sticky_match_r = np.nan
    
    # 5. Entropy (concentration)
    nonzero = probs[probs > 0]
    entropy_actual = -np.sum(nonzero * np.log(nonzero))
    max_entropy = np.log(len(thoughts))
    entropy_norm = entropy_actual / max_entropy
    
    # 6. Top thought share
    top_thought_share = counts.max() / counts.sum()
    
    # 7. Dwell time per thought
    transitions = np.diff(valid_winners)
    n_switches = (transitions != 0).sum()
    
    runs = []
    if len(valid_winners) > 0:
        current_winner = valid_winners[0]
        current_run = 1
        for w in valid_winners[1:]:
            if w == current_winner:
                current_run += 1
            else:
                runs.append(current_run)
                current_run = 1
                current_winner = w
        runs.append(current_run)
    mean_dwell = np.mean(runs) if runs else 0
    
    # 8. Mean dwell at high-dyscontrol thoughts specifically
    dwell_at_sticky = []
    dwell_at_nonsticky = []
    current_winner = valid_winners[0]
    current_run = 1
    for w in valid_winners[1:]:
        if w == current_winner:
            current_run += 1
        else:
            if current_winner in top_thought_indices:
                dwell_at_sticky.append(current_run)
            else:
                dwell_at_nonsticky.append(current_run)
            current_winner = w
            current_run = 1
    # final run
    if current_winner in top_thought_indices:
        dwell_at_sticky.append(current_run)
    else:
        dwell_at_nonsticky.append(current_run)
    mean_dwell_sticky = np.mean(dwell_at_sticky) if dwell_at_sticky else np.nan
    mean_dwell_nonsticky = np.mean(dwell_at_nonsticky) if dwell_at_nonsticky else np.nan
    
    thought_dynamics.append({
        'subject': sub,
        'n_valid_trs': int(valid.sum()),
        'n_thoughts': len(thoughts),
        'weighted_mean_dyscontrol': weighted_mean_dyscontrol,
        'weighted_mean_rank': weighted_mean_rank,
        'sticky_share': sticky_share,
        'sticky_match_r': sticky_match_r,
        'entropy_norm': entropy_norm,
        'top_thought_share': top_thought_share,
        'n_switches': n_switches,
        'mean_dwell_per_thought': mean_dwell,
        'mean_dwell_sticky': mean_dwell_sticky,
        'mean_dwell_nonsticky': mean_dwell_nonsticky,
    })

thought_dyn_df = pd.DataFrame(thought_dynamics)
print(f'\nThought dynamics computed for {len(thought_dyn_df)} subjects')
print(thought_dyn_df.describe().round(3))

# Merge traits
thought_dyn_df = thought_dyn_df.merge(sub_traits, on='subject', how='left')

# === ANALYSIS 1: Group-level — do high-dyscontrol thoughts dominate rest? ===
print('\n' + '=' * 80)
print('Group-level: do high-dyscontrol thoughts get more time at rest?')
print('=' * 80)
from scipy.stats import ttest_1samp

# weighted_mean_rank: if random, expected = (n_thoughts-1)/2 ≈ 4 (for 9 thoughts)
# So we test against null = (n_thoughts-1)/2
# But thought counts vary, so simpler: test sticky_match_r against 0
match_rs = thought_dyn_df['sticky_match_r'].dropna().values
t, p = ttest_1samp(match_rs, 0)
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
print(f'Spearman r (count × dyscontrol) across subjects:')
print(f'  Mean = {match_rs.mean():+.4f}, t = {t:+.2f}, p = {p:.4f}  {sig}')
print(f'  (Positive = rest favors high-dyscontrol thoughts; group-level test)')

# Sticky share vs chance
# If random, top-k of n thoughts would account for k/n of TRs
sticky_shares = thought_dyn_df['sticky_share'].dropna().values
n_thoughts_arr = thought_dyn_df['n_thoughts'].dropna().values
top_ks = np.array([3 if n >= 6 else max(1, n // 3) for n in n_thoughts_arr])
expected_share = (top_ks / n_thoughts_arr).mean()
t, p = ttest_1samp(sticky_shares, expected_share)
sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
print(f'\nSticky share (top-3 highest-dyscontrol thoughts):')
print(f'  Observed mean = {sticky_shares.mean():.4f}, '
      f'expected by chance = {expected_share:.4f}, '
      f't = {t:+.2f}, p = {p:.4f}  {sig}')

# === ANALYSIS 2: Trait correlations ===
print('\n' + '=' * 80)
print('Trait correlations on thought-reactivation metrics')
print('=' * 80)

METRICS_F3 = ['weighted_mean_dyscontrol', 'weighted_mean_rank', 'sticky_share',
               'sticky_match_r', 'entropy_norm', 'top_thought_share',
               'n_switches', 'mean_dwell_per_thought',
               'mean_dwell_sticky', 'mean_dwell_nonsticky']

trait_thought_results = []
for metric in METRICS_F3:
    print(f'\n{metric}:')
    for trait in trait_cols:
        valid = thought_dyn_df.dropna(subset=[metric, trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid[metric], valid[trait])
        sig = ('***' if p < 0.001 else '**' if p < 0.01
               else '*' if p < 0.05 else '†' if p < 0.10 else '')
        if abs(r) > 0.2 or p < 0.10:
            print(f'  {trait:22s}  r = {r:+.3f}  p = {p:.4f}  n={len(valid)}  {sig}')
        trait_thought_results.append({
            'metric': metric, 'trait': trait, 'r': r, 'p': p, 'n': len(valid),
        })

trait_thought_df = pd.DataFrame(trait_thought_results)
thought_dyn_df.to_csv(DECODING_OUT_DIR / 'rest_thought_dynamics_F3.csv', index=False)
trait_thought_df.to_csv(DECODING_OUT_DIR / 'rest_thought_dynamics_F3_trait_corrs.csv', index=False)

print('\n' + '=' * 80)
print('Top trait correlations:')
print('=' * 80)
print(trait_thought_df.sort_values('p').head(20).round(4).to_string(index=False))

In [ ]:
# Just sanity check
for metric in ['entropy_norm', 'top_thought_share', 'n_switches',
                'mean_dwell_per_thought', 'mean_dwell_sticky', 'mean_dwell_nonsticky']:
    print(f'\n{metric}:')
    for trait in trait_cols:
        valid = thought_dyn_df.dropna(subset=[metric, trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid[metric], valid[trait])
        print(f'  {trait:22s}  r = {r:+.3f}  p = {p:.4f}  n={len(valid)}')

(Explanation of the above):
The setup
For each subject, we built 9 thought-specific templates: one neural pattern per thought topic, averaged across that subject's 3 TB blocks for that thought (3 runs × 9 thoughts = 27 TB blocks total, averaged to 9 per-thought patterns).
For "burger king" we average the brain patterns from the 3 TB blocks where this subject thought about burger king. We do this for all 9 thoughts. Now we have 9 patterns per subject, each representing "what your brain looks like when thinking about thought X."
We also know each thought's dyscontrol_imputed score — how sticky each thought was for this subject (from the prescan rating).
Winning thought
For each TR of resting state (240 TRs total), we compute the Pearson correlation between the brain pattern at that TR and each of the 9 thought templates.
For TR=42: similarity to "burger king" pattern = 0.21, to "unzipped pockets" = 0.05, to "after graduation" = 0.34, etc.
The winning thought for TR=42 is whichever template has the highest similarity. In this example, "after graduation" wins at TR=42.
So at every rest TR, exactly one thought "wins" — the one whose template the brain pattern most resembles.
This gives us a time series like:
TR:    1  2  3  4  5  6  7  8  9  10 ...
Win:   3  3  3  7  7  2  2  2  2  2 ...   (thought indices)
Now the metrics
For each subject, we look at this winning-thought time series and compute:
Selection metrics (which thoughts win)
1. weighted_mean_dyscontrol

Multiply: (proportion of TRs where each thought wins) × (that thought's dyscontrol score)
Sum across all 9 thoughts
Interpretation: average dyscontrol of the thoughts that won, weighted by how often they won
A subject who spends all rest at their stickiest thought = high value. Spending all rest at their least-sticky thought = low value.

2. weighted_mean_rank

Same as above but using ranks (0 to 8) instead of raw dyscontrol values
More robust to outlier dyscontrol values within subject
A subject visiting the highest-ranked (stickiest) thought has a high weighted_mean_rank

3. sticky_share

Take the top 3 thoughts by dyscontrol per subject
Compute: fraction of rest TRs where one of these 3 thoughts wins
If random: ≈ 3/9 = 0.33
If chronic worriers favor sticky thoughts: > 0.33

4. sticky_match_r

For each subject: compute Spearman correlation between (count of TRs per thought) and (dyscontrol of that thought)
Positive r = thoughts with higher dyscontrol get more TRs
Negative r = thoughts with higher dyscontrol get fewer TRs
This is per-subject — measures within-subject preference for sticky thoughts

Structural metrics (HOW thoughts are visited)
5. entropy_norm

Shannon entropy of the distribution of thoughts (normalized by max possible)
0 = always one thought (totally concentrated)
1 = uniform distribution (all thoughts equally likely)
Tells you: how spread out across thoughts is this subject's rest?

6. top_thought_share

The most-visited thought's share of total TRs
Different from sticky_share: this asks "how dominant is the #1 thought regardless of which thought it is"
High value = one thought dominates rest; low value = spread out

7. n_switches

Number of times the winning thought changed from one TR to the next
High switches = frequent transitions; low switches = long stays at each thought

8. mean_dwell_per_thought

When a thought wins, how many consecutive TRs does it keep winning on average?
Computed by tracking "runs" — periods of consecutive TRs at the same thought
If thought 3 wins at TRs 1-2-3, switches to thought 7, that's a run of length 3 at thought 3
Average across all runs gives mean_dwell

9. mean_dwell_sticky

Same as mean_dwell_per_thought but restricted to runs at thoughts in the top-3 by dyscontrol
"When you DO end up at a sticky thought, how long do you stay?"

10. mean_dwell_nonsticky

Same but for thoughts NOT in top-3 by dyscontrol
"When you DO end up at a non-sticky thought, how long do you stay?"

In [ ]:
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

# Adjust these to your actual column/dataframe names.
# `thought_dyn_df` should have one row per subject with the per-subject metrics
# and trait columns merged in (or merge with sub_traits if not already).
df = thought_dyn_df.merge(sub_traits, on='subject', how='left', suffixes=('', '_y'))

METRICS = {
    'dyscontrol-weighted mean':   'weighted_mean_dyscontrol',   # adjust col name
    'share of top-3 dyscontrol':  'sticky_share',                # adjust col name
    # 'sticky_match_r' was already corrected in earlier output, but include
    # here too as a sanity check:
    'thought-dyscontrol match':   'sticky_match_r',
}

TRAIT_COLS = ['pswq_total', 'ptq_total', 'ptq_core',
              'ptq_unproductive', 'ptq_capacity']

for metric_label, metric_col in METRICS.items():
    if metric_col not in df.columns:
        print(f'SKIP: {metric_col} not in df')
        continue
    pvals = {}
    rvals = {}
    for trait in TRAIT_COLS:
        valid = df.dropna(subset=[metric_col, trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid[metric_col], valid[trait])
        pvals[trait] = p
        rvals[trait] = r
    if not pvals:
        continue
    names = list(pvals.keys())
    ps = np.array(list(pvals.values()))
    reject, p_corr, _, _ = multipletests(ps, alpha=0.05, method='fdr_bh')
    print(f'\n=== {metric_label} (BH-FDR, n_tests={len(ps)}) ===')
    for n, p_unc, p_c, r_flag in zip(names, ps, p_corr, reject):
        sig = ' *' if r_flag else ''
        print(f'  {n:22s}  r={rvals[n]:+.3f}  p_unc={p_unc:.4f}   p_corr={p_c:.4f}{sig}')

In [ ]:
# === Reconstruct illustrative_df and exemplars (defining cell was lost) ===
# Original definition from the F3 figure build.

# Build illustrative_df: thought-dynamics merged with PSWQ, guarding against
# a name collision (thought_dyn_df may already carry pswq_total).
if 'pswq_total' not in thought_dyn_df.columns:
    illustrative_df = thought_dyn_df.merge(
        block_df.groupby('subject')[['pswq_total']].first().reset_index(),
        on='subject', how='left'
    )
else:
    illustrative_df = thought_dyn_df.copy()

illustrative_df = illustrative_df.dropna(subset=['pswq_total', 'sticky_match_r'])

# Exemplar selection: joint rank on PSWQ and sticky_match_r.
# High exemplar = high on both (chronic worrier whose rest tracks sticky thoughts);
# low exemplar = low on both.
illustrative_df['pswq_rank']   = illustrative_df['pswq_total'].rank(pct=True)
illustrative_df['sticky_rank'] = illustrative_df['sticky_match_r'].rank(pct=True)
illustrative_df['joint_score']      = illustrative_df['pswq_rank'] + illustrative_df['sticky_rank']
illustrative_df['anti_joint_score'] = (1 - illustrative_df['pswq_rank']) + (1 - illustrative_df['sticky_rank'])

exemplar_high = illustrative_df.nlargest(1, 'joint_score').iloc[0]
exemplar_low  = illustrative_df.nlargest(1, 'anti_joint_score').iloc[0]

print(f'High-trait exemplar: sub-{int(exemplar_high["subject"])}, '
      f'PSWQ={exemplar_high["pswq_total"]:.0f}, '
      f'sticky_match_r={exemplar_high["sticky_match_r"]:+.3f}')
print(f'Low-trait exemplar:  sub-{int(exemplar_low["subject"])}, '
      f'PSWQ={exemplar_low["pswq_total"]:.0f}, '
      f'sticky_match_r={exemplar_low["sticky_match_r"]:+.3f}')

In [ ]:
# =============================================================================
# Figure B: Stripe plot — winning thought identity colored by dyscontrol
# At each TR, show which thought "wins" (highest similarity), colored by 
# that thought's within-subject dyscontrol rank
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import pearsonr


def compute_winning_thoughts(sub):
    """For one subject: at each rest TR, which thought wins (max similarity)?
    Returns: winning_dys (dyscontrol of winner at each TR), valid mask."""
    if sub not in per_thought_templates or sub not in sart_stats:
        return None, None
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    if not ts_path.exists():
        return None, None
    rest_ts = np.load(ts_path)
    spike_mask = np.load(spike_path) if spike_path.exists() else np.zeros(len(rest_ts), dtype=bool)
    rest_z = (rest_ts - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
    thoughts = sorted(per_thought_templates[sub].keys())
    template_array = np.array([
        (per_thought_templates[sub][tk] - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
        for tk in thoughts
    ])
    dyscontrol_array = np.array([per_thought_dyscontrol[sub][tk] for tk in thoughts])
    
    n_trs = len(rest_z)
    winning_dys = np.full(n_trs, np.nan)
    winning_idx = np.full(n_trs, -1)
    for t in range(n_trs):
        if spike_mask[t]:
            continue
        sims = np.array([metric_fn(rest_z[t], template_array[i]) for i in range(len(thoughts))])
        winner = int(np.argmax(sims))
        winning_dys[t] = dyscontrol_array[winner]
        winning_idx[t] = winner
    return winning_dys, dyscontrol_array, ~spike_mask


# Compute for exemplars
HIGH_SUB = int(exemplar_high['subject'])
LOW_SUB = int(exemplar_low['subject'])
HIGH_PSWQ = exemplar_high['pswq_total']
LOW_PSWQ = exemplar_low['pswq_total']
HIGH_STICKY = exemplar_high['sticky_match_r']
LOW_STICKY = exemplar_low['sticky_match_r']

low_winning_dys, low_dys_arr, low_valid = compute_winning_thoughts(LOW_SUB)
high_winning_dys, high_dys_arr, high_valid = compute_winning_thoughts(HIGH_SUB)


# Create figure: 3 panels (two stripes + scatter)
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(3, 1, height_ratios=[0.8, 0.8, 2.0], hspace=0.45)
ax_low = fig.add_subplot(gs[0])
ax_high = fig.add_subplot(gs[1])
ax_scatter = fig.add_subplot(gs[2])


def plot_stripe(ax, winning_dys, dyscontrol_array, pswq, sticky_r, title_label):
    """Render winning-thought-dyscontrol as a colored stripe."""
    n_trs = len(winning_dys)
    times_min = np.arange(n_trs) * TR_SECONDS / 60
    
    # Normalize colors to within-subject dyscontrol range
    vmin, vmax = dyscontrol_array.min(), dyscontrol_array.max()
    cmap = plt.cm.RdBu_r
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    # Render as imshow: 1 row, n_trs cols
    stripe = winning_dys.reshape(1, -1)
    im = ax.imshow(stripe, aspect='auto', cmap=cmap, norm=norm,
                    extent=[0, 8, 0, 1], interpolation='nearest')
    
    ax.set_yticks([])
    ax.set_xlim(0, 8)
    ax.set_title(f'{title_label}  •  PSWQ = {pswq:.0f}  •  '
                  f'thought–dyscontrol match = {sticky_r:+.2f}',
                  fontsize=11, loc='left')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, pad=0.01, shrink=0.95, aspect=8)
    cbar.set_label('Dyscontrol of\nreactivated thought', fontsize=8)
    
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)


plot_stripe(ax_low, low_winning_dys, low_dys_arr,
             LOW_PSWQ, LOW_STICKY,
             'a. Low-trait exemplar (reactivated thoughts skew toward lower dyscontrol)')
plot_stripe(ax_high, high_winning_dys, high_dys_arr,
             HIGH_PSWQ, HIGH_STICKY,
             'b. High-trait exemplar (reactivated thoughts skew toward higher dyscontrol)')

ax_high.set_xlabel('Time during rest scan (minutes)', fontsize=10)

# Bottom: scatter, using PTQ which has stronger effect
plot_df = illustrative_df.dropna(subset=['ptq_total', 'sticky_match_r']).copy()
ax_scatter.scatter(plot_df['ptq_total'], plot_df['sticky_match_r'],
                    s=42, color='#7791a8', alpha=0.7,
                    edgecolor='white', linewidth=0.6, zorder=2)

ax_scatter.scatter([exemplar_high['ptq_total']], [HIGH_STICKY], s=200, color='#C73C3C',
                    edgecolor='black', linewidth=1.5, zorder=4,
                    label='High-trait exemplar')
ax_scatter.scatter([exemplar_low['ptq_total']], [LOW_STICKY], s=200, color='#3a7cba',
                    edgecolor='black', linewidth=1.5, zorder=4,
                    label='Low-trait exemplar')

# Regression line
slope, intercept = np.polyfit(plot_df['ptq_total'], plot_df['sticky_match_r'], 1)
xs = np.array([plot_df['ptq_total'].min(), plot_df['ptq_total'].max()])
ax_scatter.plot(xs, slope * xs + intercept, color='#1f3a5f', lw=2, zorder=3)

r, p = pearsonr(plot_df['ptq_total'], plot_df['sticky_match_r'])
sig = ' ***' if p < 0.001 else ' **' if p < 0.01 else ' *' if p < 0.05 else ''

ax_scatter.axhline(0, color='gray', linewidth=0.5, linestyle=':', alpha=0.7)
ax_scatter.set_xlabel('PTQ total (perseverative thinking)', fontsize=11)
ax_scatter.set_ylabel('Thought–dyscontrol match\n(within-subject correlation between\nthought reactivation count and dyscontrol)',
                       fontsize=10)
ax_scatter.set_title(f'c. Across subjects, higher perseverative thinking predicts preferential '
                      f'reactivation of more uncontrollable thoughts'
                      f' (r = {r:+.3f}, p = {p:.4f}{sig})',
                      fontsize=11, loc='left')
ax_scatter.legend(loc='lower right', fontsize=9, frameon=False)

for spine in ['top', 'right']:
    ax_scatter.spines[spine].set_visible(False)

plt.suptitle('Resting-state preferentially reactivates more uncontrollable thoughts with higher trait worry',
              fontsize=12, y=0.995)

fig_path = FIGURES_DIR / 'fig_rest_F3_stripe_v2.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# =============================================================================
# Figure C: Timecourse plot + per-subject bar chart of winning thought distribution
# Bar charts show fraction of TRs at each thought, ordered by within-subject dyscontrol rank
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter1d


def compute_all_for_sub(sub):
    """Get everything for one subject: TR-by-TR sims, winning counts, dyscontrol."""
    if sub not in per_thought_templates or sub not in sart_stats:
        return None
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    if not ts_path.exists():
        return None
    rest_ts = np.load(ts_path)
    spike_mask = np.load(spike_path) if spike_path.exists() else np.zeros(len(rest_ts), dtype=bool)
    rest_z = (rest_ts - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
    thoughts = sorted(per_thought_templates[sub].keys())
    template_array = np.array([
        (per_thought_templates[sub][tk] - sart_stats[sub]['mean']) / sart_stats[sub]['sd']
        for tk in thoughts
    ])
    dyscontrol_array = np.array([per_thought_dyscontrol[sub][tk] for tk in thoughts])

    n_trs = len(rest_z)
    n_thoughts = len(thoughts)
    sims = np.zeros((n_trs, n_thoughts))
    winning = np.full(n_trs, -1)
    for t in range(n_trs):
        for i in range(n_thoughts):
            sims[t, i] = metric_fn(rest_z[t], template_array[i])
        if not spike_mask[t]:
            winning[t] = np.argmax(sims[t])
    sims[spike_mask] = np.nan

    # Smooth
    sims_smooth = np.copy(sims)
    for i in range(n_thoughts):
        col = sims[:, i]
        mask = ~np.isnan(col)
        if mask.sum() > 0:
            sims_smooth[mask, i] = gaussian_filter1d(col[mask], sigma=2)

    # Count winning TRs per thought
    counts = np.zeros(n_thoughts)
    for w in winning:
        if w >= 0:
            counts[w] += 1

    return {
        'sims_smooth': sims_smooth,
        'thoughts': thoughts,
        'dyscontrol': dyscontrol_array,
        'counts': counts,
        'n_valid': int((winning >= 0).sum()),
    }


low_data = compute_all_for_sub(LOW_SUB)
high_data = compute_all_for_sub(HIGH_SUB)


# Layout: 2 rows of (timecourse + bar chart), then scatter
fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.4], width_ratios=[3, 1, 0.2, 0.2],
                       hspace=0.45, wspace=0.3)

ax_low_tc = fig.add_subplot(gs[0, 0])
ax_low_bar = fig.add_subplot(gs[0, 1])
ax_high_tc = fig.add_subplot(gs[1, 0])
ax_high_bar = fig.add_subplot(gs[1, 1])
ax_scatter = fig.add_subplot(gs[2, :])


def plot_timecourse(ax, data, title_label, pswq, sticky_r):
    sims = data['sims_smooth']
    thoughts = data['thoughts']
    dyscontrol = data['dyscontrol']
    n_trs = sims.shape[0]
    times_min = np.arange(n_trs) * TR_SECONDS / 60

    order = np.argsort(dyscontrol)
    bottom3 = order[:3]
    top3 = order[-3:]

    for i in bottom3:
        ax.plot(times_min, sims[:, i], color='#3a7cba', alpha=0.55, linewidth=1.2)
    for i in top3:
        ax.plot(times_min, sims[:, i], color='#C73C3C', alpha=0.85, linewidth=1.8)

    ax.axhline(0, color='gray', linewidth=0.5, linestyle=':', alpha=0.7)
    ax.set_ylabel('Similarity to thought\n(z-scored vs. SART)', fontsize=9)
    ax.set_title(f'{title_label}  •  PSWQ = {pswq:.0f}  •  '
                  f'thought–dyscontrol match = {sticky_r:+.2f}',
                  fontsize=11, loc='left')
    ax.set_xlim(0, 8)
    ax.set_ylim(-1.0, 1.0)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)


def plot_bar(ax, data, title_label):
    """Bar chart of winning-TR count per thought, ordered by within-subject dyscontrol rank."""
    counts = data['counts']
    dyscontrol = data['dyscontrol']
    n_thoughts = len(counts)

    # Order thoughts by ascending dyscontrol (low → high, left to right)
    order = np.argsort(dyscontrol)
    counts_ordered = counts[order] / counts.sum()  # proportion
    dyscontrol_ordered = dyscontrol[order]

    # Color by rank (red for high-dyscontrol, blue for low-dyscontrol)
    norm = plt.Normalize(vmin=dyscontrol.min(), vmax=dyscontrol.max())
    cmap = plt.cm.RdBu_r
    colors = [cmap(norm(d)) for d in dyscontrol_ordered]

    bars = ax.bar(range(n_thoughts), counts_ordered, color=colors,
                    edgecolor='white', linewidth=0.6)

    # Add trend line: fraction vs dyscontrol rank
    ranks = np.arange(n_thoughts)
    if counts_ordered.std() > 0:
        slope, intercept = np.polyfit(ranks, counts_ordered, 1)
        ax.plot(ranks, slope * ranks + intercept, color='black', lw=1.5, alpha=0.7,
                linestyle='--', label=f'slope = {slope:+.3f}')
        ax.legend(fontsize=8, frameon=False, loc='upper left')

    ax.set_xticks(ranks)
    ax.set_xticklabels([f'{int(d)}' for d in dyscontrol_ordered], fontsize=8)
    ax.set_xlabel('Within-subject dyscontrol\n(low → high)', fontsize=9)
    ax.set_ylabel('Fraction of\nwinning TRs', fontsize=9)
    ax.set_title(title_label, fontsize=10, loc='left')

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)


plot_timecourse(ax_low_tc, low_data, 'a. Low-trait exemplar',
                  LOW_PSWQ, LOW_STICKY)
plot_bar(ax_low_bar, low_data, 'b. Distribution of winning thoughts')

plot_timecourse(ax_high_tc, high_data, 'c. High-trait exemplar',
                  HIGH_PSWQ, HIGH_STICKY)
plot_bar(ax_high_bar, high_data, 'd. Distribution of winning thoughts')

ax_high_tc.set_xlabel('Time during rest scan (minutes)', fontsize=10)

# Legend on the low timecourse
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#C73C3C', linewidth=2.0, alpha=0.85,
            label='Three highest-dyscontrol thoughts'),
    Line2D([0], [0], color='#3a7cba', linewidth=1.5, alpha=0.7,
            label='Three lowest-dyscontrol thoughts'),
]
ax_low_tc.legend(handles=legend_elements, loc='upper right', fontsize=9, frameon=False)


# Bottom panel: scatter
plot_df = illustrative_df.dropna(subset=['ptq_total', 'sticky_match_r']).copy()
ax_scatter.scatter(plot_df['ptq_total'], plot_df['sticky_match_r'],
                    s=42, color='#7791a8', alpha=0.7,
                    edgecolor='white', linewidth=0.6, zorder=2)

ax_scatter.scatter([exemplar_high['ptq_total']], [HIGH_STICKY], s=200, color='#C73C3C',
                    edgecolor='black', linewidth=1.5, zorder=4,
                    label='High-trait exemplar')
ax_scatter.scatter([exemplar_low['ptq_total']], [LOW_STICKY], s=200, color='#3a7cba',
                    edgecolor='black', linewidth=1.5, zorder=4,
                    label='Low-trait exemplar')

slope, intercept = np.polyfit(plot_df['ptq_total'], plot_df['sticky_match_r'], 1)
xs = np.array([plot_df['ptq_total'].min(), plot_df['ptq_total'].max()])
ax_scatter.plot(xs, slope * xs + intercept, color='#1f3a5f', lw=2, zorder=3)

r, p = pearsonr(plot_df['ptq_total'], plot_df['sticky_match_r'])
sig = ' ***' if p < 0.001 else ' **' if p < 0.01 else ' *' if p < 0.05 else ''

ax_scatter.axhline(0, color='gray', linewidth=0.5, linestyle=':', alpha=0.7)
ax_scatter.set_xlabel('PTQ total (perseverative thinking)', fontsize=11)
ax_scatter.set_ylabel('Thought–dyscontrol match\n(within-subject correlation between\nthought win count and dyscontrol)',
                       fontsize=10)
ax_scatter.set_title(f'e. Across subjects, higher perseverative thinking predicts preferential '
                      f'reactivation of more uncontrollable thoughts'
                      f' (r = {r:+.3f}, p = {p:.4f}{sig})',
                      fontsize=11, loc='left')
ax_scatter.legend(loc='lower right', fontsize=9, frameon=False)

for spine in ['top', 'right']:
    ax_scatter.spines[spine].set_visible(False)

plt.suptitle('Resting-state preferentially reactivates more uncontrollable thoughts with higher trait worry',
              fontsize=13, y=0.998)

fig_path = FIGURES_DIR / 'fig_rest_F3_combined_v4.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# Figures

In [ ]:
# =============================================================================
# Figure: Parcel coefficients on the brain (with MNI background)
# =============================================================================
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, plotting

# Load Yeo-17 thick atlas
yeo = datasets.fetch_atlas_yeo_2011(n_networks=17, thickness='thick')
atlas_img = nib.load(yeo['maps'])
atlas_data = np.squeeze(atlas_img.get_fdata())

# Load parcel coefficients
parcel_results = pd.read_csv(DECODING_OUT_DIR / 'parcel_coefficients.csv')
STAT_COL = 'univariate_r'

# Build stat map
stat_data = np.zeros_like(atlas_data, dtype=float)
for _, row in parcel_results.iterrows():
    parcel_idx = int(row['parcel_idx'])
    atlas_label = parcel_idx + 1
    mask = (atlas_data == atlas_label)
    stat_data[mask] = row[STAT_COL]

# Note: original atlas has shape (256,256,256,1). We need 3D for plotting.
# Build new image with squeezed shape
new_affine = atlas_img.affine
stat_img = nib.Nifti1Image(stat_data, new_affine)

# Get MNI152 template explicitly
mni_template = datasets.load_mni152_template(resolution=1)

vmax = max(abs(parcel_results[STAT_COL]))

# 1. Ortho view with MNI background
fig = plt.figure(figsize=(14, 5))
plotting.plot_stat_map(
    stat_img,
    bg_img=mni_template,
    display_mode='ortho',
    cut_coords=(0, -50, 30),
    colorbar=True,
    cmap='RdBu_r',
    vmax=vmax,
    threshold=0.001,  # tiny threshold to hide background-zero voxels
    title='Univariate parcel r with dyscontrol (Yeo-17 thick)',
    annotate=True,
    figure=fig,
    draw_cross=False,
)
fig_path = FIGURES_DIR / 'fig_parcel_coefs_brain_ortho.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# 2. Multi-slice axial view (often the most useful for showing cortical pattern)
fig2 = plt.figure(figsize=(18, 4))
plotting.plot_stat_map(
    stat_img,
    bg_img=mni_template,
    display_mode='z',
    cut_coords=[-15, 0, 15, 30, 45, 60],
    colorbar=True,
    cmap='RdBu_r',
    vmax=vmax,
    threshold=0.001,
    title='Univariate parcel r (axial slices)',
    figure=fig2,
)
fig2_path = FIGURES_DIR / 'fig_parcel_coefs_brain_axial.png'
plt.savefig(fig2_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig2_path}')

In [ ]:
# =============================================================================
# Figure: SART → probe2 parcel coefficients (bar chart + brain)
# =============================================================================
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import datasets, plotting

# Load coefficients
sart_p2_coefs = pd.read_csv(DECODING_OUT_DIR / 'parcel_coefficients_sart_probe2.csv')
print(f'Columns: {list(sart_p2_coefs.columns)}')
print(sart_p2_coefs.sort_values('univariate_r', key=abs, ascending=False).round(4))


# -----------------------------------------------------------------------------
# Bar chart (sorted by |univariate_r|, network-colored)
# -----------------------------------------------------------------------------
sorted_df = sart_p2_coefs.sort_values('univariate_r', ascending=False).reset_index(drop=True)

# Color by network
network_colors = {
    'Visual A':            '#7B3F9D',
    'Visual B':            '#9D60BC',
    'Somatomotor A':       '#3B7CB8',
    'Somatomotor B':       '#5B9CD8',
    'Dorsal Attention A':  '#319D5C',
    'Dorsal Attention B':  '#56C080',
    'Salience/VentAttn A': '#C25599',
    'Salience/VentAttn B': '#DC76B6',
    'Limbic A':            '#B8956F',
    'Limbic B':            '#D6B58D',
    'Control A':           '#E08130',
    'Control B':           '#F4A35E',
    'Control C':           '#F9C18C',
    'Default A':           '#C73C3C',
    'Default B':           '#E36C6C',
    'Default C':           '#F49A9A',
    'Default D':           '#F9BCBC',
}

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(sorted_df))
colors = [network_colors.get(net, '#888888') for net in sorted_df['network']]
ax.barh(y_pos, sorted_df['univariate_r'], color=colors, edgecolor='black', linewidth=0.5)

# Add significance markers
for i, row in sorted_df.iterrows():
    if row['univariate_p'] < 0.001:
        marker = '***'
    elif row['univariate_p'] < 0.01:
        marker = '**'
    elif row['univariate_p'] < 0.05:
        marker = '*'
    else:
        marker = ''
    if marker:
        x_pos = row['univariate_r'] + (0.005 if row['univariate_r'] > 0 else -0.005)
        ax.text(x_pos, i, marker, va='center',
                ha='left' if row['univariate_r'] > 0 else 'right',
                fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_df['network'], fontsize=9)
ax.set_xlabel('Univariate Pearson r (SART parcel × probe2)', fontsize=10)
ax.set_title('SART → probe2 (focus): parcel-wise correlations', fontsize=11)
ax.axvline(0, color='black', linewidth=0.6)
ax.invert_yaxis()  # most positive at top

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_sart_probe2_parcels_bar.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')


# -----------------------------------------------------------------------------
# Brain plot
# -----------------------------------------------------------------------------
yeo = datasets.fetch_atlas_yeo_2011(n_networks=17, thickness='thick')
atlas_img = nib.load(yeo['maps'])
atlas_data = np.squeeze(atlas_img.get_fdata())

# Build stat map
stat_data = np.zeros_like(atlas_data, dtype=float)
for _, row in sart_p2_coefs.iterrows():
    parcel_idx = int(row['parcel_idx'])
    atlas_label = parcel_idx + 1
    mask = (atlas_data == atlas_label)
    stat_data[mask] = row['univariate_r']

stat_img = nib.Nifti1Image(stat_data, atlas_img.affine)
mni_template = datasets.load_mni152_template(resolution=1)

vmax = max(abs(sart_p2_coefs['univariate_r']))

# Ortho view
fig = plt.figure(figsize=(14, 5))
plotting.plot_stat_map(
    stat_img,
    bg_img=mni_template,
    display_mode='ortho',
    cut_coords=(0, -50, 30),
    colorbar=True,
    cmap='RdBu_r',
    vmax=vmax,
    threshold=0.001,
    title='SART → probe2 (focus): univariate parcel r',
    annotate=True,
    figure=fig,
    draw_cross=False,
)
fig_path = FIGURES_DIR / 'fig_sart_probe2_brain_ortho.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# Multi-slice axial view
fig2 = plt.figure(figsize=(18, 4))
plotting.plot_stat_map(
    stat_img,
    bg_img=mni_template,
    display_mode='z',
    cut_coords=[-15, 0, 15, 30, 45, 60],
    colorbar=True,
    cmap='RdBu_r',
    vmax=vmax,
    threshold=0.001,
    title='SART → probe2 (focus): axial view',
    figure=fig2,
)
fig2_path = FIGURES_DIR / 'fig_sart_probe2_brain_axial.png'
plt.savefig(fig2_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig2_path}')

In [ ]:
# =============================================================================
# Figure: SART decoding within-subject r distributions (strip plot)
# Shows probe2/probe3 cluster above zero, probe1 at zero
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp

# Load per-subject within-r values for each probe
PROBES_TO_PLOT = ['probe1', 'probe2', 'probe3']
PROBE_LABELS = {
    'probe1': 'Probe 1: Dyscontrol',
    'probe2': 'Probe 2: Focus',
    'probe3': 'Probe 3: Effort',
}

probe_dfs = {}
for probe in PROBES_TO_PLOT:
    df = pd.read_csv(DECODING_OUT_DIR / f'sart_loso_{probe}.csv')
    probe_dfs[probe] = df

# Set up figure
fig, ax = plt.subplots(figsize=(7, 5))

# Color palette: muted slate-blue
DOT_COLOR = '#7791a8'
MEAN_COLOR = '#1f3a5f'

# Per-probe horizontal positions
x_positions = np.arange(len(PROBES_TO_PLOT))

for i, probe in enumerate(PROBES_TO_PLOT):
    df = probe_dfs[probe]
    r_values = df['r'].dropna().values
    
    # Add small horizontal jitter so dots don't overlap
    np.random.seed(42 + i)  # consistent per probe
    jitter = np.random.uniform(-0.12, 0.12, size=len(r_values))
    x = x_positions[i] + jitter
    
    # Plot subject dots
    ax.scatter(x, r_values, s=24, color=DOT_COLOR, alpha=0.6,
                edgecolor='white', linewidth=0.5, zorder=2)
    
    # Group mean
    mean_r = r_values.mean()
    se_r = r_values.std() / np.sqrt(len(r_values))
    ax.hlines(mean_r, x_positions[i] - 0.25, x_positions[i] + 0.25,
              color=MEAN_COLOR, linewidth=2.5, zorder=3)
    # SEM bars
    ax.vlines(x_positions[i], mean_r - se_r, mean_r + se_r,
              color=MEAN_COLOR, linewidth=2, zorder=3)
    
    # Significance test (one-sample t-test against zero)
    t, p = ttest_1samp(r_values, 0)
    
    # Significance marker above the column
    if p < 0.001:
        sig = '***'
    elif p < 0.01:
        sig = '**'
    elif p < 0.05:
        sig = '*'
    else:
        sig = 'n.s.'
    y_max_for_text = max(r_values.max(), 0.05) + 0.05
    ax.text(x_positions[i], y_max_for_text, sig,
            ha='center', va='bottom', fontsize=11,
            color='black' if p < 0.05 else 'gray')

# Reference line at zero
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

# Axes
ax.set_xticks(x_positions)
ax.set_xticklabels([PROBE_LABELS[p] for p in PROBES_TO_PLOT], fontsize=10)
ax.set_ylabel('Within-subject Pearson r\n(SART decoder predictions × probe)',
              fontsize=10)
ax.set_title('SART-trained decoder: within-subject decoder predictions',
             fontsize=11)

# Clean up spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis='both', length=4)

plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_sart_decoding_probes_stripplot.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# Also print summary stats for each probe
print(f'\n{"probe":15s}  {"M":>7s}  {"SE":>5s}  {"t":>5s}  {"p":>7s}')
for probe in PROBES_TO_PLOT:
    df = probe_dfs[probe]
    r_values = df['r'].dropna().values
    t, p = ttest_1samp(r_values, 0)
    print(f'{PROBE_LABELS[probe]:15s}  {r_values.mean():>+7.3f}  '
          f'{r_values.std()/np.sqrt(len(r_values)):>5.3f}  '
          f'{t:>5.2f}  {p:>7.4f}')

In [ ]:
# =============================================================================
# Figure: SART-TB similarity coupling with probes (strip plot, 3 panels)
# Shows: across templates, TB-similarity during SART predicts focus (probe2)
# but not worry-on-mind (probe1)
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp

TEMPLATES = {
    'C1: same-trial':       'sart_similarity_C1_summary.csv',
    'C2: subject-mean':     'sart_similarity_C2_summary.csv',
    'C3: high-dyscontrol':  'sart_similarity_C3_summary.csv',
}
PROBES_TO_PLOT = ['probe1', 'probe2', 'probe3']
PROBE_LABELS = {
    'probe1': 'Probe 1:\nDyscontrol',
    'probe2': 'Probe 2:\nFocus',
    'probe3': 'Probe 3:\nEffort',
}
SIM_METRIC = 'max_sim'  # use max_sim for headline; mean_sim or block_sim also work

# Compute within-subject r for each (subject, template, probe)
def compute_within_r(sim_df, sim_col, target):
    rows = []
    for sub, g in sim_df.groupby('subject'):
        valid = g.dropna(subset=[sim_col, target])
        if (len(valid) < 5 
            or valid[sim_col].std() == 0 
            or valid[target].std() == 0):
            continue
        r, _ = pearsonr(valid[sim_col], valid[target])
        if not np.isnan(r):
            rows.append({'subject': sub, 'r': r})
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

DOT_COLOR = '#7791a8'
MEAN_COLOR = '#1f3a5f'

for panel_idx, (tmpl_label, csv_fname) in enumerate(TEMPLATES.items()):
    ax = axes[panel_idx]
    sim_df = pd.read_csv(DECODING_OUT_DIR / csv_fname)
    
    x_positions = np.arange(len(PROBES_TO_PLOT))
    
    for i, probe in enumerate(PROBES_TO_PLOT):
        within_df = compute_within_r(sim_df, SIM_METRIC, probe)
        r_values = within_df['r'].values
        
        np.random.seed(42 + i + panel_idx * 10)
        jitter = np.random.uniform(-0.12, 0.12, size=len(r_values))
        x = x_positions[i] + jitter
        
        ax.scatter(x, r_values, s=20, color=DOT_COLOR, alpha=0.5,
                   edgecolor='white', linewidth=0.4, zorder=2)
        
        mean_r = r_values.mean()
        se_r = r_values.std() / np.sqrt(len(r_values))
        ax.hlines(mean_r, x_positions[i] - 0.25, x_positions[i] + 0.25,
                  color=MEAN_COLOR, linewidth=2.5, zorder=3)
        ax.vlines(x_positions[i], mean_r - se_r, mean_r + se_r,
                  color=MEAN_COLOR, linewidth=2, zorder=3)
        
        t, p = ttest_1samp(r_values, 0)
        if p < 0.001:
            sig = '***'
        elif p < 0.01:
            sig = '**'
        elif p < 0.05:
            sig = '*'
        else:
            sig = 'n.s.'
        ax.text(x_positions[i], 0.55, sig,
                ha='center', va='bottom', fontsize=10,
                color='black' if p < 0.05 else 'gray')
    
    # Zero reference
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)
    
    # Title and axes
    ax.set_title(tmpl_label, fontsize=11)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([PROBE_LABELS[p] for p in PROBES_TO_PLOT], fontsize=9)
    
    if panel_idx == 0:
        ax.set_ylabel(f'Within-subject Pearson r\n(SART {SIM_METRIC} × probe)',
                      fontsize=10)
    
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', length=4)
    ax.set_ylim(-0.6, 0.7)

plt.suptitle('SART-Thought block similarity → probes: within-subject coupling by template',
             fontsize=12, y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_sart_similarity_probes_stripplot.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# =============================================================================
# Figure: SART-TB similarity coupling with accuracy/RT (strip plot, 3 panels)
# Shows the C1-specific accuracy dissociation: only same-trial template predicts
# behavioral accuracy; abstract templates (C2/C3) do not
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp

TEMPLATES = {
    'C1: same-trial':       'sart_similarity_C1_summary.csv',
    'C2: subject-mean':     'sart_similarity_C2_summary.csv',
    'C3: high-dyscontrol':  'sart_similarity_C3_summary.csv',
}
TARGETS = ['NoGo_ACC', 'AccIndex', 'Mean_go_RT']
TARGET_LABELS = {
    'NoGo_ACC':    'NoGo\naccuracy',
    'AccIndex':    'Skill\nIndex',
    'Mean_go_RT':  'Mean\nGo RT',
}
SIM_METRIC = 'max_sim'


def compute_within_r(sim_df, sim_col, target):
    rows = []
    for sub, g in sim_df.groupby('subject'):
        valid = g.dropna(subset=[sim_col, target])
        if (len(valid) < 5 
            or valid[sim_col].std() == 0 
            or valid[target].std() == 0):
            continue
        r, _ = pearsonr(valid[sim_col], valid[target])
        if not np.isnan(r):
            rows.append({'subject': sub, 'r': r})
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

DOT_COLOR = '#7791a8'
MEAN_COLOR = '#1f3a5f'

for panel_idx, (tmpl_label, csv_fname) in enumerate(TEMPLATES.items()):
    ax = axes[panel_idx]
    sim_df = pd.read_csv(DECODING_OUT_DIR / csv_fname)
    
    x_positions = np.arange(len(TARGETS))
    
    for i, target in enumerate(TARGETS):
        within_df = compute_within_r(sim_df, SIM_METRIC, target)
        r_values = within_df['r'].values
        
        np.random.seed(42 + i + panel_idx * 10)
        jitter = np.random.uniform(-0.12, 0.12, size=len(r_values))
        x = x_positions[i] + jitter
        
        ax.scatter(x, r_values, s=20, color=DOT_COLOR, alpha=0.5,
                   edgecolor='white', linewidth=0.4, zorder=2)
        
        mean_r = r_values.mean()
        se_r = r_values.std() / np.sqrt(len(r_values))
        ax.hlines(mean_r, x_positions[i] - 0.25, x_positions[i] + 0.25,
                  color=MEAN_COLOR, linewidth=2.5, zorder=3)
        ax.vlines(x_positions[i], mean_r - se_r, mean_r + se_r,
                  color=MEAN_COLOR, linewidth=2, zorder=3)
        
        t, p = ttest_1samp(r_values, 0)
        if p < 0.001:
            sig = '***'
        elif p < 0.01:
            sig = '**'
        elif p < 0.05:
            sig = '*'
        else:
            sig = 'n.s.'
        ax.text(x_positions[i], 0.55, sig,
                ha='center', va='bottom', fontsize=10,
                color='black' if p < 0.05 else 'gray')
    
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)
    
    ax.set_title(tmpl_label, fontsize=11)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([TARGET_LABELS[t] for t in TARGETS], fontsize=9)
    
    if panel_idx == 0:
        ax.set_ylabel(f'Within-subject Pearson r\n(SART {SIM_METRIC} × behavior)',
                      fontsize=10)
    
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', length=4)
    ax.set_ylim(-0.6, 0.7)

plt.suptitle('SART-TB similarity → behavior: within-subject coupling by template',
             fontsize=12, y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_sart_similarity_behavior_stripplot.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# =============================================================================
# Figure: TB decoder LOSO strip plot (per-subject r)
# Shows the TB → dyscontrol decoder's per-subject performance
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp

loso_df = pd.read_csv(DECODING_OUT_DIR / 'loso_per_subject.csv')

fig, ax = plt.subplots(figsize=(4, 5))

DOT_COLOR = '#7791a8'
MEAN_COLOR = '#1f3a5f'

r_values = loso_df['r'].dropna().values

# Vertical strip (jitter on x, r on y)
np.random.seed(42)
jitter = np.random.uniform(-0.18, 0.18, size=len(r_values))
x = jitter

ax.scatter(x, r_values, s=28, color=DOT_COLOR, alpha=0.6,
           edgecolor='white', linewidth=0.5, zorder=2)

mean_r = r_values.mean()
se_r = r_values.std() / np.sqrt(len(r_values))
ax.hlines(mean_r, -0.30, 0.30, color=MEAN_COLOR, linewidth=2.5, zorder=3)
ax.vlines(0, mean_r - se_r, mean_r + se_r, color=MEAN_COLOR, linewidth=2, zorder=3)

# Significance test
t, p = ttest_1samp(r_values, 0)
if p < 0.001:
    sig = '***'
elif p < 0.01:
    sig = '**'
elif p < 0.05:
    sig = '*'
else:
    sig = 'n.s.'
y_max_for_text = max(r_values.max(), 0.05) + 0.05
ax.text(0, y_max_for_text, sig, ha='center', va='bottom', fontsize=12,
        color='black' if p < 0.05 else 'gray')

# Reference line at zero
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', zorder=1)

ax.set_xticks([0])
ax.set_xticklabels(['TB → dyscontrol\n(LOSO)'], fontsize=10)
ax.set_xlim(-0.6, 0.6)
ax.set_ylabel('Held-out Pearson r\n(per-subject)', fontsize=10)
ax.set_title(f'TB decoder LOSO performance\n(M={mean_r:+.3f}, '
             f't({len(r_values)-1})={t:.2f}, p={p:.4f})',
             fontsize=10)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='both', length=4)

plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_tb_decoder_loso_stripplot.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print(f'\nN={len(r_values)}, M={mean_r:+.3f}, SD={r_values.std():.3f}, '
      f't={t:.2f}, p={p:.4f}, % positive: {(r_values > 0).mean()*100:.0f}%')

In [ ]:
# =============================================================================
# Figure: Trait moderation scatterplot
# PSWQ × within-subject (C1 mean_sim × NoGo_ACC) coupling
# Each dot = one subject. Negative slope = high-PSWQ subjects show stronger
# (more negative) coupling between TB-likeness and accuracy.
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Compute within-subject coupling per subject
sim_df = pd.read_csv(DECODING_OUT_DIR / 'sart_similarity_C1_summary.csv')
SIM_METRIC = 'mean_sim'
TARGET = 'NoGo_ACC'
TRAIT = 'pswq_total'

sub_coupling = []
for sub, g in sim_df.groupby('subject'):
    valid = g.dropna(subset=[SIM_METRIC, TARGET])
    if (len(valid) < 5
        or valid[SIM_METRIC].std() == 0
        or valid[TARGET].std() == 0):
        continue
    r, _ = pearsonr(valid[SIM_METRIC], valid[TARGET])
    if not np.isnan(r):
        sub_coupling.append({'subject': sub, 'within_r': r})
coupling_df = pd.DataFrame(sub_coupling)

# Add traits
trait_df = (block_df.groupby('subject')[[TRAIT]].first().reset_index())
plot_df = coupling_df.merge(trait_df, on='subject').dropna(subset=['within_r', TRAIT])

# Cross-subject correlation
between_r, between_p = pearsonr(plot_df[TRAIT], plot_df['within_r'])

# Figure
fig, ax = plt.subplots(figsize=(6.5, 5))

DOT_COLOR = '#7791a8'
LINE_COLOR = '#1f3a5f'

# Scatter
ax.scatter(plot_df[TRAIT], plot_df['within_r'], 
           s=40, color=DOT_COLOR, alpha=0.7,
           edgecolor='white', linewidth=0.6, zorder=2)

# Regression line (least-squares fit)
xs = np.array([plot_df[TRAIT].min(), plot_df[TRAIT].max()])
slope, intercept = np.polyfit(plot_df[TRAIT], plot_df['within_r'], 1)
ys = slope * xs + intercept
ax.plot(xs, ys, color=LINE_COLOR, lw=2, zorder=3)

# Reference line at zero
ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.6, zorder=1)

# Axes
ax.set_xlabel('PSWQ total\n(trait worry)', fontsize=11)
ax.set_ylabel(f'Within-subject Pearson r\n(C1 mean_sim × NoGo_ACC)', fontsize=11)

# Title with stats
sig = '***' if between_p<0.001 else '**' if between_p<0.01 else '*' if between_p<0.05 else ''
ax.set_title(f'Trait worry moderates accuracy coupling\n'
             f'r = {between_r:+.3f}, p = {between_p:.3f}{sig} (n = {len(plot_df)})',
             fontsize=11)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='both', length=4)

plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_trait_moderation_scatter.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print(f'\nN={len(plot_df)}, r={between_r:+.3f}, p={between_p:.4f}')

In [ ]:
# =============================================================================
# Figure: Trait × C3 block_sim (between-subject scatter, 2 panels)
# Subjects with higher trait worry show more high-dyscontrol-like activity
# during SART blocks
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Load C3 similarity and average per subject
sim_df = pd.read_csv(DECODING_OUT_DIR / 'sart_similarity_C3_summary.csv')
per_sub = sim_df.groupby('subject')['block_sim'].mean().reset_index()

# Add traits
trait_df = (block_df.groupby('subject')[['ptq_total', 'pswq_total']]
             .first().reset_index())
plot_df = per_sub.merge(trait_df, on='subject')

TRAITS = [
    ('ptq_total',  'PTQ total\n(perseverative thinking)'),
    ('pswq_total', 'PSWQ total\n(chronic worry)'),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

DOT_COLOR = '#7791a8'
LINE_COLOR = '#1f3a5f'

for ax, (trait_col, trait_label) in zip(axes, TRAITS):
    valid = plot_df.dropna(subset=['block_sim', trait_col])
    
    r, p = pearsonr(valid[trait_col], valid['block_sim'])
    
    # Scatter
    ax.scatter(valid[trait_col], valid['block_sim'],
               s=42, color=DOT_COLOR, alpha=0.7,
               edgecolor='white', linewidth=0.6, zorder=2)
    
    # Regression line
    xs = np.array([valid[trait_col].min(), valid[trait_col].max()])
    slope, intercept = np.polyfit(valid[trait_col], valid['block_sim'], 1)
    ys = slope * xs + intercept
    ax.plot(xs, ys, color=LINE_COLOR, lw=2, zorder=3)
    
    # Reference line at zero
    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.6, zorder=1)
    
    # Axes
    ax.set_xlabel(trait_label, fontsize=11)
    
    # Title
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''
    ax.set_title(f'r = {r:+.3f}, p = {p:.3f}{sig} (n = {len(valid)})',
                 fontsize=11)
    
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', length=4)

axes[0].set_ylabel('C3 block_sim\n(per-subject mean SART-TB similarity\nto high-dyscontrol template)',
                   fontsize=10)

plt.suptitle('Trait worry → high-dyscontrol-template similarity during SART',
             fontsize=12, y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_trait_C3_blocksim_scatter.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# Sensitivity check: re-run trait correlations with various outlier-handling
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

sim_df = pd.read_csv(DECODING_OUT_DIR / 'sart_similarity_C3_summary.csv')
per_sub = sim_df.groupby('subject')['block_sim'].mean().reset_index()
trait_df = (block_df.groupby('subject')[['ptq_total', 'pswq_total']]
             .first().reset_index())
plot_df = per_sub.merge(trait_df, on='subject').dropna()

print(f'Full sample (n={len(plot_df)}):')
for trait in ['ptq_total', 'pswq_total']:
    r_p, p_p = pearsonr(plot_df[trait], plot_df['block_sim'])
    r_s, p_s = spearmanr(plot_df[trait], plot_df['block_sim'])
    print(f'  {trait}: Pearson r={r_p:+.3f} (p={p_p:.4f}), '
          f'Spearman r={r_s:+.3f} (p={p_s:.4f})')

# Outlier sensitivity: drop top and/or bottom value of block_sim
print(f'\n--- Top outlier removed (subject with highest block_sim) ---')
top_idx = plot_df['block_sim'].idxmax()
print(f'  Removing: subject={int(plot_df.loc[top_idx, "subject"])}, '
      f'block_sim={plot_df.loc[top_idx, "block_sim"]:.4f}')
trimmed = plot_df.drop(top_idx)
for trait in ['ptq_total', 'pswq_total']:
    r_p, p_p = pearsonr(trimmed[trait], trimmed['block_sim'])
    print(f'  {trait}: r={r_p:+.3f}, p={p_p:.4f}, n={len(trimmed)}')

print(f'\n--- Bottom outlier removed (subject with lowest block_sim) ---')
bot_idx = plot_df['block_sim'].idxmin()
print(f'  Removing: subject={int(plot_df.loc[bot_idx, "subject"])}, '
      f'block_sim={plot_df.loc[bot_idx, "block_sim"]:.4f}')
trimmed = plot_df.drop(bot_idx)
for trait in ['ptq_total', 'pswq_total']:
    r_p, p_p = pearsonr(trimmed[trait], trimmed['block_sim'])
    print(f'  {trait}: r={r_p:+.3f}, p={p_p:.4f}, n={len(trimmed)}')

print(f'\n--- Both outliers removed ---')
trimmed = plot_df.drop([top_idx, bot_idx])
for trait in ['ptq_total', 'pswq_total']:
    r_p, p_p = pearsonr(trimmed[trait], trimmed['block_sim'])
    print(f'  {trait}: r={r_p:+.3f}, p={p_p:.4f}, n={len(trimmed)}')

# Robust test: Spearman is rank-based, naturally less sensitive to outliers
# Just printed above

# Also: Mahalanobis-based outlier detection on (trait, block_sim) bivariate distribution
# to see which subjects are unusual
from scipy.stats import chi2
def mahalanobis_outliers(x, y, alpha=0.05):
    data = np.column_stack([x, y])
    mu = data.mean(axis=0)
    cov = np.cov(data, rowvar=False)
    inv_cov = np.linalg.inv(cov)
    diff = data - mu
    md_sq = np.sum((diff @ inv_cov) * diff, axis=1)
    threshold = chi2.ppf(1 - alpha, df=2)
    return md_sq > threshold

print(f'\n--- Mahalanobis outliers (alpha=0.05, bivariate) ---')
for trait in ['ptq_total', 'pswq_total']:
    is_outlier = mahalanobis_outliers(plot_df[trait], plot_df['block_sim'])
    n_out = is_outlier.sum()
    print(f'  {trait}: {n_out} outlier(s)')
    if n_out > 0:
        print(plot_df.loc[is_outlier, ['subject', trait, 'block_sim']].to_string(index=False))
        trimmed = plot_df[~is_outlier]
        r_p, p_p = pearsonr(trimmed[trait], trimmed['block_sim'])
        print(f'  After removal: r={r_p:+.3f}, p={p_p:.4f}, n={len(trimmed)}')

In [ ]:
# =============================================================================
# Figure: Trait moderation scatterplot — PSWQ × within-subject probe2 coupling
# Across 3 templates (3 panels). FEAT-like preprocessing.
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

TEMPLATES = {
    'C1: same-trial':       'sart_similarity_C1_summary.csv',
    'C2: subject-mean':     'sart_similarity_C2_summary.csv',
    'C3: high-dyscontrol':  'sart_similarity_C3_summary.csv',
}
TARGET = 'probe2'
TRAIT = 'pswq_total'

# Best-metric selections from focused-strict analysis
# (you can also just use mean_sim across all for consistency)
BEST_METRIC_PER_TEMPLATE = {
    'C1: same-trial':       'mean_sim',
    'C2: subject-mean':     'max_sim',
    'C3: high-dyscontrol':  'block_sim',
}

trait_df = (block_df.groupby('subject')[[TRAIT]].first().reset_index())

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

DOT_COLOR = '#7791a8'
LINE_COLOR = '#1f3a5f'

for ax, (tmpl_label, csv_fname) in zip(axes, TEMPLATES.items()):
    sim_metric = BEST_METRIC_PER_TEMPLATE[tmpl_label]
    sim_df = pd.read_csv(DECODING_OUT_DIR / csv_fname)
    
    # Compute per-subject within-subject Pearson r
    sub_coupling = []
    for sub, g in sim_df.groupby('subject'):
        valid = g.dropna(subset=[sim_metric, TARGET])
        if (len(valid) < 5 
            or valid[sim_metric].std() == 0 
            or valid[TARGET].std() == 0):
            continue
        r, _ = pearsonr(valid[sim_metric], valid[TARGET])
        if not np.isnan(r):
            sub_coupling.append({'subject': sub, 'within_r': r})
    coupling_df = pd.DataFrame(sub_coupling)
    plot_df = coupling_df.merge(trait_df, on='subject').dropna()
    
    # Cross-subject correlation
    between_r, between_p = pearsonr(plot_df[TRAIT], plot_df['within_r'])
    
    # Scatter
    ax.scatter(plot_df[TRAIT], plot_df['within_r'],
               s=40, color=DOT_COLOR, alpha=0.7,
               edgecolor='white', linewidth=0.6, zorder=2)
    
    # Regression line
    xs = np.array([plot_df[TRAIT].min(), plot_df[TRAIT].max()])
    slope, intercept = np.polyfit(plot_df[TRAIT], plot_df['within_r'], 1)
    ys = slope * xs + intercept
    ax.plot(xs, ys, color=LINE_COLOR, lw=2, zorder=3)
    
    ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.6, zorder=1)
    
    sig = '***' if between_p<0.001 else '**' if between_p<0.01 else '*' if between_p<0.05 else ''
    ax.set_title(f'{tmpl_label}\n({sim_metric} × {TARGET}): r = {between_r:+.3f}, '
                 f'p = {between_p:.3f}{sig} (n = {len(plot_df)})', fontsize=10)
    
    ax.set_xlabel('PSWQ total\n(chronic worry)', fontsize=10)
    
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis='both', length=4)

axes[0].set_ylabel(f'Within-subject Pearson r\n(SART {TARGET} × TB-similarity)',
                   fontsize=10)

plt.suptitle(f'Trait worry sharpens within-subject brain-behavior coupling for {TARGET}',
             fontsize=12, y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / 'fig_trait_probe2_coupling_scatter.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# Robustness checks

C3 defined by high-intensity vs. low-intensity blocks rather than median split

In [ ]:
import pandas as pd
from events_loader import load_events_tsv

# Make sure tb_blocks exists (rebuild from block_df only if needed)
if 'Intensity' not in tb_blocks.columns:
    intensity_rows = []
    for sub in sorted(tb_blocks['subject'].unique()):
        for run in [1, 2, 3]:
            ev = load_events_tsv(sub, run)
            if ev is None or 'Intensity' not in ev.columns:
                continue
            sub_int = ev[['thoughtNumber', 'Intensity']].drop_duplicates()
            sub_int['subject'] = sub
            intensity_rows.append(sub_int)
    intensity_lookup = (
        pd.concat(intensity_rows, ignore_index=True)
        .drop_duplicates(['subject', 'thoughtNumber'])
        .assign(Intensity=lambda d: d['Intensity'].str.lower().str.strip())
        [['subject', 'thoughtNumber', 'Intensity']]
    )
    tb_blocks = tb_blocks.merge(intensity_lookup, on=['subject', 'thoughtNumber'], how='left')

print('Intensity present:', 'Intensity' in tb_blocks.columns)
print('Unlabeled:', tb_blocks['Intensity'].isna().sum())

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, pearsonr

parcel_cols = [c for c in tb_blocks.columns if c.startswith('parcel_')]
analysis_subjects = sorted(tb_blocks['subject'].unique())
print(f'{len(analysis_subjects)} analysis subjects')

# --- Build design-based high-dyscontrol template (Intensity == 'high') ---
tb_high_mean_design = {}
for sub in analysis_subjects:
    sub_tb = tb_blocks[tb_blocks['subject'] == sub]
    high = sub_tb[sub_tb['Intensity'] == 'high'][parcel_cols].values.astype(float)
    if len(high) >= 2:
        tb_high_mean_design[sub] = high.mean(axis=0)
print(f'Design-based high-dyscontrol templates: {len(tb_high_mean_design)}')

# Swap in the design template so existing functions use it
tb_high_mean_continuous = tb_high_mean
tb_high_mean = tb_high_mean_design

# === R2: pre-error reactivation (C3, lag=-1, w=3) ===
eff = get_subject_trial_level_effects(template_kind='C3', lag=-1, window=3, trial_kind='nogo')
eff = pd.Series(eff).dropna()
t, p = ttest_1samp(eff, 0)
print(f'\n[R2] pre-error C3 (design): M={eff.mean():+.4f}, t({len(eff)-1})={t:+.2f}, p={p:.4f}')
print(f'     continuous was: t=+2.49, p≈0.016')

# === R3/F1/F2: rest similarity + trait ===
metric_fn = get_metric(SIMILARITY_METRIC)
rows = []
for sub in analysis_subjects:
    if sub not in tb_high_mean_design or sub not in sart_stats:
        continue
    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    if not ts_path.exists():
        continue
    rest_ts = np.load(ts_path)
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    spike = np.load(spike_path) if spike_path.exists() else np.zeros(len(rest_ts), bool)
    valid = ~spike
    if valid.sum() < 30:
        continue
    mu, sd = sart_stats[sub]['mean'], sart_stats[sub]['sd']
    rest_z = (rest_ts - mu) / sd
    tmpl_z = (tb_high_mean_design[sub] - mu) / sd
    sims = np.array([metric_fn(rest_z[t], tmpl_z) for t in range(len(rest_z)) if valid[t]])
    rows.append({'subject': sub, 'mean_sim': sims.mean(), 'frac_in_state': (sims > 0).mean()})

rest_df = pd.DataFrame(rows).merge(sub_traits, on='subject', how='left')
t, p = ttest_1samp(rest_df['mean_sim'], 0)
print(f'\n[F1] rest~high-dys (design): M={rest_df["mean_sim"].mean():+.4f}, t={t:+.2f}, p={p:.4f}')
for metric, label in [('mean_sim', 'F1 mean_sim'), ('frac_in_state', 'F2 frac_in_state')]:
    print(f'  {label}:')
    for trait in ['pswq_total', 'ptq_total', 'ptq_core']:
        v = rest_df.dropna(subset=[metric, trait])
        r, pp = pearsonr(v[metric], v[trait])
        sig = '**' if pp<0.01 else '*' if pp<0.05 else '†' if pp<0.10 else ''
        print(f'    {trait}: r={r:+.3f}, p={pp:.4f} {sig}')
print('  continuous was: F1 C3×PSWQ r=+0.389 p=0.005; F2 frac×PSWQ r=+0.375 p=0.007')

tb_high_mean = tb_high_mean_continuous
print('\n(Restored continuous median-split tb_high_mean)')

In [ ]:
# =============================================================================
# Decoder generalization tests with consistent (zscore_against_sart) scaling,
# plus principled corrections for all exploratory families.
#
# Prerequisites: cell 60 must have run (defines loso_models, zscore_against_sart,
#                LAGS, WINDOW, sart_blocks, sart_stats, ATLAS_DIR, REST_PARCELS_DIR).
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.stats.multitest import multipletests


# -----------------------------------------------------------------------------
# Helper: print a corrected-family result table
# -----------------------------------------------------------------------------
def correct_family(label, pvals_dict, method='fdr_bh', alpha=0.05):
    """Print uncorrected and corrected p-values for an analysis family."""
    names = list(pvals_dict.keys())
    ps = np.array(list(pvals_dict.values()))
    reject, p_corr, _, _ = multipletests(ps, alpha=alpha, method=method)
    print(f'\n=== {label} ({method}, n_tests={len(ps)}) ===')
    for n, p, pc, r in zip(names, ps, p_corr, reject):
        sig = ' *' if r else ''
        print(f'  {n:50s}  p_unc={p:.4f}   p_corr={pc:.4f}{sig}')
    return dict(zip(names, p_corr))


# =============================================================================
# PART 1: F4 — Decoder generalization to RESTING-STATE TRs
#   Scaling: zscore_against_sart (matches templates approach in cell 61)
#   Tests: group-level mean prediction, trait correlations
# =============================================================================
print('=' * 80)
print('PART 1: Decoder generalization to rest, with zscore_against_sart scaling')
print('=' * 80)

rest_decoded = []
for sub in sorted(loso_models.keys()):
    if sub not in sart_stats:
        continue

    ts_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_parcel_ts.npy'
    spike_path = REST_PARCELS_DIR / f'sub-{sub}_run-01_spike_mask.npy'
    if not ts_path.exists():
        continue

    rest_ts = np.load(ts_path)
    spike_mask = (np.load(spike_path) if spike_path.exists()
                  else np.zeros(len(rest_ts), dtype=bool))
    valid = ~spike_mask
    if valid.sum() < 60:
        continue

    # Z-score each rest TR against the subject's SART distribution,
    # matching how cell 61 treats trial windows.
    rest_z = np.array([zscore_against_sart(rest_ts[t], sub)
                       for t in range(len(rest_ts))])

    # Apply this subject's LOSO model (trained without this subject)
    model = loso_models[sub]
    pred = model.predict(rest_z)
    pred_valid = pred[valid]

    rest_decoded.append({
        'subject': sub,
        'n_valid_trs': int(valid.sum()),
        'rest_pred_mean': pred_valid.mean(),
        'rest_pred_std':  pred_valid.std(),
        'rest_pred_median': np.median(pred_valid),
        'rest_pred_max':  pred_valid.max(),
        'rest_pred_min':  pred_valid.min(),
        'rest_pred_iqr':  np.percentile(pred_valid, 75) - np.percentile(pred_valid, 25),
        'rest_pred_frac_high': (pred_valid > 0).mean(),
    })

rest_dec_df = pd.DataFrame(rest_decoded)
print(f'\nDecoder applied to {len(rest_dec_df)} subjects')
print(rest_dec_df.describe().round(4))

# Group test: is mean predicted dyscontrol at rest different from zero?
vals = rest_dec_df['rest_pred_mean'].values
t, p = ttest_1samp(vals, 0)
sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else ''
print(f'\nGroup mean predicted dyscontrol at rest: '
      f'M={vals.mean():+.4f}, SD={vals.std():.4f}, '
      f't({len(vals)-1})={t:+.2f}, p={p:.4f}  {sig}')

# Merge with traits (assumes sub_traits and trait_cols are defined upstream;
# if they aren't, adapt to your trait dataframe.)
rest_dec_df = rest_dec_df.merge(sub_traits, on='subject', how='left')

# Trait correlations
print('\n--- Trait correlations on rest decoder predictions ---')
DEC_METRICS = ['rest_pred_mean', 'rest_pred_std', 'rest_pred_frac_high']
rest_dec_trait_ps = {}
for metric in DEC_METRICS:
    print(f'\n{metric}:')
    for trait in trait_cols:
        valid = rest_dec_df.dropna(subset=[metric, trait])
        if len(valid) < 10:
            continue
        r, p = pearsonr(valid[metric], valid[trait])
        sig = ('***' if p < .001 else '**' if p < .01
               else '*' if p < .05 else '†' if p < .10 else '')
        print(f'  {trait:22s}  r = {r:+.3f}  p = {p:.4f}  n={len(valid)}  {sig}')
        rest_dec_trait_ps[f'{metric} × {trait}'] = p

# FDR-correct the family of decoder-on-rest trait tests
correct_family('F4: rest decoder × trait', rest_dec_trait_ps, method='fdr_bh')

rest_dec_df.to_csv(DECODING_OUT_DIR / 'F4_rest_decoder_sartscale.csv',
                   index=False)


# =============================================================================
# PART 2: Apply principled corrections to each existing exploratory family
# =============================================================================
print('\n\n' + '=' * 80)
print('PART 2: Multiple-comparison corrections for exploratory analyses')
print('=' * 80)

# --- 1. TB decoder × 6 probes (Bonferroni across 6) ---
correct_family('TB decoder × 6 probes',
    {'probe1 (dyscontrol)': 0.0000,
     'probe2 (focus)':      0.1762,
     'probe3 (effort)':     0.5309,
     'probe4 (fear)':       0.0005,
     'probe5 (sadness)':    0.0000,
     'probe6 (alertness)':  0.0089},
    method='bonferroni')

# --- 2. SART decoder × 7 outcomes (Bonferroni across 7) ---
correct_family('SART decoder × 7 outcomes',
    {'dyscontrol (rating)': 0.1232,
     'probe1':              0.5200,
     'probe2 (focus)':      0.0001,
     'probe3 (effort)':     0.0132,
     'probe4':              0.5419,
     'probe5':              0.6603,
     'probe6':              0.7216},
    method='bonferroni')

# --- 3. E2: rest similarity × trait (FDR across 3 templates × 5 traits = 15) ---
correct_family('E2: rest similarity × trait',
    {'C2 × pswq_total':           0.0136,
     'C2 × ptq_total':            0.1162,
     'C2 × ptq_core':             0.0485,
     'C2 × ptq_unproductive':     0.2681,
     'C2 × ptq_capacity':         0.2400,
     'C3 × pswq_total':           0.0013,
     'C3 × ptq_total':            0.0219,
     'C3 × ptq_core':             0.0089,
     'C3 × ptq_unproductive':     0.0706,
     'C3 × ptq_capacity':         0.0708,
     'C3grad × pswq_total':       0.7252,
     'C3grad × ptq_total':        0.5841,
     'C3grad × ptq_core':         0.3281,
     'C3grad × ptq_unproductive': 0.9652,
     'C3grad × ptq_capacity':     0.6013},
    method='fdr_bh')

# --- 4. F2: rest dynamics frac_in_state × trait (FDR across 5 traits) ---
correct_family('F2: frac_in_state × trait (primary metric)',
    {'pswq_total':       0.0057,
     'ptq_total':        0.0763,
     'ptq_core':         0.0318,
     'ptq_unproductive': 0.2043,
     'ptq_capacity':     0.1626},
    method='fdr_bh')

# --- 5. F3: sticky_match_r × trait (FDR across 5 traits) ---
correct_family('F3: sticky_match_r × trait (primary metric)',
    {'pswq_total':       0.0534,
     'ptq_total':        0.0033,
     'ptq_core':         0.0059,
     'ptq_unproductive': 0.0077,
     'ptq_capacity':     0.0086},
    method='fdr_bh')

# --- 6. Pre-trial coupling × trait (FDR across 3 templates × 5 traits = 15) ---
correct_family('Pre-trial coupling × trait (expected null — supports trait-invariant claim)',
    {'C1 × pswq_total':       0.6407,
     'C1 × ptq_total':        0.1412,
     'C1 × ptq_core':         0.1840,
     'C1 × ptq_unproductive': 0.2816,
     'C1 × ptq_capacity':     0.0935,
     'C2 × pswq_total':       0.2528,
     'C2 × ptq_total':        0.3281,
     'C2 × ptq_core':         0.2114,
     'C2 × ptq_unproductive': 0.8264,
     'C2 × ptq_capacity':     0.2248,
     'C3 × pswq_total':       0.1104,
     'C3 × ptq_total':        0.3160,
     'C3 × ptq_core':         0.2749,
     'C3 × ptq_unproductive': 0.6746,
     'C3 × ptq_capacity':     0.2075},
    method='fdr_bh')

print('\n\nAll corrections applied. Paste the output and we can update the')
print('Methods and Results to reflect the corrected significance levels.')

In [ ]:
# =============================================================================
# Decoder applied to pre-omission windows (parallel to cell 61 commission test)
# Tests whether LOSO decoder predictions differ between miss (omission) and hit trials
# at pre-trial lags. Uses zscore_against_sart scaling for consistency.
# =============================================================================
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

# Reuses: loso_models, sart_blocks, sart_stats, zscore_against_sart,
#         ATLAS_DIR, load_events_tsv, classify_sart_trial, TR_SECONDS

LAGS = list(range(-3, 7))   # same 10 lags as cell 61
WINDOW = 3                   # 3-TR averaging window

def get_subject_decoder_omission_preds(sub, lag, window):
    """For one subject, return per-trial decoder predictions split by
    omission vs hit, averaged over a `window`-TR window centered at `lag`.
    """
    if sub not in loso_models or sub not in sart_stats:
        return None, None
    model = loso_models[sub]
    
    sub_sart = sart_blocks[sart_blocks['subject'] == sub]
    miss_preds = []
    hit_preds  = []
    
    for _, sart_row in sub_sart.iterrows():
        run = sart_row['run']
        tn  = sart_row['thoughtNumber']
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)
        
        events = load_events_tsv(sub, run)
        if events is None:
            continue
        go_trials = events[
            (events['trial_type'] == 'go') &
            (events['thoughtNumber'] == tn)
        ]
        
        for _, trial in go_trials.iterrows():
            outcome = classify_sart_trial(trial)
            if outcome not in ('correct', 'omission'):
                continue
            center_tr = int(round(trial['onset'] / TR_SECONDS))
            # Build a window: TRs in [center_tr+lag - (window-1)/2 ... +(window-1)/2]
            half = (window - 1) // 2
            tr_lo = center_tr + lag - half
            tr_hi = center_tr + lag + half
            if tr_lo < 0 or tr_hi >= len(parcel_ts):
                continue
            # Mean pattern across window, then z-score against SART, then predict
            mean_pattern = parcel_ts[tr_lo:tr_hi + 1].mean(axis=0)
            pattern_z = zscore_against_sart(mean_pattern, sub)
            pred = float(model.predict(pattern_z.reshape(1, -1))[0])
            if outcome == 'omission':
                miss_preds.append(pred)
            else:  # correct go = hit
                hit_preds.append(pred)
    
    return (np.array(miss_preds) if miss_preds else None,
            np.array(hit_preds)  if hit_preds  else None)


# Run across all lags
print('Decoder applied to omissions vs hits (zscore_against_sart scaling):')
print(f'{"lag":>4}  {"n_subj":>6}  {"miss_M":>8}  {"hit_M":>8}  {"diff_M":>8}  {"t":>6}  {"p":>7}')
print('-' * 65)

dec_omission_rows = []
for lag in LAGS:
    subj_diffs = []
    n_used = 0
    for sub in sorted(loso_models.keys()):
        miss, hit = get_subject_decoder_omission_preds(sub, lag, WINDOW)
        if miss is None or hit is None or len(miss) < 1 or len(hit) < 1:
            continue
        subj_diffs.append(miss.mean() - hit.mean())
        n_used += 1
    if len(subj_diffs) < 5:
        continue
    arr = np.array(subj_diffs)
    t, p = ttest_rel(arr + 0, np.zeros_like(arr)) if False else (None, None)
    from scipy.stats import ttest_1samp
    t, p = ttest_1samp(arr, 0)
    p_bonf = min(p * len(LAGS), 1.0)
    print(f'{lag:>4}  {n_used:>6}  {"":>8}  {"":>8}  {arr.mean():+8.4f}  {t:>+6.2f}  {p:>7.4f}'
          + (f'  p_bonf={p_bonf:.4f}' if p_bonf < 1 else ''))
    dec_omission_rows.append({'lag': lag, 'n': n_used, 'diff_mean': arr.mean(),
                              't': t, 'p': p, 'p_bonf': p_bonf})

dec_omission_df = pd.DataFrame(dec_omission_rows)
dec_omission_df.to_csv('figure_data/decoder_omission_lags.csv', index=False)
print(f'\nSaved to figure_data/decoder_omission_lags.csv')
print('Specifically for the Results paragraph, lag -1 result:')
lm1 = dec_omission_df[dec_omission_df['lag'] == -1]
if len(lm1):
    row = lm1.iloc[0]
    print(f'  lag -1: t({row["n"]-1}) = {row["t"]:+.2f}, p = {row["p"]:.3f}, p_bonf = {row["p_bonf"]:.3f}')

# Save out files for use in figures notebook

In [ ]:
# =============================================================================
# Save objects needed by figures_pnas.ipynb
# Run this as the last cell after everything else has executed.
# =============================================================================
import pickle
from pathlib import Path

FIG_DATA_DIR = Path('figure_data')   # adjust if you want a different path
FIG_DATA_DIR.mkdir(exist_ok=True)

# ---- DataFrames (CSV) ----
block_df.to_csv(FIG_DATA_DIR / 'block_df.csv', index=False)
peri_df.to_csv(FIG_DATA_DIR / 'peri_df.csv', index=False)
thought_dyn_df.to_csv(FIG_DATA_DIR / 'thought_dyn_df.csv', index=False)
rest_dyn_df.to_csv(FIG_DATA_DIR / 'rest_dyn_df.csv', index=False)
sub_traits.to_csv(FIG_DATA_DIR / 'sub_traits.csv')

# illustrative_df (constructed in figure code, but easier to save the input pieces;
# you can reconstruct it the same way in the figure notebook)

# ---- Per-subject dicts (pickle) ----
with open(FIG_DATA_DIR / 'per_thought_templates.pkl', 'wb') as f:
    pickle.dump(per_thought_templates, f)
with open(FIG_DATA_DIR / 'per_thought_dyscontrol.pkl', 'wb') as f:
    pickle.dump(per_thought_dyscontrol, f)
with open(FIG_DATA_DIR / 'sart_stats.pkl', 'wb') as f:
    pickle.dump(sart_stats, f)

# ---- Decoder coefficient vectors (pickle) ----
# loso_models contains sklearn Ridge objects. Save just the coefs + intercepts
# (more portable, smaller, sufficient for any figure use).
loso_coefs = {sub: {'coef': m.coef_.copy(), 'intercept': m.intercept_}
              for sub, m in loso_models.items()}
with open(FIG_DATA_DIR / 'loso_coefs.pkl', 'wb') as f:
    pickle.dump(loso_coefs, f)

# ---- Constants the figure notebook needs to know ----
import json
constants = {
    'TR_SECONDS': float(TR_SECONDS),
    'REST_PARCELS_DIR': str(REST_PARCELS_DIR),
    'SIMILARITY_METRIC': SIMILARITY_METRIC,
    'parcel_cols': parcel_cols,
    'trait_cols': trait_cols,
}
with open(FIG_DATA_DIR / 'constants.json', 'w') as f:
    json.dump(constants, f, indent=2)

print(f'Saved to {FIG_DATA_DIR.resolve()}')
for p in sorted(FIG_DATA_DIR.iterdir()):
    print(f'  {p.name:35s} {p.stat().st_size // 1024} KB')

In [ ]:
import pandas as pd
print('YEO17_NAMES type:', type(YEO17_NAMES), 'len:', len(YEO17_NAMES))
print('First few:', YEO17_NAMES[:5])
print('Parcel cols len:', len(parcel_cols))

# Save the mapping
if len(YEO17_NAMES) == len(parcel_cols):
    # One label per parcel — direct mapping
    pd.DataFrame({'parcel': parcel_cols, 'network': YEO17_NAMES}).to_csv(
        'figure_data/yeo17_parcel_labels.csv', index=False)
    print('Saved parcel-level mapping.')
else:
    # 17 network names — need to figure out parcel-to-network assignment
    # The `parcel_coefficients` function must do this internally. Look there.
    print('YEO17_NAMES is network-level, not parcel-level.')
    print('Check parcel_coefficients() to see how it maps parcels to networks.')

In [ ]:
# =============================================================================
# Peristimulus similarity for all three templates (C1, C2, C3)
# Long-form peri_df with `template` column — for figures_pnas Figure 3
# =============================================================================
PERI_PRE_TRS = 3
PERI_POST_TRS = 6
PERI_TRS = list(range(-PERI_PRE_TRS, PERI_POST_TRS + 1))
metric_fn = get_metric(SIMILARITY_METRIC)

# Pre-compute per-subject mean-thought (C2) and high-dyscontrol (C3) templates.
# These don't vary by trial — one template per subject.
c2_templates = {}   # subject -> mean-thought pattern (unstandardized)
c3_templates = {}   # subject -> high-dyscontrol pattern (unstandardized)

for sub in block_df['subject'].unique():
    tb_sub = block_df[(block_df['subject'] == sub) &
                      (block_df['condition'] == 'Thoughtblock')].dropna(subset=['dyscontrol_imputed'])
    if len(tb_sub) < 2:
        continue
    X = tb_sub[parcel_cols].values   # rows = thought blocks
    # C2: mean across all thought blocks
    c2_templates[sub] = X.mean(axis=0)
    # C3: mean across above-median-dyscontrol thought blocks (matches the C3 definition
    # used elsewhere in decoder notebook — confirm this is your definition!)
    median_dys = tb_sub['dyscontrol_imputed'].median()
    high_mask = tb_sub['dyscontrol_imputed'] > median_dys
    if high_mask.sum() >= 1:
        c3_templates[sub] = X[high_mask].mean(axis=0)

peri_rows = []
for _, sart_row in sart_blocks.iterrows():
    sub = sart_row['subject']
    run = sart_row['run']
    tn  = sart_row['thoughtNumber']
    
    if sub not in sart_stats:
        continue
    
    # Build template patterns (z-scored against SART) for this subject/trial
    templates_z = {}
    tb_pat = tb_lookup.get((sub, run, tn))
    if tb_pat is not None:
        templates_z['same_trial'] = zscore_against_sart(tb_pat, sub)
    if sub in c2_templates:
        templates_z['mean_thought'] = zscore_against_sart(c2_templates[sub], sub)
    if sub in c3_templates:
        templates_z['high_dyscontrol'] = zscore_against_sart(c3_templates[sub], sub)
    if not templates_z:
        continue
    
    run_str = f'{int(run):02d}'
    p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
    if not p_ts.exists():
        continue
    parcel_ts = np.load(p_ts)
    
    events = load_events_tsv(sub, run)
    if events is None:
        continue
    sart_trials = events[
        (events['trial_type'].isin(['go', 'nogo']))
        & (events['thoughtNumber'] == tn)
    ]
    
    for _, trial in sart_trials.iterrows():
        outcome = classify_sart_trial(trial)
        if outcome is None:
            continue
        if trial['trial_type'] == 'go' and outcome == 'correct':
            label = 'hit'
        elif trial['trial_type'] == 'go' and outcome == 'omission':
            label = 'miss'
        elif trial['trial_type'] == 'nogo' and outcome == 'correct':
            label = 'correct_withhold'
        elif trial['trial_type'] == 'nogo' and outcome == 'commission':
            label = 'commission'
        else:
            continue
        
        center_tr = int(round(trial['onset'] / TR_SECONDS))
        for offset in PERI_TRS:
            tr = center_tr + offset
            if tr < 0 or tr >= len(parcel_ts):
                continue
            pattern_z = zscore_against_sart(parcel_ts[tr], sub)
            for tmpl_name, tmpl_z in templates_z.items():
                sim = metric_fn(pattern_z, tmpl_z)
                peri_rows.append({
                    'subject':   sub,
                    'outcome':   label,
                    'tr_offset': offset,
                    'template':  tmpl_name,
                    'sim':       sim,
                })

peri_df_3templates = pd.DataFrame(peri_rows)
print(f'Long-form peri dataframe: {len(peri_df_3templates):,} rows')
print(f'Templates: {peri_df_3templates["template"].unique()}')
print(f'Outcomes:  {peri_df_3templates["outcome"].unique()}')
print(f'Trial counts (template × outcome at tr_offset=0):')
print(peri_df_3templates[peri_df_3templates['tr_offset']==0]
      .groupby(['template', 'outcome']).size().unstack(fill_value=0))

# Save for figures_pnas
peri_df_3templates.to_csv('figure_data/peri_df_3templates.csv', index=False)
print(f'\nSaved to figure_data/peri_df_3templates.csv')

In [ ]:
# =============================================================================
# Decoder applied to trial-level windows (both commissions and omissions)
# Long-form: subject × outcome × tr_offset × pred (per-trial)
# Then aggregate per subject for figure plotting.
# =============================================================================
import numpy as np
import pandas as pd

LAGS = list(range(-3, 7))   # same 10 lags

decoder_trial_rows = []

for sub in sorted(loso_models.keys()):
    if sub not in sart_stats:
        continue
    model = loso_models[sub]
    sub_sart = sart_blocks[sart_blocks['subject'] == sub]

    for _, sart_row in sub_sart.iterrows():
        run = sart_row['run']
        tn  = sart_row['thoughtNumber']
        run_str = f'{int(run):02d}'
        p_ts = ATLAS_DIR / f'sub-{sub}_run-{run_str}_parcel_ts.npy'
        if not p_ts.exists():
            continue
        parcel_ts = np.load(p_ts)

        events = load_events_tsv(sub, run)
        if events is None:
            continue
        trials = events[
            (events['trial_type'].isin(['go', 'nogo'])) &
            (events['thoughtNumber'] == tn)
        ]

        for _, trial in trials.iterrows():
            outcome = classify_sart_trial(trial)
            if trial['trial_type'] == 'go' and outcome == 'correct':
                label = 'hit'
            elif trial['trial_type'] == 'go' and outcome == 'omission':
                label = 'miss'
            elif trial['trial_type'] == 'nogo' and outcome == 'correct':
                label = 'correct_withhold'
            elif trial['trial_type'] == 'nogo' and outcome == 'commission':
                label = 'commission'
            else:
                continue

            center_tr = int(round(trial['onset'] / TR_SECONDS))
            for lag in LAGS:
                tr = center_tr + lag
                if tr < 0 or tr >= len(parcel_ts):
                    continue
                pattern_z = zscore_against_sart(parcel_ts[tr], sub)
                pred = float(model.predict(pattern_z.reshape(1, -1))[0])
                decoder_trial_rows.append({
                    'subject':   sub,
                    'outcome':   label,
                    'tr_offset': lag,
                    'pred':      pred,
                })

decoder_trial_df = pd.DataFrame(decoder_trial_rows)
print(f'Decoder trial-level dataframe: {len(decoder_trial_df):,} rows')
print(f'Outcomes: {decoder_trial_df["outcome"].unique()}')

decoder_trial_df.to_csv('figure_data/decoder_trial_df.csv', index=False)
print(f'Saved to figure_data/decoder_trial_df.csv')